In [ ]:
# SH01 integrated editor — paste this entire file into ONE Kaggle Python cell.
# Enable GPU + Internet in Kaggle. Models download on the first run; no paid API key.
SOURCE_PATH = '/kaggle/input/datasets/vivanjaiswal/video1/military_seth_raw_cut1.mp4'
BRIEF = ('Create a polished, informative vertical short. Select a coherent story and a strong opening; '
         'preserve the speaker’s meaning and voice. Add brief grounded narration in separate intro, bridge '
         'and outro sections. Use relevant supplementary images or video, readable motion graphics, '
         'purposeful camera moves and transitions, restrained SFX and a quiet music bed.')
TARGET_SECONDS = 35
NARRATION = 'intro_bridge_outro'   # 'intro_outro', 'intro', 'auto', or 'off'
REQUIRED_FEATURES = ['voiceover','sfx','music','camera_motion','transitions','graphics','supplementary_media','captions']
REFINEMENT_ROUNDS = 2
OUTPUT_SIZE = (1080, 1920)       # Use (720,1280) for a faster draft.
LOCAL_ASSET_LIBRARY = ''        # Optional annotated media directory; uses Commons otherwise.
MUSIC_PATH = ''                 # Optional local track; empty => original procedural ambient bed.
PRIVATE_ENV_FILE = '/kaggle/working/sh01-fastapi-server/.env'  # Read if present; never overwrite.
TEXT_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
VISION_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
VOICE_NAME = 'en_US-ljspeech-high'  # English public voice; change for another language.
ASR_DEVICE = 'cpu'             # 'cuda' is optional; director and vision use the GPU either way.
KEEP_LOCAL_API = False         # True leaves the loopback API running after this job.
TIMEOUT_SECONDS = 7200

# -------------------- automated setup and execution --------------------
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request, venv, zipfile
from pathlib import Path

# Embedded, inspectable Python source from the matching repository patch.
BUNDLE = (
'P)h>@6aWAK2mk;8VqLIRB-*_W0049{000dD003cWUt@1=aBp*EE^v9pTWxdOxDo#D8UF{C9Z$#`QC6JXwbfLeDz<XbMSiL5T<_|7I4}uHI8!7K'
'BxS3v|9f`<kOW^O+c#~h4>3tB7K_EcJiBB})BH%%Y<w{_c${Z6HAv(ee$L5lNS$BeTZ2UzU2u}II5L8m8{?_r#j7Mv89wTprn$ct@OWjoZjfg='
'r><)-80Q(VisDSnO!xQp_ll?KCh}NZ-Y-eI4B2gUzan0F6UK`Lix%a5oR*iImX{e@RX2Igd?ivsGGtxIa{;$%B-<pwwip;k8x&BC1fUL*uyW{P'
'I!DFup7DtiU;Mqgx|lq8G{K$KXZ1Nv6WA{3QO*CzLONFud&Uo!=V!%Txgue%)F^dn1iPih+++E4!g&nK@;q`i&9^5rUQC?$EV~Fv2TkfN>57z_'
'o2n(|^w*rGSx+q~7C;6)!7ApAgcothyp7_xqCO)on1`FfXeqF`sgZy_;+%VxKGKBTN3I;qNP5>Z)+A&;$*8+zi)A$d)sykfaDUH$pT^Q73GrCA'
'DL9@^PNrjbJ~=y?%-z}f@#p<LDo)3k3Id_Np<(>qNH|%n$k2#lFyS@jhGXO@4f4>qU((1>y2)bLDcVMgDmO?NM^p;YTFDp{*4&@P-koKm&}g#2'
'dsNtNe?haEcxIW}Nl14yVT!oI38k%B38w*j7@3D+%Vy8W$RZtGo8WvuGM~o!Ihi+N+40mpK0m!Uznr)~oPTgXPtQ((rM4pY>B*P*_`_`CUQJIY'
'=U=Ye%gOQi*~ul)t;mD*Z4aH@ic^%BMr+H|cy5^rqM7dorfq`^xa@p#H95PQo}V>%zx!@5*u}ncE;4`nhoe6Z)B^i^XmaV$@R^rgrx`d!bp<|r'
'a|5eAJ}ELcgd^!VA@`PecMDK@<O>S~ekska3$iPCgnZzcJkR|-p9Y2-#rKwNIDY{n=gKHKg?JZ<=+lR%0bW?<!OzaY$~o|_4n7SJPKO7V|G@J@'
'n#D>`*g9Kj(|2^Ug}JUTX@AGF>4giX;(ng|j2bd+T%0Aq0K&6Iel?j*zMqdz-OH=<`S|-u_at2y4q<>n1V?eeEps|VfsB6}XW#=uKD<x@(uah4'
'8TCO|`Yz-k3&6{=l15Dt`ywo*^%!bF?9B2M4ArNjZ_!YEnh_R`I(Ce1iJZMS3fKbDjc?YM6mp{`@CI{9AMKyOu6(wD7$-N@D@C(kA=~@GTTM2W'
'rr_R(ED{mS_{jkg9n`O`QjC$zx3~<ja4A&UIBQT`LyNcp@Li|!UCk!Mt|EGG0coE`D4J#Wj6j<(e5_V{GBqNVZTsnx9DVzp-CxoNr5F2p`0nVY'
'A^MuZGH@|$c$(Ni{ZZX+A3?UEIt+vjaNAS2EHZ#&pJjBF!cNt>;a))3JWKCcwzN#ww4bCf!ZQsaQbKvkK-Lb4AVJC<4_j$X88E20wBuU{@=D+;'
'$>1*`n=mG5)Q@JRLUSlY2G6qer=ZW`7AVwJe+8-zT|ZwX7G93bUs8zD_7m<@(HJxYLpv)o3;xJGVI?~1<Bbc7vnm2Xq=vKD&6XShNn;#;A`ADE'
'Ji`<0b&YdW;I%1j6Y@s~QAax2&PpmqdWtD+H;?WhV#2mNW+@$ajMQwdYihX|#jM`q-n_?n3mJ8^C3|VqwiX4X1z+);7LiWCi83a@Dd02BYGlrb'
'L*o&I1k7v-B+~**8_IM6DFPz&aU-JxNIeHBgyV8o>O^b|PhMN3k0g}1vE~%0OIIypNe)$zsXK}Cgig!n@jYhCZYyltowPELT3DGO*R^}ksobuq'
'v`o5kl7MP$?LMC}-c@0_L-k=C-{pyr%r{^eFj0XgVU=3yKn=k9@JF7;9gE;BjNgsAykXcnoywg7#O7fJ+ALYgCt9jvkUL-xkw+~#Cb5j3jX+Xx'
'V&H-UimFf_*g+TNV_`u<bs!L+lv@rfVG2iuLFq=?nDVP9$y_qNHkBq#1V)u#G)CP#K^*V9Bol-sPwlx~OA?YnJIVv@rUJuRP0-=xSsVnwTmUKw'
'rvS!x!bdqn^2MBDRIpIf9VjEj)Noyo98!HEbrqVF)v)AJWUu-pIM;SGgei|Enq&yL46&Dt0OnI8vP0b+fGcNJKy?Eav2+QVqe<*7Wn|ShZX17A'
'3aMD#RF{?PD1+GNu)e0;N)jhlu}6_0H4Kn(0LvD_<e{M><mS1wKy{z^zL=*}gqQft^<fg!aAAlutoA&gr5@qF-LeY$Kw-1B2)85WHwApo4DfJA'
'a_CyFpftG)5M1j|Ze9fjaC4PhxIyV9);He>lmw{(0(Ns3O47cfm6Vzh%IK}bq*6t8v0^>yYkFywx5Ji86bnn^9+}{%nDD@=;mHDaClFlugiEg<'
'$>OQuEW2fGQ#Se^1*8^!*;Cr9XP4&_HZAjso;PYK+AMQh=MfZ|P)=PcqP!OYb%a0(Q#xSm!C(NUQo44jo}SV0_mS@EFF#}Hxhryq6;GOw%v+*a'
'ztC@b2FW0_B`^fldd~<c!Pe4NM4@zQn&vV1G-&O1qxv+(;*^U>lr5`-3@bq$aLXb6EoBE0c@&jHwbfQO(}rrkSa@6Y(5M2pHmS^omF3n)7ZeAD'
'e12^#;qd}eFwPzH2l|N87YwK?eb57}q*C8d^$_%7nE;XL^-Wt&H|HS?1?FQ_)xWv;@?kbT7Ul4V@#W-A&v^63?sHkqzd3wkKgp)iVO0!bA2&UG'
'l<Ih@ZL932xy3=lg2kmGY?Wc0F2M3AZ`oI8mH3JvT!!DOHbEc(*HVPP%2pX<EHHdi+acnLN<OJ{w`zS^JIwvTB?)sywOaK$U{TrawSXT1+%>|d'
';2R+$;OttdUjVjR@1*3Z+;&Tgvk<GFM<H7=w3Bj7%`BW<LqvtMl$Jg+tZw_qpo87;KK}C_f=Jo)kvtSQ7<F_h?0SCMeq{6kOaa@aDtQXz(h8UW'
'j7mZTI>k*FR4SZL^HHP1Pfv}cq=JUtSg+LvM}vXBdSk)SNMkLF@}t2~54%g?Z4jBudbD6rMB*Zp6%|ug6;`4Bf?9jIvP+V7ZLw>AmAH!amQ>$W'
';BD(8$m&iUg-avxw^3TRf-C->?;1RI25woYyc|7kE8KUNUFtBlm0sd>T1C=U{0>+6ZjAFh$q>%J-Y~0!s}*-Io2^;tV}0cvSyK#eJAek+*1EA*'
'Cdg*HrOcqOm{O(Ba*}$4(*|8je3jHF7Tc1a#aK}~i_B1sJ-rx%f+=>t3<2ytp1B{#v)PC7@n=+C1~s>7_rv%eQj#s26~ZjvmV@wjL({euzM^#!'
'0ECO;h(U2CA)LZ#c|_JQj*(?(Ji;EHS`8HZ$qMR!Y>OSxtY&!I#<#+^h7Gocb_2L)jpVCZs|?qHs;yx`Y}M$d*;)2=rgi+)is1r=l?+}~y{Y~|'
'^QEuVDCVoF<y9vp`mF)7-d48jU{Zvf?Mkj8OKf*)?{HgX)z<)%TOr|dX9~CA+0WrLLDikhDm4nCqO(&28PflD;0zsEfDDc5`uV8&x`_6-mqswy'
'xfHz~pf#)@hwHea-L%!zB_~`L#5XPD?dsOn0%Cp1+H0pWBI-0+W48#1TZXHMrxVl@ZzHl*8Bd>5?-&0jRNO|pYCu&w&%Ts>Uv>Y#$k$F-{m_R5'
'T*x=H#tsNzKIi>d5Z|(pWlVmZQ8~Zn_&SmGHJrYqH*z2Iy8sXYX1#Pdc&lyVuWxHbzVKEk);aYuae(;tbIXSqpFX;$=O>fd&MymetTH=wzsz@&'
'&t~J(v3oh0|2UbqiS6y8V%~JMH~l*$|1V2I(iDQd$~>_0qmnITx18`zkzL^&*d3Ff>6DKWNN-`<A@0TGr^)QH{vifBTEZU4E!lL<nEo{WF#fqc'
'C6Cf1B<Zqa&X=>xi`n?{lRG{>o?Kqm=LMW%K8?Dx`Efp(T%Lb9J83MZ*IQ-)fCx@QM1=+bSy7VWK{*lO@{>U)g@eQo0NaR74meW74#|BMJD9L}'
'9JYv+Yxs(6&C;BNPAN%~9Q?+?eWE|CU~vcBhcmh+ZO$am;uVPMIaGY<=B(eg*|CTs1$ce;Ue>}|9ei(*I^L3ccOihH)WP0u2$S100FQbU!(8a3'
'!9#0!m8Y<tpL=(W9;FzKg%~hZSY)9MPBAsNh*=tkh+rL`lK^D8g(L!}5=!;A$ip~V6ldZs^Qcn*2tB^ooWETsDP55WwI&?NQR%Bq2Vpv79?O*e'
'8SEqxpBi>(8L%b<QD2ZcVEt|JK?gpK(4Kh40^((D&Ep6f4<{??AbISBWCOfAtS*884(qLAVX6yGk|p!ngd}1sp<@qta^isf3HEKuhX8Vd1E@HX'
'Z!xwlTG+Pqh^~F0Oly77B#*qM!#Y?J9}j6wJ58-=OfmB)ejM3hlVQf3HGw0FGDkv#7EwmMrKljCU%@6>w%KNu-~ekHA+Q1pkFuoWq~ug3SinLc'
'#iXqh*1G2M7=%_KHknY%s(y_x{-z-uwwou_wU1w@ly@Nwvg(er#Zuggm$ITAN!Ju2NNvdljaLv`Fml6sd<A>{7hsNu!_DD6^RwmQk|G~KAmB1A'
'B3a!gO!;9J=H%|A_T%(qat?PVlVcPCqgAbiB2WP4%dH&#em?)~o=oRBz5Hx9Etxz$1Bc{Cv0H(V9R7bnpsHcm?CYz}A+`$(8Ea}2!DHC`-1)g#'
'Wr%WBh#EG(kA4XUS1n$ArM+X$bEkt&QPrx4k9Z$JIM`jbB2~R^E#`g2GPd~UhW3Lz3|IIX$>OF8FYfig`P0qg(bGTstK_{|3HZ9NHMK8Er|K(B'
'{WY)RS$>of$Jji!@V^Oc!I;~e$u<s)Iu`#6Kpd`VQI+HWK@hKKko#e^ktUte2a@+$3aW#w4KdrzVlmY&K4}#M;%!7~)gQtvak8(|(fiI%r;w50'
'90;gbcR~ICG+4NFPrdTZHVGY>BBvhK&uvuhdw=ggP)h>@6aWAK2mk;8VqLKh>=}~=005{D000#L003cWUt@1=aBp*EUt@1=ZEtRKE^v8`S8Y$+'
'MiBmv#D5skN_#Gcxlmd~ms}!Bqc%!JQ6Q}<M65k~?rwwab$8cyp}>Fd%&vX*9jORH2G7ph&dW37y}iA6czZB2i13%XQ3Ym7!l^C_U0N{cjKZbN'
'VWr#xDx;TJ$}&ZeWd``A%2lcy&=X6GHs0OY-MK(%(ghUgWG0;iWg#VQMK1`rR2h=|5K8S>pPm8z&SdIpvSKhNYjhsN<pLqssmynGmdb|1rNR{&'
'uukRes|81lsV*vQF@u?r1^EGIxFkmDoFrG6!v?SZsnOi-?l|4^DMOnYRXN2srV~}pa;C4H0h?;$Z@0I%x4Sblx_~5^)vh*}B%sK)aTF1{lJr_Q'
'3=MVRihTWIu#v7!o~vnGNn^2<Cz^j}Vb3I0(%dUqI9IiNA1~f~NKSrxb9Om|bb;v&ap^qMt8xf>YSAn)A#W@4!|U4J$#4c$E>$_8Ox75UcXxn('
'D7m#My@zpKQV?W?1B0=xrvoEiUVVCXz5hxKK>$6hGmUf<#q^@8LDXsYRg<s}xt3W%R~b+g^$>1UnYH#Pc;^OZoENCAoaRWGsf%eTRZgx`mdr{~'
'Je;}pDI-y=kRGMWZ^Kn7!`i61DrN3#jn$q&rK_4VmIZxeor{hOti%+pl}3>QS%OSctem&p`U+&A(yDZE(Dn|o^49&84i?&S+W5jLr}hP@<jQiU'
'ww!YBHF&N9sbFqlim9wQ@~n*&ef;nqWK~i9k%Ov)lk>Oh=7{0kXs1)1ACNE=&$2PhQkJ5G@cZTE`9-{rVE89pXlgkN6S(3y$8&T8(OzCeL+GfU'
'a1Cnaa{-eH2vx{A3c#GyUu|;+12b*Fr<*-)tTk!mAJWF*RaTYOaf^i7@MRKFiu|c3wYAa{_7~8lQ)P`EHCUT$N^MwdPo!~{;b<TZ55>0PR*m46'
'v10-Ifz)wPd2Pw~wxiIjTd=A0O)i<|dga*$K_IqNlLh*adTYX?VspqfQOf4HuDI6XRSU4OHgmD5zHh11LKR@V7Aoj7U1I;NlGd)Y$=b7T(dQ9_'
'QaWIPAsiq781=4+U${TuPrm&dW3+iPYm}BIE?cY(xb$M*P=cpxSYo%5H5OY6?Jb9D6FH~>>)iQr@6~5TRm^v3L_!gJfG51Vdx!#oxoku5-q?*|'
'<nj+GonsL2-N9txEBWf~;X1k+U3Vhfyd=^!4t{ZMLkB}0FP#%>E{ka<ffG4ykeCI$G!uL7dKVdVML#jkfK5U{f#-Z5Xcr4g=^4>QkN-RPqa-Yv'
'$wK8y8VVpoNjs*8VVbYUcW<mylE^zMlt;ZzgQ1WvUC?OwG&nsS-JhPqFMkhSj}I<RqSy2n?!S-j&+94aBy=rEO+@$Tdp=|(Z=Z&OLit;E0cJvo'
'$%b2NK;xxChwk&T9-B(I3kQcQoE9t|-t>Ily%g24c=(no-v-aew*0~2=L)_0@K%+*7jmQeHe=n*t@@X5K8oNKJRkk=Y;<(&A6%IQl~KpJQkh$L'
');{_#P|X+4lOuQuN5`Wl?eqAYJTcK9RR^6vskrIln1^AR4ekWt7rI1cVl2|Ia3*4(7Y%O4E8n?vcuf})R@Wv)Vq8P07;f4VPPv>oLVI=X9LS-e'
'=-FHL#Mm>u<>dU@BS+hs`%w^MdaNA^lrG#o?ad$-O#_JWMkj1pXwSU5es_Otw9qDN)U@uoePW!lPDK0S1&Do5I;9OmxySx<%EXwTq{+#-ZB(2R'
';?8NGB7BTm&}vY+$yEo`VSgTYcnw9sy<I3bdEPekddVdw%d$*t*Q;(MA09Tz-W-RAt3z3-xLQ=A-EsQ6?Vwe(_A_j_q_;KbQ_!;YVz5A&p|O)Y'
'@sXzK!O5H!N-^&3(GNXl8c4EWUYUkqw-ZIs$RPkE!e>kLp&&JAq|=5-Av)V7>4QJFra2vZP5T07Do1x)q2EphEqZ+g(ENrbrpgtvdX&~yJJ*w~'
'BzMQpLnipNh%v*%(P-%3qM;y=ugKGGx_wQZQVkmw;yuILJp<7F#(*OxIt_3$es=VH^e}Y9scSbGJo{;QJZcBm=AyOzY0_nCoy%wgD?BMSl0FF%'
'ihV+hpQ~+uCu|14FM)=gbJMcS_%k8z`><w9bNZ@k`6%!LdD{`aZK5y5F2IMn<ZJr1h+3!lKn|%LLyKEp_Nq~L9Y7(~83B-}%bSu48Nyi<Z8qHh'
'E<z37{s&M?0|XQR000O8003fL>#Mwj`Xm4VfPer16aWAKVQF7uZ*6dIb7fy;WNCD7axQRr)jesG+cuWpQ<eXKakh$*o6SCACoVbkuH!fp?<BF;'
'?PO<5qhe7MBr#2i8XkR2Z~ymw2LJ**)N<@E+f|7r5_ouccsCxPM~@zTpV!-kT!^@?@=aW&iI9u5%8QARSLs4-6Df{gh;3XH@bZejEM%Fkw)iP7'
'OIelpsesn<I{5O7FTeP$l%=TFQplUMtkUgDYye~|<aV*k(`|L~&sx^f7cVc~i|f3&l!b`53z6iTU0&kQfp~o-i(9cM;$`*a7wfnbDU6a|Z^a@l'
'WCBfYpNm*5<1_=zap*G5w<3wRVk@Dm$l-t2MOsP1<2v1L5sjp-0KrF(9)0=6vdA|gik5X%7cz=O3S$=);IYjsm}kB%aVGj{dAm*0+<c$MrTpfJ'
'`T9qhZ_SsyG#?9TK9=jcN;5U=F0R&DI@e>qgReLky^GU`=^&#rS<6kVTfIa|zn4I|vhv0EfWd`aZR8fdyi$Y%tlwqv))#**;@vt;_{+ue2B^8M'
'x&!R;EKP2O!M4n^D;e?Xr*Wp<cl)J<?t@QY38+uBgW<Z3|5?XbTA5MOZMlO5Y<l?SQ33oRi!Ovsjx*B$Ht{Y3rdiPXeEG#fE=3t%$q^FO7gups'
'%kkvPF9iI-Lcfzm2|U=aatutKCUKVCiZX|xD_Iovjt0miz+uMC;*uwqNnoKROfev?L8#ScCqi1o;5x0=QCTmS>CMOs;Dt9fZ31GysFF9;2pJ?;'
')SF#7A_Q^HfKC;j`ydo0a$sC0X&U|*XQdqD!z5n-$A(^AEsvksU<(OUOL)HHh6YKzi{~j&I+f+9U^dZoC?qiXR^0J7{PVILXb`dVdJ~~K1oR*('
'^SVf68E_-u)FK99L=m0md8VNU77WuU2hl5o6;N$^HS%7*iC(?^<=xwh*U`(1i`O5bpWl4{{^k4s@y27{LS`p#WHyWjzi!6226*@S@2`Kph+e*X'
'6aDRV2W)y1&*R%+xct5!@SDHC{Ql+t3_z}dV}V81gP?VTUf93=dhzb(mlr=pFJHZSeR0tP|2n-)No5WWU|Kj6F7iY!>H=1!T;5oHS)>E^z1_1S'
'E+9l%Qw7(cKh`6l;Z7FbnC4*?{whxsi5iwa4KMnK4;RrdZ-03GGtAGK;HECqL^F*ZS%c@7b&(AN{^hreH?N|1FF*X$q8u0k&^HkqBMrD8#_w+~'
'-n{)a`sqiFA`Z8|?QidYR<tmr;PE#3LS_b&drLKel%w7$N2IeX&GZ4*udL$D4h$IRKbfq>T*g)TT<q#?Qq^RHPFC@*guym(x-~k?S2ixm;&f9s'
'iU|~Woq;(>>H<`01xji<^Wi^!yM);x4xoBK^A(j`FiO6-l(%6PZ{~|wTu;RH)C07N%A09q@h@E(u*3@3)ie~x`sMWaY&P{kkHf<kBKYPh(lnFX'
'k$OKCFG6wt#D?5p9>lvHbTw_CJ2Po|sHeI)qB3?cEo4y2xB&hwylL>4f1mxE=Zmr`s3|V*$iq*b?*aD&@FLxf#(y5ChBg+6{_c)G<|YlVEYodT'
'uE{8BS?~p$NEUU`=Al>Mrc?#;1~DKxZm^)`Hr}DH1GX&JyoIX<b}SRCLiu~jiA!LdhKJg=w~&C9L^ng*-)Nf`8!+kUKaj!i>n67==w`YAhbfb-'
'=oN2oM`dscR#5yU6dr3ek9f3121UpLdRbA)PEKhr56rlf;&-&9uZtotM&1v~D-@tRz}b=+d=m^6#3bE;R4E4*?0lK!K$^IaVh6?){sk3%OJ<)v'
'5L9>uQ)BG1O@yCQY>vbPKT%DC$ALL7#0%4&L<6u-^6i3bm&;wI7qt`LiA`!2Jpf3iC(a2Mf9j#)M*O{*-+40r45VYKNt_wFQRzYxkenoV@*F0&'
'1l<gdbO|02u*3p1WwAlF#jpY3pi#Kh)I&@_Oj-=%vUWg>3w*P?9WA8Xg&&F<9UzC{JBy+0S+k_bMEP>*Su-W*#!{6euR?r<vs=lngpQ5k1qitO'
'^Lzn(P{he4gZWDQN{SX&c9SlyW8@MLpB0Lej$Rgl_#r25*vMifPYMaopc4W#K~vD^V6`g!OJ(7o6nROs4nf_64>`JK6^<Q=EJ%<>Spy3xAT~dK'
'hb>4Q15JO;x6)OVW(*3y)RzY2i#AAR)*}&|x0Gei&|w!K_t9JnEy{Z1$j@<0EcP)q1KXg2#lcKipsm_q7qYTAxG~KPL~|_%Md4sP)G%;AlX5FP'
'k+@!@xd=n_&SVL4ByA@B7>2OvOi-g0YW`511*h%7Ib@1f90!(t^6`~;3D#f<k<c6%UZ4buv=slWa}*3x=td*3m}tKP<8E|NG9VEJIY9#zmBFUo'
'5cwQgg%<{$rnwm*<Yk#oCY$(Xbmo)OK##^JC+d-cJs!`d6ZLuKk`kk;8=pWSIpwzAkf4wBIBg>j6vKgt?ixGtQ55`Eb7G%D#+UA6jPYOwp74S{'
'a?s=CYE0S3m>CD75nse_IypO^ja@h~067g1DVhj!&8dLU2O`ick1L|b^mNuZXm0npAiPIfz<dCBB#?OL!BeIvVmdxKdo~_-=s=U8aOF5g->og;'
'BpWuJv~Oc28%Ai+LvNkumtNmAJmRh>iP8D$f+vo^sf96yz>9G16Ivnuk0d%FS0wbReAhofmS4*v%H*;d8X$&+CGvz`2}Vvvi~M@q4{QgcpyPO('
'$<T|F1cn}h3QSQVNEV++^IOD{15mW{jviGRPL~+00VOw+yLb|uFYk}<=JYSD>ev?2ABud3eiVpxTEQBwVo*-BR@Z9@EPkWj7*VI3n~$yyCVf!3'
'FY<bxfs^N`eso<{?17<j;w3{4T^U7W1fs<W^b$Knit*8~IEn?#E~oP(A%N@6@jE9wfG>ewix{vHS-ONq8(W9uNmB1pQVP?SfP=GY#V-z)P8_qE'
'h!wkFJrWN<_#Fzm7M31v2L}w$uciA2d$WCWWew#_z4NUhj#T&~1D3t>(;nGF@4?KPr`cerUtny7(0ee|O`34SHuMgegavSN{$$oMTlH?2k(5DX'
'9&vcv*bn3R5=>4>C`Dhk%$iohtk=&blXF(vO_(3w=;Tp+3m%#;z8=q7lY^GUt*u&3C^0gRjHY!JEI2$*+oVPXJD^IR_o>gLuY;%E)xrfO#K2OB'
'b6aOV8Pb_9rSvk{DP4vb(*OOo7VOtKtZY%Q<esA)cWFXdZ7{j&BfP%}gVSgJZ5TXx=3l3aY8?j8p7?8-uGUo;oPT|&x~x$r&cTCTrZCi-EUPgD'
'tz_F<fL0(tOV)ZJ7@&UmhwDY3f;)&Ns)*u@aOx#7Wp2wxX<CfDycu9ycoUIwEFE{~?KZm==r~<bXuzxw3jni2FEJ1pZDSxryC6x-`Q+<0ghD!w'
'0PwfAo<+q*t|sfNyEth4+Suq>mpSgm|BAbn)v&9Oah9sZi_?3HK`Ct=isPq&D~lwEgiKc(R>#x}6wT-?c-kT$nud6_0?)OCNqcM;JLwz4<_%>v'
'i|fq?<r!Rq*(iZEVzBA{Ri1%=7TP)i00jKl56;g)gtc8&c{aToUR$!Q*;h`IrL{Mr;!TQg(J{1QP|VMd##Tzbar`-**iOo5wWR`%C8!h-wKAkQ'
't4Y%Q1lCWe)B*YQYzZ}RaUB=9IfZ!}%toLnk@v0l05}*=tJ@|^la?hQ0@?vX__CGg`>gSmQg=MnF~pNSZoOI?Zey_qW2ZS`>AlzKle>q2+-vFM'
'ug`TX23;*|?t9WS?O<)do*`eWC25qy0~7Er02Bx^Es26-GD7pZ!puSeI`AsSBq_=?ENy8}KLpvKu#Hgc2~>KQL%$xL$eS(gStTikoyHT>{Y?5s'
'i4F4CT^Z=FEv7L!7|d8v^Ly?No+-P)4jeB93C_KV{&4T}hbXU|=f1}e?(O70x11O0VkNsB_<7Uwo-(~dDI*`&XJlE89(0cz5)9Mwmy|OoR(es<'
'VIsUEFoHTJWE0RG=Gc51Tj+QubLAnaTeHw?7ipbicz86hDCkwTGGgQ5Nf}9PW`%q3z~E&9v211Wax-WX5ukN5;YAqk4=$Wt2GVN=jbHO7zEKJR'
'K|R+y#v+Gyl#XPNW87rfRG{Dt<!)y<ZaFw_XU|SsX01gO73Yerln~97Yh2KR^Cmc=kt0qb16s%cdW7lZ+39S&2Z0j`Ac8Yr2N=gD2p~7J?OF~X'
'w@6ET6c?EL&%u9AlS^=Gs+{l}!U!{oXvC~B{zPJp(9-VQw%*J+?OKu)220?DILk(b_m2h1#UB5iGitVtKVVJOL<WX4lpcdzx3TQ;X9nm?%4Si6'
'KPdDJ<2hy$RmP0{Kn>re_t~k#0-XCxxVB+Xn;twt)!|bn`KyukkDsPJd}peFwc;nOr+T;OxDD?iGD{3?&b1W9kLeb4gSCqapl)^A{#Yk^pQ{=r'
'O3m-$TePfWo#}tImdT|sc}=DNw0`vo=_IeweM`4XAe>H;5-?c1Ek-_6u@e}fGU_=k2>^PAc9_hsqh%_y1)Il4?N6G|r7De)xMfK@`}lwRA#It%'
'W8OM%Ubs|)cnR~Ns-Om%tM9bs;S7Rawz5jl%=$JA+qe%C^U*=u@xxScgXegCc5@TiP0YvU^}Q=Zij0TY$*wfvvgo(uFu02{RbOZRX<yVcWH&WR'
'w$L#8C@bo1OBZZmF?{2P;B2O&K>}HB@Yo;qQM~Iejt4?;>u=OzECZy8>}b4R)-;VRAET4ev7~~?O|w2lS60vbK{7ABJM48oX^3^FKi&I!6Bi!#'
'4P=4_g^cDH!)0=dR-82z&8GJa@4(rWAd2Nw>xi<(-2{Y#QV(mlnXHGwU@$;JcU3H=XQfNMJn*<CI3NQ6bITvRiFu&oPH&>Z$(BHuNe<Bw?Z=$@'
';_Ns$fBG1gm;sEBgVS&5JNE{*5Y2}4M1QhMVYo)K6H6}(SQx+u+;_S3#baM>JF8l_CXRkf6ULwr>(0bmpvcQN$Czb?k<jt_{pVjMT7y&NsMw`g'
'ImXJn()Bha)u<xS*Qy>B)iZ6)6~~maTEPk}0`Z<`3AEl7c`31xvL99<4q1hT&R&zl3SJ>%xr@_<nzMNM?hST_ps?O>hSwy^Ow}!xBnD=(Du&Vs'
'h68m_R$USnvuJb4z1K^wOM0lcsetN&PzusOqS_x~D~%Z{F0$u6^-z;uNmGL2RBD;higpnWl`FglvkR-I3Ko9nNI<QH>R5{5c_EIyG!sX{#@E&i'
'Mviu}0Obs;=1qtpeM}r+P6{m3&%}sojhR5q5SoXyrP6kWSiMt?;Z6^DXd#iO7OK67ui`YL&;rfGl|6mWP1A4(j2E8SZU%!1#gsoysRC$}J|0<p'
'j?cy?-;UV_Psx<V+m#$W8|xhctTd_<l~@8D{DZ_;a83o*FbP#FqQ;^G=T;@T0`Wb{_C(x~jFWeGI=c^qXT|m`Y~nvJ-u^1?9+B>OG&v1U{YR`{'
'9!;h*|B<frel&ShS@m=H5b0Ca^FKhWK;mQep8X~)Gy9umy;`vjE#sx6N^!8H&~Sx49<}hItnfn5^?DtXwon5G;zNPnCPce7fhaqIe&I6Dt9S)&'
'Kq+92i<&iej_zm`(4M0r8|GE51p{gjb5m|#lvwUQ&oPY{<CcF}v9kPRf<D41I$c6oqX|f_9=U{OQW6dAIkpMK4m43Y15#2g@~w}h0`f!|E8Ka?'
'>XP<WkWqByiDK!i7-m!5ntGaadqwqzR6&u?Rqh>z4lK6Us>$JZw-`+cs47O&YZ)=CHr>F-y@r53Y_}L$v)1NHjgQM_D>+?A&zbz@g<5hH!8@<I'
'-AQz+fl}yc<3AH?i2K%-Ck==UrG$6#tnE^1hEh&K{|HMeI0c^wp-erL9&biFE3lH|!vLlmU}h}L1*pyE$(=|4c#F7-C-nT}kDXjS=Tj2Ccf=op'
'`8Q7}r!7Z#;$fsP!8&%Vl0ZR?=C>6Pd^`?V1&+q|_l{C7cen@BRF_a%vKnb?<kL}%@Xlj>KIyYc5G={E1SG-nfz>GT45wBDcVXDUGxsLy!~N`G'
'pSkk~6bG$TkfY~KPGP|KCaUsFiQ&w*r>FNmQa`V&aCG+6KYQlv2<tfpWE&#j0L;}_bvYXMu&QZ|^B#?-Ub05tyYyzXxv6Ff-^|Q(HSpeoO)_k{'
'FpxXYxdj-}c@;#vxPaa?IwRHgLoNG>rG36}E7R@h%xCA4w`6Gq_KUd<c<bX%(BiT7@+MYu^~j%Mm7Awkmp8#W9IKgmL3O{NHJ_r^WCxW__y}}U'
'*%gn-fgG`VHkDrcH-M;%Z|&#QJKSTkT-f&nTf}%!X+axMDA<EXn=pb*=tPoGr#{SSCRq&E?7C~I=94u0U|1ZIgL+ew;vF(}Kl%ZhB(egj%4W^;'
'X6tnz`P>sH?XBI6uj8~*<QtV}LPO62d%H!SBW-pnbQy1`daCnUL6t|<8Wqc^jq>1HwieMUWz}u)@Q)fW;OFKmH_+QBpHfE%+F@6tBSR(KJ*E6;'
'EMu-3WKZqPOex^b5v8k+&YV0yK*P$+_OY!8wl*1IgQyxiRR3CvX@x8ub`N=t@RLPzy(z|t4aPLIa~e+MfC7M!gZ#2LKCn4>L=BrAUDi!Dz&_?O'
'IeX8-jZa4npk=ZQ!#vj~c?T(!k1{!~@=;4-Yau4R;Zbs@dL}aUlx#$phDB1$Y))4Xt=a~ethX4p?xBEtvd^cJj-zHkK0u}%07f)0)i<q$HiWWO'
'm*KBa^A}Ov?8G-iwca?jlPYBpe(Z+-VbGga>LjE-oHV0^uOXv1X-Mi+Bw+4FBj)J@hm`_VszQ5fdc3hK_2xZMxmEGNc#_3-6;H=F)?1~=8ruAk'
'&)cSMzeO6w3B~M@76xZcDcQ{ace1XxmvVtgQA|-+${{~UJ{&*A@3n$jF|(ZeBC2=bh%6!|fom8wkszRhWg|ZB&#C<P+xLHqet7d9lkzyB2*?A5'
'3OD%B9f5CC5B?XW3&SM{G}v5%`5&RIZ9(Udt3;KYQGUs1BDifM*6Wnw!j@@B@Vr>qe;Rb4{$L;zV^Wp~a@smrH|r!Z939*w!ZV#BX>u0e0IBT|'
'D^W6tT&Fg~+Ct)nOIamrwTKU()dhwqfNcX4i)VR-4J%w?zQ@9>T}#k~M%%!Dz7Pev_wJJ%va>OE4kgi=U2-<(61$Jw6TiMbbL*Q^vVL=_)wd3y'
'j=QIa29~<)mdD^$Iry?0H1CEPxHQH(Fx;>KY68r86j;l<hw_c{$aw<HcY_D%dzel2?un2eS3ogtu{$1!J($TYbf`Aq9Uht`fUy<;Iw$L78I7Fj'
'9r6bdRPvo;1>7P{nu;3Q5VQ;Ea2R4N+IiT^ld8mzh-c+;)OKQ<-|^{TfZe7x=w@Tjmf_AGO9yysW2Fytvjcf>fmI)v-kr)Jh4UvxzDpHnV|Eat'
'(~clvasbxKIm~Tu-nOXNyrbSp?VM=%N_@amYZ8kvN?A}d;<!9Ros9=1(1*ZOR=(tm68J!ZPNHs_y%2`?UwG38Zmz&2(toj#Y`|0C@3PbYz`2O)'
'3{OAPzN?_aQdnP`mNqyTvXG$EB#ElI)S<$(ETV(ED0pB`RSjjgs8QDGJmno#oReLjYj3v?(%iH$D|c^XQP~bt;d{v%*S|uY<H~9PIBd$%81>l|'
'DThm~n9{~w!{Z4W?4jz%R2k@=Y|BjP6rikEu~n32h8pj+zeZfTar^<Yb%0GaeOl(Y1v0cMHu|DCb&S}j5^`CVVQ~7yw_TwSmAe9<GVdmMaePwa'
'-J)-%j~db)Hv3WxjTg&M^hj95MWy`Ku{h^^u>YyYf6eOAq0f+qe1Pz;H@oUqli?a<JHOsXT<w>W7Pe`8re+&ye8M68c+=cDI8RMEJ-3caJ}v2N'
'f`3KlY;`tFIgUo~K?~7QWUN8r!-@KAulW5fX(il-f+%8(2{x*8vBY2FGqj>Tp@<4ucuA?Enm(CR;_-G>1G#Lz=_96c8{=)9-C`okBIo(Jk9b)*'
'5mgPRLo2nJH-vO6R#FICr~+pG**Q<as+er;Gqm>)^_GD+QryMwtK!ybX>HQ%!O$rLuyP=}5<nAERRpMV04oTRH;LR;;x+w)W5syvE%{8_%|p|1'
'a@nH28BEa~ATgd!&N;2D@UVbPEeN)AspLhWXzz4UlE8srlGwA}EhrryHbv`l6L0Z!z7lk`8CGx2uJUu0c&5j6bC;2HQurHL#kjQ-fa$PM(K&UR'
'K4sKDHt)%*($0E*CElq+(xTkqh6$h+XZflIB-t~xgM|+tE<}#ov&aFU7q-W$0+6SwLDD(N=5jO83u&pj!-TnwwbULB7)>nSCm#-v)+ur?mq@BB'
'YY(db3WFhgx;O8gE?Z8+)bdc;+kSmI{SCP|PDs)+sZBw(25ehZXNnz^qj|kp$>vsz&^258TQ_nI+QWMOIC%23cXYRzMeo*&<HIrT8>YF;h~p;V'
'E$b}voWhN!+RKU#xk9Yxus_A4YQu(VGb;%(>0aNd@iCo>F*}G=k*>>hT%-uovyJ_MbCX$f)~h@IVes1B4b>WoJ$M}gABP$m!=cCULic%_UkT+R'
'gLVL7ff*;qK&{LPu8Y=Icu$VnRr(i~gfw>-vQh~Sqc5T`EK2=RMY~r=hyT^wSJL#tLVt>s0d-Bv24}4{_vF<~gibk=-sN_i3QynHMS6+^!M(?8'
'#^(b^)<tC0R^=1ya7Uf6w^R>0uLC28_SI(lQ0-<Qgl8^$Q52@j0?`RN@w6LByXT&N=c!<y4HK21Lj=v)eviEqZ=$@E9&7DQtS?l;YyuB&g7oq|'
'M$Vg<kf}+--y0kFOT0S)64hX>vm6<a?|j3{%=qReAO`sE7%dT_(p8z3U1VQgF)V-S-p4^D>%b}zH*htixoQzFro;#2jM8C{Y8@mp%|;{RQI1a-'
'gdM?R>lYoCi*KUVun)wzOYefc7S5~P5#W2FfvCxqcpxZ2Peh8oF`9ZyPz1^iqCHd1dS*A?4o=K{BFz>n-$O!3IBC(?$Ft)^j3esum`C=}JmP%-'
'{Axt+j?Tu99fA>4)^kjX!svas{KVgd%|$=o&8-3<eSRdgM8R@LIOquPJSW=pCcxZQkm*h4c9qSZ&G##5i0gzb&yIq#^T#y36PrkWYbioafo3$`'
'Sp(y?H>h!Tb4F^`W8QvO5jRoLcgnLIWr~HlpiF5b>wUBKTHPX0x2&>hGk$SqU(tj$H8-lPOCN^vCezXpZ0o?Jdu(b3;X{>HSbww&=)*m=-BI$$'
'jA<=5YXyv69$aw>o|>!{eLeF2vAxiOkmQ(4137JO_SNTEN@*3S7#TBGW+P)HDaHfV#ioiFSXXmS<%plS$jw}xb!Kh+QOn#4k<_(h4S}bpv*Adc'
'Z^?^Hh-CBq6V1q~(Vtxxab%?=|FQjnFXoH8J1U&P;3NHODq7c9wwa5@#NQwU5e}b0u$Qb+fi2vMO=>w%tz%uBW_ma0o%HefMS}sA{<I20b1a(&'
'zDQ89<H=2*ua`Q(G;Lm2CkfV!X5h1&U~(bDJz}~m#%~{_*5!U&z)MAE;^N1@y927SQFSSmV1-Uvoa#vRrB%qKfp}$E=Sho`F-hny$%Q>lY~H#z'
'KvkIl@1NPc?`B7sOK?B?<0PTsE?>BcI3!(Vi-#K`_FEJC(n2x}j(KtJGu(GUz^qjaYfsT_ih)9>e@7s7-vfpEV{$<D<&FDOJ{{j%3F8j<&`p}~'
'MTq7$O|1FMkmKkuq{#)HePa$n&SmLdmc+N0M|=)a>Z7unb@Ga8QuGv>i#Q`-ZWh_{=*;p}^wY*)AsDu!!lz<FsyH}Vdh|8AGd(A#=ZkyqIX6=k'
'34KKfL%OmlRF8Nu9c6ib1d;w@GuR1X=Uek4Qii6%xWgg#1*t~M`fcN2_j@oG)qv@T&^JweUrPnL8#JNk^8>uP2lp7vF*x^L7rS^&7ZPp2kS;0r'
'`H)6v)vB~+&&6LYehORPKf)=(%W$L#H{y(rG*$8#>%axWBc7AAg}|p2_%_?tEn_`&Fn3urcZ#sf*p2?$^%dh*5@4Y6gVs&staZ<xB%S_S;(YZo'
'V#1e34fqZ!oGyJw@lv;x3=G+xFt~?tbLG!qGgQrLFqP|Z5Sq1-EtGBC1GDESB{fj@z6q}4rx@LS9tw4<l#0^%%fM_SRTD)#c;YqBRjJ9oZV%`f'
'{M+P(G#7-?m(L(H1`FTC=OaYu5HZ5Px`h7e!r~`hqI7WxOC@OC4g+8jisBZxsp2S&O=}$Prl-LmKLR*%poM9B1=V_K@VF1IE!M^W*RALxMtCvx'
'G<a{eX8_FcZNN_jKW_(KDu(x%6TV)f>)9mvdux+0d7_JbEw!c3LV?)szlLijj>I&xnq<vF&RpYVou5XyWy`U$-YN$zo|oXH#no{&{x10XX>*I0'
'!fxzDg>+b~r+&3%CX^VpZJe=WJr^z(66r=Yy4J;u>>w1YET7{Rkm??BW6q}a+^4RZX+9v28w)P6LeY%E=YZ%}Gde$lVVh7&<0+lf_#Gybhx3K7'
'x_!Hs@9CesFj~aZzZIbNBb)i)BlXb^X7`rn<Dc3zV0cqJi9PE(0f7f?lS=!geYgF#^^1n6j&j>o#;q6^|J>kX5PMdheWza7Bih45%I`2Wsj3Lr'
'9rR&G-J)BMx65SeX--V#dhZq_^0xc0-LU^u6>W3Wjhs+#V!Tja7k3lU`Am^=?Vq-?9QL1;Wma3xdL-sh^=+VcCklD5f8mIcQK17~Xi~-L)?5-t'
'AvrCI8-<*^*Rj}tJ=Yl%VQuCOdAEDGm1al_UqthSPY-ZZ!YH?*qXEPYS`}An6<>RkY44i+`SoCX#KRt);XR2IxO^N#(yl$#dj7er-H+*wO3Wz+'
'Q1?QpI|?YBr+%zRC1UJ^2G4kmfZpNfye=O|Wb~2n&n59_-o1GXza0Q~ysZQBe){haR}BC9@ATDttVCU%Cq%}d<~`>&&@j?AuVT8EYu*<#wqUd{'
'D`Q-|eP@YURXuzFfk&s4p6*c^`;ApMrp%?n!GAVB#9*k*<Dum|jk!>h@*QUW9LC{eDtfH$ZHv%3NN}jp!M_7El#5^2e6*9~?Y~bmA^10JMw|l~'
'pV6A&-+)EIzX9!NSghx|4O3v%jQ@30U$u9(zqd}ESn0b5C!{q3qeiNLKf29Uc}`NBWuqQNF||&2%`8W-zio5KOSf&^tupL4jyo!;HJz_O#8ll+'
'=Wa9lB^-{1K*F#1*KqKL_AI`^(245p=~QL!-02@W>i|U1=TFqlg(a384PJ}oG1^pg_%KC`)BDg|KS5Kpr^FqdvOVj0>#RDu2En(znu`vw0?;9+'
'Vc3mQgmdL}p#pkJO5=XaY4GhS{xpp&*Lk++SQo^QlAt^cV~o-3sDC2qvUlc`3_9&n+mLIdRV=AjjBsh>7o>X4xPAAisoeOTz8!%6@rR>0jzYlM'
'%eq^wb`-wns9;YdXnaT=Bzfk5DPEjKKd#jc7h8(o(oCP?#;OnZ?U47Q3_{Cy{jS$_ZFaFOm+{UHgQs_oF>k-|@t}&{HKl26^gU+gDTczJQlHN5'
'og9yWhMV3kKM$6h(+X|>@1TR89TKYFT8jpl=icx6#m%nr{}gYX%&b&9L``<ZlpBH9&bK+J->)@0KvbcL8>R($@-hbpO5N-a5qr*;8`1T;YOB^d'
'=%P|w&5xOxt=XoYHy|?*r^W$L_xN*O-M)#xxr7ItxLc9xYyPoJ0Jto<6breds!|-vT$`ydpj=ka6!Yq>S9IcmCeQU@<DRfZaU<=V^*BiVKI;Gb'
'(Bz~1;I@|iOi7brZ}Ic^hn8R1n#2PKRq#eM+PExXnaU;pMoRNzeJ9OD`dm7lRL%X+J!-PG*RNjjowM5W(8(QrI)ZMO4g0R&@RbkZ{@N6qJFUuR'
'zH<e~LSl)gK2)G~VimaPPJN)A*kH4JVr`$OA7(`rbgp7QH}3N-(C&>wbw)chz!IvX-}DP{;q0FM_hag^!FrzI9;}*N|8Hih(051<Tj%Vpv3+wy'
'UJc8~<b8e7h<#%!E=qIf0gh`9)+Q3qE}%Pszb?Ma@@u>?cYS(xVvYeE^L+l}+|ynLFB*NGp<T8wC@J-8^{vORP}>#Sna~}SlQFYg3H{3QE%3G`'
'-!Osgbab2c4BhgaRVU5Oc$%Aw@Ra+ZUegzRYwiKXN}LgLNr06;m-4n8nR{@0jB#&;tF^;k<oLtc1Jj-ATG*N2@@Bk=`p`3C`-+kYu>pWnE^O;H'
'-3Zl%(m7n+IVRg_(F?`B^kXv3^_dv=g@1Q%1ucZJj$bX8&bpc+QY@_eI`wVfI9XMfp{LEL=c{~E^T1SKoCP{B6cP@U&kq&wK37Yxy>2sYSY@V>'
'YE;Mrvlye9yu>;>(;CJ3{{T=+0|XQR000O8003fLn}w#+&ujny^q>I%6aWAKVQF7uZ*6dIb7fy;Zf9w3WiD`e?Y(Jt+sL&j`aNsqKOo4>C24~c'
'CCiS(4m(G-)rnp_9ZTKm9JVe4k)VW_Bp3kH&^r9@Z%<XIDilbm<Ic@_>oIAI1Zu8bd#XKzVfZ$i#PjYf&5JTPIteD(VwDwXnP$r%Sze~gr1#5T'
'{PGuPS7{L}vgvxB1k+@mUL<*3CiA<XNLDd^ErRkYPZFrVoFw_O7+|d@L3U9j`E`uV4T9Nhm1h@86cq7dHBY9&ERPpS5e4yb8r)?0v^!6(lX+0)'
'@v@lY=?bfTh0RRUJehDKaZoJc`8=58xN%5O0h)zD!x-YAyjwxTud7*noX6Zoa+^%n+(v241}B`(Y5YlW9naG#%w`&_pjuF7Fth-A%Wk^!w1DTo'
'{6!Fa^UY$FTn2G|c^%+WS)e3%`}SP`4KHC5d2pFtCrhfHU!*Xj9M<Y8fq|D<Hm4bLIha@iMT=mS<cqW@V1U2;1$2Cqr)7fmrMp==FR|G?UR}w0'
'%<rJ<FbseBi&>s6g7J8^F4uW79tSB*KQE#7GAn8E3ml_*THGxsX{LW)PW0ESxVVDN)xXnKJe}rAQRpXsFK~SFW1d|Cgk0+1i@3b1ex!>8ccZGC'
'RbTT&e=V+HSLW5vZ1OoNtDox&*xiXagyOFHUM9EY4J=9Z0D6`6SixvuGHNkC(%|R~3=4*>UK}m&qTn^`a(ppYb=UbEs`dc3i$s-!XMfMqWrSbo'
'1+FQTj?*!pn`B%}u98Kps<aVa0ytL;kjbayAM2zjBj+)~sCrfuNof~*6&J~Aa=A#B>WKs1^7M}g8tKG%@+m2@`E@dV8{Z|lJUgA;9$!~YeV3(^'
'Bm+Q{YMuI&uF7(yCiLy;@u%_8ACKRkMFGro@)=5H^J#Vi%fe?p83Vkm0OH`B&*F5B069%c_*T@LPIJ}p@iZ-I2*Ct!LLQH2Sw5c3(^aHS=s1BQ'
'V?3Ajc2?Ouo!qIm_}3WVX0a-v%o3p_0+^%Kpmwr6jptIW-rPUdfCoz5Ub-w+aNMc^MEX&lET>71fPiNK&gW1air!^XzNKgFuxdK&VA=0Z-W-1z'
'AAf&#{QmUh!~0XHbszQ?tHUq~!ro1Cv7j%D?3(|6mcNJmH@;5kZ&~o;1NgG}<u6X&!N$~D{rl>YO8p&>2z^;CFI6`y`f{1h=<mg1h0Pp&`{txR'
'xSLqUuzDtAiRmxQkp9l*@q`Blf1#DrvkxEt^5N4Ps?MMm#uwST#6s~5P#Jx>jMF83nHL%TT~6swp5UL0WD>6n`I;>GZ;IoW7B7<WMqW&^^_=SB'
'X5!zg_?l{_%aWR2WZdW?D`*hea(;(@%d5E54FVBJbAFmE<S$o)FNJ(b3;vs_2YI%>yuz<*ga-Tq2kK5YdP6<nUmog>YV!t;>y><&^OLmX2WmEV'
'*;<W<CYR;9S$IYB%x<`eJ8ob;r&%*_q-9G68!x39^_PlE>$ESw{KW==^OwJv%;TaEB$FKHd6u_7t(S;d=|^Y416ajH!4;;-EEq4dn|3GYejOBL'
'E+z5==+TmJau1%)@Jl=V&0o8}S#*ChJ^Rg{2ETbX_|55m!4Rm-<$_9dI!3dL_;y^tFel0Q;tu!>?gKW}?gNUopB)5Gg8lx%q5Q#PTDG$Sk7BaC'
'Zih!F<JTYFef)5GJbrid{rL35x1U}gk6-=u?D!Oh03&SgzmU35o;>Sxtbyd~WgAZM0zh;?bbkaD&Uxgpego_*3V?_t^8vQSYoW<k57%=J1DSs0'
'2QvOLzmsqHAxyLYL=z(#rD+!m4Z#4${7Qy_LsJ!@^m<;xVXKu=gVrOJL3X%uayv;@WpF}$(EbiQts3(%)&bXS3M~$e*B?(l9@FzA&+E@+2tz#-'
'A|k_Y;QS{tpmZkP_IPV2z~TkJ8U}r1Tue}Kwp^0C>rE3Z+z!{}tossnG{NZ<!!S<(Yfk|Aj=G2aet&Ro%rH)Y!dtLD3-4)xHblP?*zJ4nZ`9p?'
'fwebb$AUc`gaImwjU2Dm7eHagz%aDozX~kgXUjyNBH$#ZFcrmBv=I*BM+c8gx}wt{>%wJw#A74@@OA)mfSi#)6DEMaKna2@r*~2s&{VkunmX7z'
'>F@)f!+^&d{ORoMWBardeEaDw@YFf5p%Vm>WU3~PZNiY99(uqh1G($;avpiu3#+kH6&fya?pK0^^^YEXn`R572%8;)eXD9!L&O=o!?e$qu(EKV'
'fFuEPy%-&ykNW2U;M4$r;^>(BM1+B*)8#B{V>gkSD)X^<aCrDEB3WZ7HBLXg{`2_s?9=hlyUxZsS*po6C2$1x8{cf7;4FNN1b`+bYzzEdWGE3J'
'gGj3haEX5a8X0XstK29~NG6MO((^X@Ojoc7n!E3%tFe4-%ioSsb_OMTX#osX3Zh(s^XCWiEL&Z~lh4MJboqJAqM3dM%^_bnt)494xTKS~D62<n'
'VyMzt3g@lrx@rGERwG$v%P!F05}9{doJN??2egQ1$rvQ61i@Ax%?R+yEQYa-i}h@l-qJ%@Ct!^LCLP+wMc^&U6>WWZ>ENQ|0mqk0s6(L-f>e~#'
'TVn<PJE-w-{~fqInXyEA9QN2P1f?ET^S_D@p1)}0I0rNVXL8f@GNBVL0%1=o$SjacA@Qo0t&(Ls%r9Vd0Is2dBwm>03^r(TwO)S4slpk{+w*vF'
'F^vaO6i|FTZ5t%J69pGx*l|FBhtpfH5UIAQK^p^0fve<}zqMtvNz@H!$|$4Evv@kDXV#t<Wx5m<1+GODJc*E>V19$>I&1-=Y*-Nd<DzgI%_vbC'
'knwF3iG2_fAbfs{8igF>L6G>Tw4N3oP_rs(NMN$gK{zRyu<)aKx=8VXo*SMY4}-;}dF^PcX&}8LeDHdnLaS{>$M;|Mdwme}GSrd4NoU1GAp{+#'
'IiU&;OpFm8@_2cfv|qTJZHtv{nMcPi<R!8GqQ*RUEIo$3cKFBRGax}yD+;av7876>?!#{ZVRer#;b}N9MBWWeNQt-3u<_$cz-`^OdeW&^Vt$9-'
'!pP!sU5tt4L0b8&zaIt9`Um*WGyLbb_|MCYr*GO|K)j^68&*xqtdOL&LscO3j6=qPcu5s&=%xu?KberJU;$AGG#b2BW8(oejlcq1RJT-Xs7BRP'
'6Xg7KxlSIw!e0*qU<Vr}>7~lf3=#LhsYG@uxc7Cj@l6xB)Onl}&*$GI(pW6ZE|8~75C{D*qgz2$BFZ`HK_~c{Px`|rX3q=JRWfCTC=6wmUaqkp'
'L=N1rZ$xEQ#>kNOee=}_78PoE0vDeFINJQ97@p<p1eo<(kg3MmXZq>sfljd57So;^p*?_&R_mrYGY+t*kzV>3>5`y6lM-zKexB^pVE{X1kt5#+'
'_KM4I-xl@xdA3J=c$IpjTy7jW1GW%Y&v4GN3~0#mPM`52AO(h|MyOn+!CFk0rocai01t!b`;K`3ATpe@Dz*6MD6wkg3ligPWN;@5d?5K#Fix0;'
'XFwRnA-IBI?Wen`p;LjWDj#Nt0{fT)r;K7K+6TZ+09Vm5-WxNr_n091u{8G!(c6ED);^5R{v0&+r_onGZPoLpfs)3a7qsm(@lR5`UDU2Z6&_sn'
'#nEX4rTYR@?x#{XKV2iTCJXxz!X_|D4Q-)O&1_b+I1iwV236d)ed_b`QN^7U(WHj>nZfSN=Pnx3%RZUI2-N}jg49!AO|yDMM7Fj_ZY;?w^&%2c'
'r5?|+atv*i*y=Dal;Q7=-<%wc-+p*~^j2xY-<*6xom{9@k6LL#iYTM0hv?^6j}0&h>r40I&_=REJ<as10Qan61f6Q+O1)xs-RUK_D?kJ5f_06o'
'9z(5ZIw{R##Y>}_Fc#{;Ef^C=9TlGDj?%7jd#vA@!*K-?RAw|#PZ3l#wL3c&%@|>@SQlk*0l<y-{z#jO6iC$e1w-8u_lQ`yjhi?w(jP(2@d^14'
'+Cf=Db-1?OpV$f+u2@sTl*v4~%;SaT1=~2iaXM{c1zjGmu3Raz&&kq;?6c#y$AA2E^ltp>!`b-k!=I1eZ-Hv6zhg1#4Jvr9P>ouE2dPaC`@Ozj'
'J7TbG@ePP3N@sy&a>g3|!QNiHN_%Q8ds%+DcafF%G?UF9l>CNo6VWnffgGGHGO%y*ZR24Qw^H#{e23d*5^%HxF^f>s1iV94oQXSMkHC2DHywVW'
'i<(GT_QR^0eGc_$(v=>dA3l?{38tx1q7zukaWcE+Rd&^bTb3GIFxUHD4{Q@!5M0wi#uJhp6zUe1d{8$JA&g@j00W;B_e!ODCh9l0lV;jPFFciH'
'A&imZHY4`k3j`^V0=#JeD9IVOEZ+63S8<mL>n3xfb&V?&im3Hb$0kj6{hbxN?!?|_Lc;_P7VFgtj^orpc1GJ3!6s8TzKIpuV{GUY2+fEv-qG^z'
'Tt*@0ad_KGCQ&^y4}jCc&qEt;$<rM(P;DA=Yc+Ve+ohZBs+o~C(+j&~^Fh~E_@!H1fgA_eC{L3@tPvJ*Ih?L@ntX)RW*~B4x|}Ar5lU8*F<z8K'
'H|Z2a`>Q0qyegx7OtybxG#!*ecoh?|WuQF1ikC~+CF(jO8x_Pka#PQ1+BrAm#C?z*t6w2+9j~1{_udP_DT!eju<ag+@9pYV8IZMetK2bAmYO)!'
'o<U8JN}wr0akMlh(o=;QqagvXbQ10F;`y4lEeg(H!Q?9q>;F0Z@cs=AgDo%~rnOL;uN9T)lCXjR^#U(BVZb$Q1@9nUlUDPQXFT|kT%cbAQ<6ck'
'WpZ1#+jNI&U8YJqc!3Cyl&Gg50@@l5gOC~tJG4@QlGoGppKg{Kv6@9OQf=7pL4H$&g$A%=DNh*zh@8DH=(drk&%<$0!o2~9dgBMWe-ljc&@fXL'
'4p_9^0z_C?0Lavkgmk#1J`IbD^J7nWK8E!Q^4wE{3X!H{0@aKv)#(kb3bR#V7c|ejT@;pe%=<M6F0yRirXN-jdZN3hUya>wDl7~Tl*f~jTet8q'
'Vv#rpDoZCf(pytK*u{&^W@MOKN7wkchOP18&httQdT@0eMPo_Ci+7Be5lPO%d!AD7*=$3POMa@SreCBs5+zhmc41w%;@ju%g!R$prBA;n1G4#B'
'O%~%~{{^v_*oaA5O>~SVSx}d)u=L4$61?mLPZ3ww_g8Hf9+nDYtG!fa;XqbNPEV%phB9-pV_)a-@^T$tCVack7rQl%i|o5>Dy_(L7@|g$6b*So'
'X1G~?*Fx2BHYd~ZSzMG!-UV<dR!NSA?VsTfAg+rH?E|N*VkvrLxs5HN>3cEo_6g9XkS)~$As9rCX~DO+?PsvMi*(uEf7#z=>~{9)=>6&IPbVMG'
'PCmRJzy8y=@BchLJ$?-|<&<{#pdYfan~8lD2WVO2lN~6RW5r2xL7Wr`31Ox1&So<d{I%A*PkR3LJ`Wzv$9t0`ooi@;K}Gh&Z8bT*Fo-ZNg;l!R'
'{-7(J+Y*P<1d@f`z9V2<0Dk{mJ@TOx8}<OanH`WGBd42)*Vq_|D52I%w4Z5{8^H3?qa|$O*$62VcSVn#9FRKUfh7x8HU_K?kDOXTvGkq0?^I_4'
'(SJy1zRTVZ$zoOBh3Aga9{28d>V!K>`2>zQ9duilPbWwpmA}Xtu|OyEtNzi^Ls{C{5I~@2b8xbtXkIc#>+g1gc7(8~Lk}B=kcn*|TpBGpU_k80'
'kXZLNzIT<>Xiyf?{)gw9A*>h_eiISnhwI`141<cV$z(b8Hkgp9^dp+%6B(~oP?OrRSr&R-r=0Ag<6znvQmxP@o#=G2UZ{J!!<Synj2d)Qf^0{&'
'%rQ*U+`uzDqR=VuU~T&Zu&`Lcv=145%7~k=5Ohjd3N2miXIQIKp-xoqARvxlW`te1t6{KD(q_;$un`n_-hoq+LQ8Ef*bVkO%;{DblHPRAjmhq&'
'6#pD!!qC87RhXN_Ii6t&LK==RS*{o42S$1=3pI-#sM@|gU_|)WJvBAxA5J&qoUrY3&x7XhQ{#WGfXf_Tpt#6_>O8w7S29O@Ra}yNR`i^<@<+1v'
'*A;toO(JPAiRa1i@MV9{J)p<`;4+8O1Zh}Y#EGBZ^*s;KR2l6hmnEr`c>z6ak^0nc%E&B_;2uro@w$wj*eYS)B@)d~p@;hrR|Poj9C?<uUZN;?'
'hO0pev_UmLQ=cgE>@od8Sg*;d_+fV$U?=h=VCA-9?<$j(sRN?;P48d^K>wQ}G~f@<_QXbD3S{+_$z-0y%Q30x+Se#u8+Q4wk3ydGiuFZ14@dte'
'?*7>C{;qZYl(fcCKvk6)r&1z^t<b}ou1cmhTR6HJegp#9?>mFSM$qsS{dVw<78kn0{Mb5cw6#jx5|Ei5;&s5TYW4zRucLts@7!3(&&eIN*~Sv0'
's&{7bvclx(79sPXr7H|~2@^}9`9yrfcb2^wU<F_`a08CI{{>rcE`M<=y65)?5uQx!U+;tgQR))F+Hz>zX7;!g9tr*?Cb*TWQtg$D-v(WNs7?x%'
'xBHnZcq=++W2RQBk*k#D=PJgs)0&Ud<(NsqR-m$lo&<eV==&}LD}9fM|1lXM4Mfa$;LkeW@iq5L3+I-z8J^$cm`s?ufep`Y#>FJd|Cy=LH)WEY'
'%#%q;W(MB1lcuN*n+mF$bvg*6H}08;xLZJmN35E&al|^-YN`&--NqQ)uqb~^m+GYg*mI?xUl!<-$XDxJreks@Fe%_5XofPm)p|K8*Gf1l){ExR'
'DwmB;51J|P_Nv_u|9~%lh5rn~hWTioYB>WEyacMsP>fTn0AduPA~L>6(W^2%_culQJdk@}2GEW%OtUT!pEmhfF$5^asOn!PS{o2971<=zUl60_'
'j-U9oe($%>Z7M<35Lw3O#0>JDApdQy^%JOg_#7Tq%|^kYK`v&1>CkV2oJqIc1Ci`DaIOoYZ3+WA6pnwe@-$*6*<qt2tQv{*EB&b3bUNrPW9!=-'
'd{)WY2#RD5PSr=1;`3bbyGrG~oJvczl99gufkIl_0pL<AsniwMaY~-s!qS8M?mG+c6@bYZKR>J{pxdgjL)HB6hpLgMFqs3>1%sWl3Fr-H?Y7Bh'
'4h8kdGZ^47K?gdZ0wA>x)#X29#ZbYub4OKNkx9gFS-McOAcnCNU3!jJeODP^UIyX){hnc{Gyz0Sw}J|K%JpiVj0Clse=LE5s;=`QLotjdQaxwe'
'6?yx6S$SBW=dM}5>Uv!5KCOUMeM(PohLNmur_<1aroC;h8`+IVKH{SKamsdToyceym<|w{Y1Q#eKYwl%UM9(uNR3y4C6Ka5m90*I9(|^I&<$X{'
'b~URg06uHT4l;mgZjkxD86D7(xK8pSakW93YCU!@O=gkD<LWwQXtYVZ<7odJK|e!BX!Mqos`&M5^?0A1=;((<9XxV|dgoC@7`-E^)~#SiyxgAl'
'9PL~)K{q>x^{Cc;G~mY1>nq#CO=|nLrGIYj>3X?j`>MWq+DyFQtUkOND~teN32F&i&E>0^n4xbZ_apGtRbAH>uqy1;X?j);KQYp<xp6|P!Knem'
'LDg+{+|}jzs;zCtg~seFd{}iZYcZsMD;z1tjAJ3Yp?h}KHo)OECep3O?g0}XaScp<5<=G~O=}hyk5x0!gx|pPrZ(MFW+NLz!duk?%*A}d-B?$n'
'GH_X#*?K-7i!A9CQKNu_PcE&bJY=7>#HWbNMV<oV@)Si<RBx+>qu_<MdjcXfR%YgclFVo(CK5tD1+g4GGO0*#&~bh@zD^6+wgTO~!)!JSNw82u'
'KpQshWsdPo3VNvqYV07C&hJd=S(O<Lo|&iC9<c&i8xqV3cob38)sX)6D03K25GoFO&kgp--Ro|!k1~>{1qD@34Q}fY8+F@4I`z!o82P~CN)N*o'
'FpU_P1>__7>MEA#JSpA;_6YFc7|{C|O_K3$qRgP8uhK=bOlB$c@h<+%w`5X<Lqn~|W^MxfbV5dQF#dqu^RWmPU*+inqa`ilJSQbs&(}@GS`YiZ'
'7YDwzyB3Z<PiDn%6zY*z2SHAuPdkSOm}5W@;|T`Gf$GHyuw{~uXBbET<Bi0m-70Fe4G6Q<0`69Q9qb<v3?s2qFp4tuGY1_#1W@s_16c9H09!v9'
'&jv5feK3O`0b{()^+7L5J1?6mZ-wCBHMRHRr5A$nV?gmmR+bp+#b5Ath=<Mp?hf8Pf7bAlj3$Qrjg{F(0$MoyZ9@s)!AWjW*o%EiT=mgM=V&?&'
';(+vAU8Mo$+8G=Syx_Zu?}YRIoFprCbOc5Ts1$q#fKyYDU%aOXKh{Zl{-S9gozt{Prz?P<@`1+*KeZ#(c!coNshui*aFu1B#SfbwTyoeeMRJTN'
'SPsR}M9UK0kfoTg!h&h5N1=+!;nh4X+j-dj!|#Wq-mm^}-ua<G1+i|lqwSHT1(V~W=z7DdH{_mCKdXBuSxvWP8!8*KriMm?{pVbt+oihIddO_4'
'2C4s<&@M7u>#<rbO=2Bfquw`T8DoJPkj!U*gJFmRXo5LLDb44z5o_%C8Ziwn+z>n6jW~dKHH|D@0`<YZ;PlS%RtarOo2PL%Plh3em{^ugg=)N?'
'abmWuk$d8dYC_5$cy82JEmjH8A*tCX5UEk7Veqoo-v&J#cP7axvKGrLK&eC5Q4YAq>1AR;{{A_@@5}zVui)5JGI_bAs1nP)bm+j93BDXWPtX!#'
'-MGyR(&juK3)|KjJPi(d2U|g%i|+<@A|vB@GHV>V0rRM7i03=tT_TPA6oBVJ{{)cd2ovkoBVeAa$s6Dom1P9J2EhJ0I10q{#x_V{Zyb^rF=R|R'
'zYD~hKzOob8gK_)vo+lr>RC}gltB&H7_>6UvsEupW@PAx)G7i%Gy#iQF~~M%OBmy3#v*M{rX}gylCLCyXu8C3j><%R0_`t~UR9e~9CNMVHcDnm'
'disJ8C>7Hi3y&C$IwxKTl~LgTQ}o4{*S|ZNF^2lk^M-)P^&WA|irMWrxh5?Nb6_gMXoJ8%yY=xN6tk)j<3LB3;3#Kwe+q_FhiwLo8-R=gqbSDZ'
'kXq`As^&VIuNTQs-(Jjad;F+U&~VR4tjPM-z8`n4RSf8)RUX>P5;!&otS=39TxCEMteiK23;}tpxt9lP_Q`IwYDFEaq55LHyNEF|t>}h^>Wfjp'
'Uhtv)(<oQ1{jmCKyjL9kSo}QoTDvcdI_w0S<tQ2?0EPB$hu;W1b42DfE}?`4MN51RHKv-tb4^l7aDymRUES0~pILaS6v1M}S{bz6J2+v`JD6>X'
'dJD(u@=O35Y6^raH6=PDZ1&VsBm$^@Nbx1AFRNEx9oWX~Qk_QQ4w3h$NaB2Qh3dLCiS}Zgfd`>7cHxTwdSTdRgkgAcxkQz-ad){l0Z<7rP$|Y('
'M6M2qVV;0o$(GF+4>90Q6T^a84oF)o7A%X7A8wGo(D-`TKG5=Mn@oFgoE<ahD7l@!k?mB$$k1E>)-5f^W4siYX<FsGGJO(Nnj^Yj9k}P#4(1if'
'tj20qLkjU_$%yk=PkO8(X*lo4PAkOomTmVbz-ZfKQGXqLN863w2I=Jzd4NJ6<vrTtV(;{u?<0<7H%Eiky8!TBo?%2inQ68(Tg2-HVFuQKB!kms'
'yrMg-D!RE`au;xbVvH)jSWB32ycU6;%;wMy>oVb(>3~Qt6SQ?FpKk0(F+>t#(^Nx1xk`uUE|<MWuv!kSclNADf!?)KW81DSZ>Q_Us*PWV>?r2S'
'0;u%2yA>~90tb&6t(6W0f(p413_1o|<{>cQr8l21(AyE$%5R(h_U*gzx1Zj!9fniv?c1YwN8{7uPv0GXstRfAEgffbFSJKW%kk)8axDF=1AX`5'
'&GB0uwzI47t}!Mxj{iV|!*sGp%ByT@42@&KAa2Ql0Rwg+odBUBjnEy3_C*IEW4M50j<62!&g{Hq)n%$c_$oHzKJZ+UMOqfYdMW-|Vic~ima!WC'
'DIdLk`{6I+H^*nkpWdCkKRG=+c|HDg{Q3iI$6v|+5d+zT9c7V5Qn5q6$W*k<*Q7JP(`L)~*;spq6JbdSy}pvXXxm4|pavY+oKd|&d*LA(Y#VPK'
'OeZv?*+N9t`vcKxKuYPdE8%;CKYsi+Kr)QCP{^zOIljD{b8ZM#^Ms;25PjqNfEKb<*+r8k4v}pJ5ctzK3{Aa2+A;);1z?#%V>_g$G1)}{4#hR5'
'=oxCW8E-|lHq_wE5N)Y+1@Vt}uz1Y15T;t9gWX(GEOap<ctFvFa`uQ^t><&AwHVsS0U5q~3!8?#CveYqEZQxSMV8<7tc5g&Q8R5;`3i7S4sSsz'
'3Jgfli3-+b_6`C2n=F3~98f%e`!1rVXJ`(V{)mz2RVxn(bwvlj=RR+Cu6DhWbuZR3?mc^yc+@o0Rr4yv598@{YR7BvK!q=01Fah0+p2aRFc(kb'
'I8t_KF(XT<C4{kAOWMO}z^O`LvgtCJwpHg5Y!a%pIR+T{CR%~4@k1`~pKkzY3fGK_;_6R91OxeN5wF4lI>D7$xUptS>)e~Hr!mG6(W~2bO)HF!'
'Mj=%iPstk=`{72&7G(d00V4BS6w8M;^X$MHW#eYP{c!YVeE90*Ort>?Th{TTt3s@woULc%6=3|M<@6Q$)4nDYVWKGSH2<-V0b%_}h-A!Yg{E6?'
'szDz?t)<KH;YC{d;-9I{#qxs6P%?-$fDA3|(Au4?d(J$HwGF$vqREjcG7Oiq!^W#8RUYK3@ibdQcU-k;KAvbw#-jFC8L%ZL{6ri%*RJ(2R{?ID'
'+yy8gcX-%^kp?PMl7Vx+0{IXhE!lYj&g>*DNU9^}fy!e5Kg=Qwy;yb2nSb)cm}kcXE+pfZ$&x%`bl@#gsTd^L!;sNnfW{Kh8wR8L?C<@X7UtO;'
'eKf-24)Aj0MX0iyfszdXOr`#stvS}atef(0XDiWO@JTqOWm4VXo7tkmEp0QiEvCJ=%NWog5(Q>Vzk*ZBoYQM)#;28YBM{P3=|#bn=fogt_Qqbg'
'Rh{byG&NZ!H)Cz5jkgZZ_Ybx?*&=m0`hNWW_%9qHh+S)6_Q|)lCWu_cMO>D7Taz8kda_#0@5Uf+p^467H77rL&tV{6Mlxd4wR@hr4V$e%tw}RE'
'osI>H5g+n$ZNqBJL61?ZLCF?YUKOQ32%=gAnGt1cLFG$|F*F;(!Bw-DZ@n_(^Vwr++7xk4wOrL48Qq=xYiTcM?m?aJEeGkz6E&OWGB$h-?Vnrf'
'O|vmaNb$zdR3Fa*cu2kac-G~CEBg4g(S7|pt&JKtaiugRy!6mR#n|_&WYpb1A9!FyxlZYWDA8hcqft<NPFG{fZV=Dati+PnA&2#IZPJg(chovN'
'_Nr-*Dk$Bt;O@G5qa)VmUoE8XteI>2zy+%FfhyD->60|!B_#nnR=C7C3e-Z{va%J#%op78*FgN{32H`Bk4*}O?Beenx>0P{U~!XD+2KgX8LGf8'
'(j~Gf=9L_N6!&2(oM@tkAX0yOIq6NZIV!K+AghlM7PuQ-mSmyvBnAT9wiQXfRc%yDxXV4zN%cmODvAF6*Xc2iiqE90&~Z1iT+&Du&gH4P_SGIS'
'=x8dtRy7G6P<|Jn`hYb~GSWS?+mXp|-VwZ2!vrVufHEaz?BGS0H&tuIT@{^PaEj73f{m!xbqK(t_iq3olex-^DcVnDwq{8(MZw%yH&}W!zsF2p'
'qOsRzVQ4{StIaA35BxK0c&AOei^^ew=ZNwB=!+pqQlcflpC^9kcCSRn#0dVek4{mwlSaA+6y+j5BVFP1IZG(2p(LwdmEwiiNA=V>X99~bo*HZJ'
'&dZJS>lHk!K!~EpCJhn>P9z1Myk_O<m{P)wDNv$x>T-hkfrA@bH*}$LKb6)V5gyuV$BZQORTTo!V#eiSA>h1~4Y8`dDtkkB-V}9TT~iFbEi`HE'
'{f!|A(P6-aaVCnKw!R8McP=(7#5{Z|y=5~$Y3%BVIzZr6LzgEWIN)209oH1n?zUBRZit%pT3x}xxr%#i;ZBE&zyy1??`)f=Y7C%ug(xk<QR~so'
'oH(P@N7=wE)73Q(^pUZgKjy%^Wb0ynr?4O!1L%u0-7fhbV3dTmdei_g4WFk$_*F1L69K@9>1>uzICL%17Hc8wm$^_4QeP*P>riDH6m64vJs-;M'
'sH}VmommDLz2G$>VlkCh{;YD$VuO11cBF6dzytMW-S~)D4-`ReoRwF_*FjTCl%~!5q%tCiy)({0d2*m@75sBkt*1RO-&B&@mGaVR(tLmzj>cOc'
'?(?@zB+tgcj9G6uyCoIb)|ObN`Kl|MwQUV;$W__5z#1<_bRp~`0E+t<Rx|Gg_kzZHTEW78WAsp;`a0IzcyWz%@d!%tP6kz}{p)@4VGJW3rh;7>'
'r8D1D8$NU>6JS8~1eX+WTF;&GWoJu{Lq(=o4Wuew-nH4Yi&xJyr+(lHHW3QnDjnm!rDtV0f(O|Rs_QFYYS6)-{4IQg7y8Tzj?EV;K&aVs6nMjh'
'dIEzQYooO-w-Xf?r<p7&4pYU|LNZ-o)gD)Dr_0HFJw+cP5bH$d4%EYSJ{NIui^N|4faHg6NpUerQ`Mwcn+{QV`Y6SQF@u6<?W<3dQI=%0P;sgE'
'@?`_4KsDse=Hv#lQN}A4w@xH2(k$?AioIu|la@!39<8Sf4+Gw|r6jzy7bX_YQiB2}{9*a}!>3QjuQ3W>@agzJzdiYM{H7QDHCyL&Es6xVl0JkE'
'SJUV<k-PAdVl5|<<5m$Xs85WNsC$Wyt>O~SISkm;`ZhNGQL;zcG*~l|rh}kOfY^5Ct6-W*0PqFybB*21fRiAZVgdXB+V^Hb47{T~{cq61j1Ig*'
'T**%a8Ac4!-_(V$RunS}B(N%Qih2S7h%5kcCNPNPI$o9{hsu4Ixdct!=%pD1ef0h1e%LV|Ll|Hi>qU^w$SP2IjA{e8qIoOMFatITU9Ud8`K!__'
'sSphp$qa4DG%7?1BgxAmGEyO^!p(iWX1GD`a}p-e=!}?tIj27?zpIXwk~+yFRgTYSfZ<BdgQtyW;A0Ka+l%-PlP#1v$il=?4t`jM9KQpFw$Wf8'
'(*}E(Udyk)P8~EZP&_T>JrACls6o!ZBBfs`Qp$7d?6jw;i<s3llQ+sZe#ra_ec-u<_nMufUJ>+gJecrrfBRdFTBw$E_^kf}rP_ZO;D9=R_@Q`W'
'#?ZoW1o34tgu*AC9qm9>Xz72M)r~5Z^F9hh^w!v&1z{L&jN}Jw@4olshmB_>r}nhDE*?wz)<d>+)$VuDW|?D%Ir@cKPzt-ArNiXSqOBbmx1N@E'
'%P9>44{ToT9i{wHT~pl3DzeAJ)EJU=+K8E{oA8hzPpkl8OMLOko4F8U1&H09n9HLxT%*wA(JO2*1;9}NK3>IXUS)p6UvxBlCII|ea#HD%Z5PtE'
'EtqQJC*Nv&a7F{#dTZM`J83Q0Fb+3i(1ckjI5dUF)a#Gb40xi^;5p#7ji;Vz!DH<bsi&$?O}sV{;Tso0H0PGllhNVnV`A}Hvrc@2$!{t4nEKj<'
'6&tj2N}uLn#_h;541p>o9Q{3C%+VOGfX3%tR^fAuc49e{IsqLzg&Z2}gQZhd_>Q8<R$8K}z^{Wd^o8eq?1-l+CL+e4rwe$=;)aSXVogqQM656^'
'0@gVL7lZbXW|M#0{~PZ5-@^UyZ$gZUBvEQ&X-s957QF|vl|5(ZqmUJbW2r<2GhuPn-`|VVt~fjV$9`x}=|68U))}!)WHik7nW%h>=P?E2r^v$s'
'0XJ8$n9Q(HUvQ2-7r5SLld;zBBfHQGEPhcbf7K<;4ca9y)qK$|MJz>o7I+(=%~X){H&}iOA%{+O!JVcsRBPl^Ij<91V?o`*NZIh~xtut$P&U=-'
'<l*+v(fUs3maYw<uc1`r5!b4KCcHgIXqHf2LSWwY0CfW(B*0NGvo7u$>=T?X1U+E8NnVgEF5*XRqPI;2>}ZzIGgUjOILcu7rL625l*jH;HLWxa'
's9|}lCDw}Nk@?%<D4Vxab>e|IUmw19oyMXOpgljCZD`Tn2jv=XfLv|)p^N$=I<y1ZX<j^kvO#;5jELB#Hnrh^QhkoAZIwP(0RU<*&L7sf9o=AO'
'?@n=}d-sjgzn=-hJETf!9OBMkYfUB})R9r%jxTP%@^d$BBSlT6ifW#((;3CdGOmvKERHdvbWvCPhIku6N?CULRGL_FL_qK6@+##}mj76jDQ?Wa'
'w97VpEh%>eW!6QUL)f9D<SG>Ec%Cjv#d{H>uGdxZhT5x>MnR69LAVBnUwKwT)8sW+jY&P1C?&NfwF@XU37k*DE(KHi%U+@DRrQ@cL}4i0xp->V'
'nk8pZYErvt2ikQs2t`k&*GjF8+0JI?Q8#es&vnQ(vX3}_FyF*jfPyJb7lqfl!ORWB$Wz^^N@LqjhDLi0w-2<=0+PGJ0#I2KdNiDDDQ|*?K*W~P'
'YR7l^<!`)~ng4QRT4U)6u54cz;yIfwO`6^R3?+a!bZYf}TgYJp{<lQjH;O%}+plmYBUlL28qH(<yE`hYd8_0N%}Of-U<KUJdO@+XVEJ_fcbdY8'
'f1O8?KA|!Wf)Z$Jwc!It!G=;5OMNsrIOOoSG$m$04_}%jNxWsXgi>?l3w?IOr21txU#W#HZSYyP5}TLL$sHSdgf5=5<d2A5u%pVEngWF6nS8f+'
'#mLd$*6wi)z7f5~71tPfE_x!q4^H_Zp2SG|Z$>qLXf|T88R?obv~I+qboH#4(y3vq3o#tlT`xGs8#F@KRLE^t-$a?=1L<N|M;PRw6h@;3n>uC6'
'B}^;I-NuzM4#(2$1@FneN|eO^3yyU`=666b%M@dsD#H}I;6dkJUQM@I0O!w0`JiKzt0UH0RKc+5e$S{>*BH3)SlPLgK~@(9c)zLphxc#)DtVsN'
'1%rlQeXUX7KGgvhS(4=vr!~OLJSck>R|&}KD5S@v3xDqk<Dv5$P}*-w{)+>l!@mn)&^1230A#X8dj?1G?%Z5pS>B_r;({8L^qT5Ft`-86<>4+@'
'GXbe62)bZucNn=OPiG=5@?9xR8)o|^6WUD+q~f4Rpp#m6f58Slh6uFf!bl*@$JJhP5rA>5vtDA9bxxoWk%od$7e=7YJZQ*SPqOgP$5pO;B>V&!'
'd@M2dHU8d9jI*q+faqey?ZUZeDF6W#vpbxd?}CTIyzBGaHP3pdzFKspbxY>GI6xJG15pU-n%L<ZZFPgT*v#DFZ6O{=ZO}op+~bR?PE7&Xy62m('
'a1iQCEm?4}wFx816)?cKPLy9@fZ^4J#gYV(QOjx9<f?oDEjPWusS+N|i%eZnD?6OXiqdYeE0M;xl|0q}a2)=p3i)F|n*b3)8>&L9s6d=Yl|crl'
'YNjXPR2Ox0h~<zXAeq}E0=H*KKzo(=vnPD%9x(TNTI$(Dk!Mex**y|={6JAv(I*^F{eJ@(Vp{1^MKTG1i2$~MFGsO;Mh<wi0jMv~phyzhP{trx'
'-g=*UGovat4zE8E)n4%DBw1C9hbb5^otzGNh+#PkEAXMd`o#e?qOzFukL!y9_y@$7vnqN5pxSwMx%LbeQCd93ey)??^qcQt6o9*ym-D*;PklxK'
'>tn?Jv-MoTJMMVQRIFken53=ZJ51SC@9JcRj_-;EQIrTlK$ca&tKw1^px{sgH&#Q^ge!=+e8u6qZxAM7F<~M`6T|@21~Uuq9d>W?!?Iq6*;ZlV'
'+1mEDZ1dBOuf{73mje>36+UM}H?@F`j29da2i}}<_-h+88U^jZ{=PoH(@Tm9V56#D1+wLYs<^*S8H^kxZ)**o(6zT|F6GvC4Z}8H1!YzuK3pXU'
'FmS8+8g>^W+!VpZ8ct}rrSh0d=&3eQ#I!yv^nzAwsUHG30-)G6c+I~YoD5QjGT8#8EEaSjnQIL~#uo|&bT$28osJ_Z4n~p^VbJeB3(q%^i_BYA'
'SIR~&q+1Qe3x$Ke)4T2zR%^-A2zRwDx~7JPN<cptjy2qz*9p#v&>BJtbekg07N$G>9YM%f0lI$&Xp4~HiC*IX(O1tjFl)d?=n)TC;d%F^<n37{'
'+R_c#Php-YgwF(H!c?{#8ZcfRpZ(?d_`Sk@sOQU9KH-QxCxk_#BUD3}-;imtM1N$H6&xh`&;a_m$xP&EF|YOc9RaV*i3KKW;frDIX6WgV(UKC+'
'B-d>vO-n3KdciBh!lhtl>k8qV;)F?N`#O@-=B&bLDv_st3Y82qRz)`T@ryx^D<xuVp1o;Wkj%eV+@z~yPizm+^n8`4c(E^2<dA$GWb@8S<P8Hf'
'Zf!(SAoLpj#oDc?1+YG7b-ZZ!OvxOUk+a?C=)OfXq&47FIFjRMBu~#f(LpC_0RvMjfY0PL9;DXbhV)D=gjX0>tHT<I8=ZT~f4?NQI5n-r0h#^y'
'ruE5_QLnIPWhqY{zGP2>cKF@N>9<F3gEz;gCx3h&d^|cm9k|K{k98>GRRh`LIh<#c&t9{xP=KI{-c(t?krYb8Ra#l9GqfnNDNk<lw4|f4@Y!93'
'Z#eQ)rL^!0YdB)0gp{f3DlS<<VrP35<Egel!GpX<cmpx|(Nq7~iTT1^7)s>&uF}h^u3;pv&p4+908SChivmM9Z%0c|sT;|wjpK?vKHpNbRYhoo'
'UvoS~qRLmwG&axy?3Bxl4mxEhHbjTN#oy{`HdYRXKRM2dYU5${M9MH#9G_T$+pM`%%jUCowJlp8-K6&JbkpUikL5R754+UhBO2*t_uhv+z}iP0'
'1vBPWu4&D3suI`0sJ2JRdboZ1Ol)nefId^(DiKg2jCbRW*W*H5bJSM+JfO3$96v-gDGJ!QCUdv#225<Rba2>UvDCQRcFMIaDs|sJ%Z-*gi}m}i'
'pKV`vL^P7%ai*gmYt+m!3_0$O-MG7>;rr%bnpIr2o!eJ)P5Khc{{@gsWBjekVa|SS1#{sDPc?_pZ2?^bFq%W@y*hgR=i~Qp=rS92AWkbZtRp00'
';MZ(Ej~8(m7#mXMXbkt_Rl0{!KN|fww3V}tT#3<n8WV)zM{hh#V0)jEw;;S%#WvN;EmxMvdJXSGA}^Rmn6{OJ7Ix@fK2PEWybuqNkZtV3S|m?#'
'w<P0Fz7L}MD#8g=7;TFO<L>9?yvaw5KnU1l5Cw<rXZ<JuZnMd=rNQe-?L&`_^cbduO4)K<xaMH$Q+Y1ys$`+s5qlG!B+Tf3BahU~h^E&Y-Qp_q'
'GXM#~z@GSnutAjvSg!Z$<#xhN^LzX>K-G>)XoQbv9Tmp|%$N3jR}DXqsYpyFn1vtnY`UI^7ND}#?V;x3aqzioP--8?`o!Fa6n+cZH%<w5WBw!q'
'l$dUdLnEXUA#2WYF2x>p{t%5#N_DZk?M(d-I&YIJl-J1O<$iOHaq)lZIaK^7a5E|@fJ=SEySzEHNxfBvSXHr3z9jqXC+;hfXX<qOVXX1Ln0=bL'
'=I0ONf_HPk{&s&F!@HB&{eoPU&d6P%8;-$K@jG6F`697Wz57zu>jvmtd`|8-elaN#;1`?mN`?Y7U(y*j$SN<sB-`8>hL}d=<V52~Z)!U;J!UiP'
'7$MR*|7$fbD@6MC$(d=zZ{lSsK`hK3<#Kobg&LQciFnIm+L_0>@hy4B5nAT)O7ecoSpK(>s#p?R^P8%1d+kM;9x_i!jJ}zDL3Q$!EHhtP>s;l$'
'ru;inv}0$q9%HE9G1@6iV)Qa8uLi+K__x`$f-c&)Ji!6h=^^hhX8!|zAifAs5vnLSNEFlxs#higK%~Nr_xLKmh93NS4CR+$c=@PyXohr^MG>vw'
'@C?IsIqSZx#Tl0YxR%__7Ldkji!q$}Mp@i~<iPf}K9?(Bv#Yo$tgXD?n1_Soeh@6Q=jlZr<JrJXlUV)UTmn8V*T8~kX-cm<N-pN}v={@9wDv=W'
'r!-iw#+V`@qsWWjoxC~zFh2hN?D+lZ$%pr+*bUu+b5;yA4PeId<lWI9kBx$A=EZtmHmK(et=Wt&VM~fHtwx$(&a;bl_#|{7i(+gb6v9*xJMkP1'
'c+~=&*ql}qa=iy@rmsquk2s`urYz$ooR`N`nhjD?vQ4GE9iwCERa{)bv@q`4!Sfewb_Ij+iqT=a)9G=XHvD##+)mR=Od^TYr|)u|3YVx3T#jA3'
'#r8u29JedCJlc0rP}OygjJuQAgo<b)FyM8@?`|0;Bm=#KjQA9J;qAd>)3xpmy5rV%Xi`|KOjN4YCzGyGEv05S2ObhQ%UTWfT!q#j3k=pf#0U=Q'
'#o9Ys^Iff?7%Y>eN%d;uf)sjo0o5cMUOxeiq+|^mHFH#ZFRACG7$x+6-gm>y7(S+o#B)3msb=LzC+`L^K9&L3@}#?{PSiMF<hCoP2ry5V%M4jK'
'Nf5FF85-ru&9C|eYhnF#FG3VXn#?XMaa_Xr@W~UTZlpV~tZv&-v$sla(G?K+XS72$W7d8={{Hyw>G<g5$@tI5f3<G_Oj3rqN~o#yHol1O{B50l'
'fAs3;uMcl*y)1w^E!f3c<8g}kA9{QE_WkL{w@0Uc8XvuWeSCVlvlr~-CjFcezsqlIj;mU>W(s682Q^h8K72@geV_KW3L0VPY-i*Ki_mGsy{btg'
'52a@F6fFpbv=|uAv+@ofqst1NfQnNzSIiiFokOC*#OYZ4JICy3s$;M7pzSKyRZ4F9IOH}CnSyJhU3YsV_@ouISZJtm*UvhgbC<oceBXI4NMxla'
'4<AJL>XMR&!T~t7BQd|DRrBz2HD`Niuz{1)MMo{dOs#FGbeq$?!m7kRNMHf)m6?iL&^RQ=eg9#Cxq+n=V!sc?8YoBVuYo4Su;>zi5yf-(R!Eeg'
'tFl}bgS|Zgl?u>$S$?^9gX-em>S`5sHqN}9xH_%BsEV?QT5kVM(K1833qTwA?}4aqE94vZvU*B1bn2!Dv)ki%o_lg~8`;#x>iepyE85ppvq<dK'
'h@#*M*hC<E!~5`PU0!9mq%wdVdf?1A$Q|YJ0-F$9U1ho)BDz>1avt#U!0QS)X3J9hC*jctL|39VQ|s&wNN4As+z0FAswkU|F0PIVly{m`VMd7e'
'V#_A1uCg-oyRQilIrW-iOaTlMVqs3Da-X7FBpoPD(Bxh?l1N}WY*W^8DBAA?zYq5NFJS?@-q%fG)l4p$Qbo@x&$Qw%m!HE9=Zv1KeAD&l!ukM6'
'x=?6}0xnb&Yr!M5h-rbGAZ$IE?zUUq79sM6(HX2M4lewT94;>If_tNB%!69#QCrC5W25Wk=Os)IO@yHKM>;berP-GI4QFktZ7TAKskzm#OcGsV'
'F-ih-aN87yQLSE8Kb}jIvyg`$5CEE!Pp2@yhP!Wf@jg;zd-142GUA~{eCVPfH)<?S^Zk&?-`|^3<}73^Hm*%3GDSr$M0!43+%tbhl@urbT#j^R'
'FDXl8S_m7Vds`)8$x@U5t0goUzam>M3PhF|G}7z)e{O&qi_U!^I$SL+1$G(BjSv1I6>@Jp#-lVI3q2O33q@d{6cJGUWty#Uu1eY>uF~s*avZ6r'
'I|#M{^cuAeebm|zF&-*iJepohCq!-WmRbn*14y-7GjypH5>LwK1UTNIyW6IKPT1RNEe6aGwqPK6lAHjaYVg0>Myhy(BbKwqI-wJj!ACP3!YuHg'
'FGuuEYVN&)yGk3vN|@`glVr22pNXA?TQCqjGMej!O9krX$S#qK-FpJB;>RI9)CN~Z^!K=H`bIKF1nC4f7qdh<!@fx__&S*0{Zta5h#F8IIjr}%'
'-kXQgupK0X+uTJ%xYU;;BI=IGMA8j!O}Zir@LFz+-&V>yo6v-(?7|RytOovbk(-B3<>#S5)%^agNRVvQgJ{rB&D3L*Qd@~r6Na@!Lp2;CIy`|r'
'Ewwa@e6#TN9L2Nl$(!K58ga`rVrJvD5E%aZiGe#Qf*_S<qCoV{#g_2f!s2;yxdsf{fE#ri4T#ZXyzOq-phtH@_ZJv8hB@;l>0vvfRA8|KQ>tPf'
'RH^4<;7N;X9h)e|p(V9f;kWA8FfXy4A+9$ahEXH`{;e?y8?l-qkh%UO>zLJkDz<GNwL-B?v-FkydD(w+um7E}uZE26==U*qJ;HU-Hb!Ww&}qK('
'(H_NZIzWTq2>-~Pj^UU=rXEq?`6Gc@jr(nCsi#{e3W3>w!unN0AB`pGpMCjfYbzL1YG)(;-r;Yzwj!~n8XJ24?4d1D^r~$Gf)-xxo>rb-UisU3'
'e(<P<0F=^(hIBj6jb>D~8*1uSBEIzQ)E82xsUKGfj<NzB6^VykpW0h42GP!{W)%q_nPsIirFaqrPV$uM{*VEX?s7hHpVPOPgyTGF@HMLW49T6A'
'x+R<?<>X4n?YrHI<Uf8Tu`6J($cf;eJ+U89nxd*iZ}AzeHh@7uKP%1$$v$A??6YYqC>Pf-TMmj^U19RYHa97ynzPLxot_?_jo+TU`gHWk;jOqI'
'ww+T(XkwkE#z2`&<d)1{S>uvy5iipjZS0QQwC(;@Xxg_}FPNraqCI6Pg*ihBGT8^xKhJk3?`5v1uRnY`#+mke`)E_7xF!dBPy}<HpjOv?<lZV9'
'tCJ_ls}+v%3sA`9)`^Tnwks5ygpwK6)5?qndUx@fQTCQr-{Qy6*l*~`H)J-Za}QY$DG!ke$#m0*susnfdK*!MzuuVFia4}35$!6>-xNtuU7#vH'
'1^8Rw%tf_HDgMHZJ5IwO0)N^D5n=$jJ^vF|9O>RzYon7T*?F19m`HJzOdSclZe(Nx3*QC%GFYz}LeDZ)AsNXz{msGB<xbPKLPLz%cADt85YY@K'
'9q*7EaN_pyh^fc}d)CGyww(xXUJpu3WWKiM+s2Z8g#;IK;Kq=UBWl;!Amffn3kH;^YfUk@yHyy^I<sW;l+4r1l&w`q6TaiN1Id$M%o2v`sijTY'
'))a-iev$$L10IA_=1cIXPyBwLjDM`-bv_zAJ9m%Hs5)}Mbgq{rfj$}$RRgTyg;A&`<<wD+%}k&52{)WQ<LsY%rLu{;9fo8zF|E*iE(}^InK=8^'
'F$FsHQmlA=?c_;ZuqYnS(Q@z(bGc8$Y8O55LbSoZjqUIvL2G*!m0&}&oZ2$C))=hp?e86O$a-|O{Prz>Wm4hGZlDw6u;YbIiu8Uy$f&onM}(dT'
'$B=P4wL5B6Z(DE_X}RbP1F!K`9GDEQX|E~O&P-0sJyF#`|6sZq>x@FJ;3-I;@hllDt43oy+^Z?Fw`gr!ttS7MR&TL7Y*qREOvHq&bBk2YEgpQU'
'_xIIhtHEHoc?1orkFhRQ+$|^8s#^8WJ5jZc{=UgVSeLOaYn^u}infs(92e=2jUH`v)2kK<fOG+l{E^DKhK`jO?8bL<-MpxzMQJrz1)!bO$CJ0p'
'Fp4zu{-PADiQf0n4YX~n6RHCwnAx_}m0Y3+LU1~}^Eo$dxljOdbdik7tfswnhs?9eTU;;GoFn8RsW9raQiBkul(tBw#>4)3S=hOPY7<5KD>j%h'
'nQd6=v`$S6f*;peXx4hI*Dp{0&Iz0l$=!Pxj9=xLbIfr6!MG+Zo3}Bn(Yztty=>z$Gwc9EZ4Oz{dq&n!Y+b}eL)9yXp_DXxvL1%YiM>lGD9xYF'
'&OW}1C!e#~ETpL>ti+{Vw%5_Ky<T4~-<+<1)dE_d%_%yzUFp}sYfNeFIAGCP-(%{eat43JvHB!P52eGB!J+7kR)xLV3Q#}*#X>1JHFJJ0tJl$`'
'aPgW+vWGG8Er%mF^y29hnZ+G${<GHmEa2^}a*VOb=@xkWDv<R;E+DwKNc^DpY_@T6=N6Ge%gMq>SZ+!#uUOTm-~u>D%dc!yai*4;Xq^uL(H2;D'
'QTwpM2dy$4C83(RtJy?}7K&S{1P|#~gBjX`tacgAz$!dT#SF9?Ngyq|y2bgK*9X`ZE6ayYVWmUwM5e8aTgvtp;H#dYSK^^~Sw69cqCPoqsvwrm'
'3kqv_JH(wBs=XLmJ2K=yMhHl!2`e<Dd{v?Enu@CTQIM_TNm{ydl&2BH_|O4V6z!0&b}I2hhlu!~MxQ+TRl5lDpfu4wx>CH-R&AktPoC((`vioV'
'TaF2OKEZ^1h3LA3y@qy_*J%{com?Wi242|U7_UJMJbdop;bj6#Q<AFZKD>p00f0jMg=!u)iyWwf9WnsH1ODP|(2TC%1YVwM5-+w^NyU-%R56lX'
'xV4O+v!04hEJSyutRS_X>P`v~x4!+_Y2fh}R5cnCGNau<Ovv+DaX~hxbAXP{(tleiq?rO=1LO&HWpB?Og*9V`Oe2(=DBdE{XD-X(fj5dAK6HFI'
't!o25x{*F~UVmwr-WXxNt>cdpNpk|iaA?mVYlCrEWt%-}RXpb>!G6E8>b(LyPtnmA<TiA@$fj*-A_|^ih&x3Sfd;S3#4VX)o{bk(Dd+<o`e7l`'
'!YQ?{3n}OE@SZx~4DO|Il3NFuh?>9l?ztuU5jrc86q9%*p02*YIHrHAHX06p_yI^-j(<AU!+U;qZ$8`bvyJ)e2fASmLi5&7B?uw7s0JhCW)QGV'
';K2)0&yNvZ73$%kGvY7|{{pS4$auiitWaZ%*Ul3A^XT+6pe6#0D^0vKvDEW)MUnU@q5>y=<0dg|@zV8@=ahpzy5Z9K!MWKXxz*pm0DAaOo9a8H'
'lf<s^JX>Cl$#q8MP3Gr7|E}&<S8xUd^zG*9?`WXc;vD&|5p^>fPzNg6gYpIse|RrrH@6)tvtK{O873#%C28ntJEN}3+M4&Oz=bcw$jDIrb6)L+'
'PBTnt#BlNBB}iuZUJQKrlCBD^Wa)9Wb!%%Iswk|M|14j^2JPpEc;WgS{&{H<Iy2u7l=>aKKX?KE{MITY>0MMazrzyGtrA9hAZZ5r!hfw&>MB<a'
'=eIbX165>WZHk@`F?L2MZT(`T#&zyJEb(R0p5ReATHe)90R?oMqqG@I(mb`Unqbz?1-5CfIWyqZP7OT}f!m}<FOZ7upo)41NFrX|>jn@Z;4`S)'
'zQ5f>_jj94c&=|nu_<$Oz&ohaN-A@bX`>Yzk<fnl0ZO*_5Bkymi+)E@21=r4w@UZYDo;~GYdA#AJS}4ko(8|`!xtWK)Q=8cKClZCa39!7_wk*q'
'vicO{PGp2DQcP}NKF8Ig?|15Z2{dj0Wk2E)KvVcN5*H~9O9}5`H9I<$m_RMyc?|3z*aUmT<mI77Srmv+%-kjrRnZNM)1y)$4TA8G+{BxDawL@&'
't15IPd9sW$Dq&Vt=r1aTmQjjxX;o$ZlXGjCc%h7q_pdP>nH*(0v=d6UOzC6X5t}D=5q~DT1Q9-WCoxiyQKb+@L_Xz5s)Wk?E$oXt^;<SbjRYgV'
'N+q&#5YU>Y<(MlT!&j_f=Qf6Nv~`BYYJng;-FXd(!SNG43mT~TdIEe4Wfaz7Qz*QM<Tl5YH)<KG9g!y08R90)>j`f2=}=YfRcSTgggFmMvZ=bp'
'jnR>ejZQfLzfykSTM_R`T#{Qlg8@fvbFDB+Pa=E6f$8`5%__#kBbg%(<i~W?u0j&a+&!7wxYe&~lQxnCNxqT#s(4**aeJ#<GbI+vz@AEL{^ity'
'BAQevFpa9Z%^QJtXSiM>=vGTd4><~vHhkTl?pjzxf;*qV7+K@kpqLX+^MHUO%C=Pwfvw_|6u-FSl9iDc?;PeVFGt<|^AWAldGIv=!f&4=`$m_f'
'T!1cHf2$s9uJd-p3vo`Ppj5xA0@XM?a3O<ML2aXIebhhKgNi(3ih>9A=;{t-;<gQfJXJ%$1u@LGWzDPN|AO&W@#01Xnjf8#^=ELhoMq<-Y5W_r'
'We$St!(PAt6rTcay!(_C-w(*u(|Gb%c=EdgdV<2oDUyUiaKJ6a^JMx8hLz+;%jxM=3_QUgxJ%fw-y}Ap?+%{^RJWi(d_(>pgWx@eO$ok%kuUW('
'yhMwFk9ms0&tGSAI7U%$O2Vn};KRDaHB~=RwXeTtn56LpWYlyL1>XV(m}60B1P<8e1Zu)p9Q*h3_pk65m2Q|C>j-lJV;eZDQzn4Y7nGtYLLtTU'
'MZvo`zf6~J)t67|%Xd+5EF!(N60(3oOQSc*-{bGr!6`7b(c#PJfB&i9|As#9*M0gQeEnUYKhZugQux39KL1DGqi68nK?EEQGC9A6|9rjwDa$@v'
'UB5W|0$m@%e=k+nhYekSz~mKH-_O5P-+J0F8~V2Sn4f15&($Cfwhw|iJ9`seP;cC|`}Lt?c>ahn2;*ok1OwsW;n9)wY<&GW20@&uJ%krq_Tm|R'
';Opzx`}B#_AbfiP!tC#M4*{58dn5W^02rQ|5c!-2;3CCi%IIhzM7qgAOhxR*G2T!J5@J-j4c*LUl)mx|)51tU25on9d1;-9P3Dn@`I-xRVuKkJ'
'Zx+6h^eTg(AKe?;M`Cz3<|CV5O3*ZJ6RBe;t0(0}e}g0JXx%F&SWG9t%<D8+623)q_&xQ83}*D>=4eK`NxV2By$eb_WKKaBo@=)7aXa2r3))n~'
'!(D63TYDko#*ri~T(df-jTI}ty8pbn1zOZjZ7X#?J~3OtqLiYt?iZ%^*tsXQcik>-!odD6AixpggPI5pPZbQ*lXum0LY>XMtyEQ0vOGO2Ud)08'
'f&)q-g)pOYv(h8Z<z0oyGvQ;<xFF^dJdgT=KK>uLs%G%t@BJ3RpJ=d#{d-F@v2o{U54SoP+vis$oaG5};wb+NIT5T35-`AiufGkDb;)jU&^zor'
'aCx=^h-wp_GMC`44&m{}<*8;{t&gD@Fz;tjQ_p<03V4i)9#Y?X?qRgDBQ3^cyXM9d#Sme(>c<y*5th6bJ92xNMmK_F@ZZ7Fw{K29SV1MLaJiJ6'
'X3Sv|u@$YqCV8*ss!=8UYD&t&!+rhX&GBn;d0UrRJt=?&#Hu~gZeAU{I2>b^FN{<YUM1I)>eoEIxP{l2%2Vm{RVz`@9-Z6mVn_j1Rj_&3<s5uu'
'h%hhncorA*ql=}w>_5s%*4c1>aKSO9hy6i5>CWLi#pt;TKha*$da$@C^CTIP;L|PFOVZT-nB*DiotKk4tvqZ@s_-rR_x`)%_q291yz2-%CNSKG'
'#1c>p1-wB9vB_MI*=(^&&|RgtT9@g(cLUIV)eiY_Xk&VQ_#oR;9jCKcU##*B6T0>C^|C$UHgL9AsNTJZmrIWS-W7-{P#`VJDLT>`l|P<*Jf`PK'
'o}16#9Dn!z+qZ8k=UWV$PFXyyiy!j?Gr2I~M%-*+yglxBd#G9)<Sfuw9nwrD|G+;7&q62XynC<wkl3{v5~KgCP-sS#oZXT!L=`+2j;bg;-LScc'
'yX4hCDQ1;_)J|Wh({H}t<_RUnXmythS-Hx#T3o%@1X|oHtTxEg%M&oU))J=|9R?$^sy2h?QWI6=Tx{8~=p}mTN7yd9)xP*<Asxw8J*;khXjt`e'
')8kR!zRBdt=-t%jh%<nI0xz;~gWG0+P{XUw;}X`e&lLXKM)}`-IzB%A@a_9I{-`k4j!^Y&JFz2FrCHM-b-+G*ItGDW1`5;{uhQ!S6@~z5`kov0'
'cUwq;Gg^atnhVAmuxr+W4v4olfD7NGhWd(P)DO-+1kcVnE_^bdqH#=^P9suhkzZ7DQ$%9_7NG||y+Ko5L`&%Tj`}XB_1qnX8??Z9-KnqQ*4PE&'
'#N3oh!irj85f-)~<rts#=xX~;HA*u6kQy4m@#DrkP-|mFJL6~FtABJCt7opO|F#-)sZkqzJxqdk;0B*)xSqm!3*ql9td5`?<)DY%R_!<r2o%)7'
'Xq$~ZF78S|{?GelX>)KWe~hu$4VW;~p@g^LYlSJ>^narU_7%=6CXt>%t%}=CxNv;|XyFRcd)t_Sl(nO?b;3^VbQ@Dp=ysPOR(%e7tN~KE^-3&;'
'+0K{dW*J}Yzch@mY8zJ(y}KnQS%ot+t`b=OlP=Ot$ke!V;ewu<m&y;?pTL=OXw7CG*KwtqFcNW!nuFw<>amTH&Agq<X;Zhql5QKyAPO}`WDQ}Y'
'b5r9|k!oh>Z$TO0l8AVn&DV>>Q<7g8RMMVB^TMps6`4KT95FprfTFTYwtd~A+g2k}b9Q!Wrl9mwKOTYTQrv8mwG?($PH#3b)Crc?swLbE-xvbm'
'BNFQ5A_D@hlWTa+gT9fb+~fAmU?Y_JeXXms?v`^~wKeD%X|iEnv^fiP1=HWGcN1S5n9>`C$S;}DQXvD}<y2Y`uRJ}G{O`!g(G_&wHpbLBM~f|Z'
'KeWMAr4~>k{$s=%ey(UUUNvkHY6uIE{KA_xk(1b0AY|$%%={GQbKAWP@rN6zo>s5!Usj1uFozy{wWv8q0egWAL$O{AJEJa!z(04UAYi4hE-h+S'
'Evsmk-9rw0q#s#nNq(v&Z*`;;;xDR%Nez~}a!_O@2JN_5r}ODJUQg3(OmOn=Ksc5^`4lJDVq8Mx0;NHE;wbpF>HF-qks#rxt_l{1)k@AZo3dT+'
'DSW&cWdrNT%ZLZ+Mp7{Hm{>H*_VCx7m@`^6wg#!G>XA3B+NuYPK$5E`OT@ra-z!26C=}2{cLb?t9!Q0#*X)aTKGkbwe%^u(TIlFoO(}R9v=F<t'
'I_A<wLGRgfdv>I@bTHJ?I^8P*HEsmKjKAr#j0~~uvS0J$tRGpO$Q7Ab4hugG?yG@rz7oh)(IONnUl-}5Q8)SS+tZWR<BvyYfAR^dRF~7Uk$%&R'
'N#krMCW3HYanFFKTW~H~o()gY?IZ*C`OYa@(}wbUiui`jSmPShj+S!b4yjyN_sg!e3UenBWV^GOBbInt@wnM~L4!CuyS|1F9`u*M^6zL-wa?=&'
'1WzWUyb-v_q)wuror<V#TYYDyM*YW3m`3q%oFU*1fKy?Q*4AM@8j<Kr8sw=K?ZPmP>lU^l9o-OX+HnzM^J1OXX<R5Dc0wODLPj7|(nMO-Wf3iB'
'(`rg-wVKkE4ui5y%!?%Bql4LP^3M&L@z~O2lH8!u<a2UI8rgAjyUN?@mm<PwC<k;GwL7ACD|5_g$amykADb`?Um<T1#M%V?n{P;+k39eziJMYh'
'#YnBoBwvv_7sR(<5+l#eo!JfoP|%3$Uu{Xd?SyqA@gDh87?U~Djw+x+>tR8eS4VOnD*HdZ1g6r-pX^9y<8lNg8gN~d3;UtIe^(2EF?{d20vW&U'
'pAUM^XX-YWd^nOBY%IlY<2wP0us3a!aEz9uTsx9(NBaW|Vq=<*qdbakEqw~WGAoT&`|@fZ5ikc{Q}2oudB?~K_n~CDkE`0Z&V<@QHJA21cSW;!'
'D!;Iv@woG(U3TwF`LTD8J@NO@M4#3grL*?DY?sk}xz9zTd#m$JbYIob4QZ9LdShkSL~t8lr^Om58tXUP{E+P0O)n+o6DvPvEA~^WL%!`5BXF+)'
'8@oW>5)cxqqtGi!k}W41>=<l&3iyA_Faaz~0oRWXN`Y2;thYD)Ym#_!C2T&sIt4dZ7+m9XlB}RJ(N9B<Qf3OLMDqH<5dlQq3(l}-67S$R=H!zO'
'YkLht%BUtIP4cMf=^2{?B`59%%)*P%)2Ay!d<ipYX+sV!Al4vNDvJ_xS<G75a_=72eba3j?+7I5HGESgBE^Z^K5{&_KjT>L9yyk~pK&a5DuCC3'
'$3uQcdojExklzgM#bb8>kOgECn7&Jpdof<%Dw&jHqG-c(IYH0sq;9w}$+K0htw=z5_~Wp3kKZ?~!R@fsPH(#D%^p4gywH386#Zs8oz|e%>Kzuo'
'>ZVs#;X5ik9qtbb)#Yt9d^JC1${=T-V@r@jI<c1a{a|%-xW%U>Eyyz*uRPe7$#_}Zq@3oPyoJcQ%zRC90KHxS(Y`X@v;T=mhgAqnG~*o`$-spM'
'7Am57<@UjO!<;Oyk(x%6d@qD{i<`-oD2`HMxBH8sEf8gI4AR6>u&T6D*CIT^p@n7vrvR|m)Zmr4F~9qZKOgQx><@(6*X$2`iRPq1Bh8#@MRA(Y'
'S)KjlA`UAn6#04da9@~ClM7BY72V12(Q#*g$9xPR;r!^RF^SV9S{$m7CO8w_>^7q<Qx+t_lCRMy?5ittR&1}8L)+jRbn8PRu0R9F`kBi#E7x*K'
'y3SqbrUg8Q^0#B^nk|(tzaQT#cqWzaVs%&rZrxoR3WZ9IPl$<v=8<vcrY2b164n+miQ7_g5?g^u``V_%*QZ+&BeEEQ6?ygavf8`gC`KwbIa%e('
'^Qp26<W{H?-D*cWYG;c@yqr1_Un`p(N_9}P+?i*WbKsTdIubWMWhQLQY}f6$yu#>wWS(*mHm|TmW7=1?Gn`P2q$*xB32o>eD=I+O0O$@^@$}oB'
';n4F-4zUh#wv2OLD>uxUEDpNHjauDVD{9T->sbnayYSDfHOb~#KAa5uxBLD5i-QT0NgZv4FAx;fbRH`rnpYK&lme~2uOptW5A6(kFJ_yTaS`s('
'nVI7cyG22s#BK-D+Oye)je)#{n$cfXGnI=He%mqQsv3I*W7#*z9zF$9Vk@RC8pzJ^u1=aG6{RYIMP22^Ay)Ba&l~W~SerI3P6@!e7|*k8MU&aL'
'rbKngXwbIR>}YF2Rnb?^=+&CD)_2U+JoPelT;)h@^-%3vr^X(%Z(c(nquhKs(V*_|VjkA9p!QvbP_E)%(=FN<qe!%;AR(DSv%OVoH`%Q7eAE<b'
'jW$dsc<Ng=!94i8>Sjb*3sZezjRP}gprYd0IR~FqjkV5qkQ9<H&A#epO*iqip>E@{gy9`$LF0~9^r)y@*QzhF!0vc1U8Q5!N}pmhV&5F1QA04u'
'O!-BN;xtlq&ov=8Yj9K7Dv;ok<*s&TPd}n3G@AOoW?VjX+WK14vZuhAJ?x6I^~YpMJxl|Q-AQYlz#68odn$%gP|UOPE8s;@|FvXMoE#b5NN>h='
'PGV9z7h`iniNW-~aiBoTu$zKb@HZ~05w(7A)-pgr?!Y&#Z8asDy)S;f`5~DNrhTDK1qNi%&&HFh_40EvWf1E}_WnfrLM~ojuH(xjs=99ufm-YF'
'5#ddJdkdVxoP!QtBxd81aaz((tNrnXKsCMDA38c?I}|cUnTNQsh)q`T>!?NFTjx*Vcw)RbehA;_k>g%tH0pHw>!7-(CRI`yFIGh$VURI&AO<?U'
'gwsW@qhP(9r=PWQ>FD&66W&q5qVV(t!v^+VuQ0^g@V;dUPOX6%U7>9<1H*n)F$_bko<Db8&mYH1qjlSg7sPJnBAec6yr|2#ys+)L)sy=*j>5=;'
'CA?mR{Jw83)@1@*p6Gm98{5T8eICi!n~QBR_@aBSX7khl0V_ew&EQ7El7bpVl<I6$!~MRWp{mG-q8`K_J5<l|SK7fqotvo&<rYRQB+}uL%O}+d'
'zRf22hxmBDiEOMWH8&?CiwlN;u;ZT|40J}`CCL1nOtLoDwRqq#Uasf!BA*N=S12;gF;N6r0fBUsWP>6``3Mz#$?))Hzuz~#1`529B?8TYjWLN`'
'GjljJw<TrYNE=R)T0@=>D^evX<PF}=6l)^)sj6a#G~6WN4;Jj^@dXUj-!m2Cz28<$ZWl<IUyNCUN4N2Y6qrV)AXu?K8Y!68X2GLFpH#y@RS1s<'
'O{7tO-!_hR>V`9>DhOAuUfMzR*3}8}yj!kRBF~n&u_pXWRsoL(fPBN4g+D==JxxTJy^5#xn(Yjg)SbNp_=Xy2{KkUy1t8zgQKe@#_+EzJ>SBL%'
'pKb=%=l9llHxb5uiSyw;ep!{p@Z;HO_w?-O(;0jjUBmW5wR6+<u?DC{7rye3QYX|#tr11eh+r_83zCT9GEWzpf)4MeB=1MDZz8d0U~Iau3&}OF'
'Wqb}I5@RYJTNHC{RR^gRYQSQZs}%=)j4$I9u;U?)95(+8%&G>Vp5a@VkET-Z_Ss{)t>Go3H@AFs{ozAR#^nf}5RbpPQGYa0!89%X7a*p6tyGxA'
'%keE<lAvv}p=Yovd~Kd`%3cGn-JwuS^IAuCy?8!v8`c1t!K0u&eJWDB|DlpBh?=r=t$XgNQf1pifw7r-bK{7@#A~P4@(?jLPL(zHQkUg`acVt{'
'&VEhq5I-jBiuW?Z8)h=VK$}EZ1$E<`E`e>pA4msU1psGW$d$1fS^d|+x6GI$CqD;R!YalMFy@-RN{fJ;|MF{$SPup7q{yD|_-1=r8qjQ}I9`jB'
'G@9{DA`<Xn0zr%%_>YOQT@5e;fH6L`ucU53Cv)EOvS%3F@FTTBqx9UPurwd@T}c=3R{XLQpJu1kr~3`<8ZWArJyIRFN(_O+3Df}4P6UOZju9*R'
'RGS!Ky(}&Y1=KdBQ|?L#!ig~KS-salwjtGb8vLG4LSx_d7&MsATBB=tgqX>rAfT~1h_iK1bvGqa>7vY5W0VAklR2sr>9>JE)o5UY376iR-PfYz'
'+hbHcnpGZ|&BJQP*MP*Z(#yd$EVRIV#H%8yLeFXV>ifa2?*N@=w^<uk0oQh*!(gbMNeVx0ZYI?fm5*;r>)UCqf!d)FwQ#;695-wNjaD_z;e8FG'
'SHBsrZw6k-z>d263?GBWo;b+e@Lslpp~;-T7n<DtH7oc)iMRP_ZJ&8H{3ynH<=b!kc*(uZR^!K}SJvns!`-MDqIK*6^7IVDuup(PgKe;gCve8Y'
'{XsnJkFK%extoyL9$ctOHL}~*C`?kia}<3T#r`O+jlw3;p(^2(Yk&47`PpZ?bG;jvLo-JYQfC>etf10mH815Xyf=E@2uI%^VH49Gd)6{awX~YS'
'xSo(ZE_DG0r`5Tbu&%}&yvXm2+l9(b!;XQ&7k(HK2$)xGz{*d>%FCkiPQzrMQM+TPd#tFMl2O-;2IbUci$X(i#z7r$czlwCgD27y{pxfuDt%Q%'
'+J9nohlL(4Z-1c<cZli#(GRyGvY)UH-Gl8CzX3`l`%}Z4NW|3q@*<9CfFiIi(%U2-#_^CVs&ptU3OWL!4HjI)CoE>yiXxjQ1K<pf;~3ch{&q)-'
'Ja?kE9q*ExnT=}UkxWRgimd@YDfNPN7F2vMu2V}BW7NMTQ^_c}qra#PEAzIRMJF<U-g(Z?10sdvN4IWR9Mi~YeLK;;>SFWs>PdUQi-ydVi9u)Y'
'zB=?xQ>wLOSv}9U4%m%5{<tA`|8ET0&0SvY<?bQhZl2zJnIYdkX2`WX>5m(8_g_0^;ug06g#ohr2Tuw(eg{uJo;-2KcVKqr>1tQheggrbBLj9_'
'{nD1nfD;XTgclUmZ=eh}HNLNoQ|<C3B9y~)3F03xGe}Edh7BtmIkc2ld=Sr9SI7nVG3AyKoLBhvsj--QXRn|Psyn6+Yw+ISwnqB{Sg_5x+XAa$'
'?uNI+);+`GrZu>OA9w0UlFi2PgopzMY#Ok=04)n&^$%MYNqLi?UM(_)8{IeYCkz@O*uf*;Ol&3*)z4-@#bK$~kfy4U@&MQ4S8@dy_<BAaU#xSw'
'q6{cmx$Vtns1Y#$UE8|mWJZ%TP)SjYFXFuF+e)_ztY1@YbrH&5#BQzjZc7U9OX`hnB|XDvToBc(KipxPDmzgUb)#;0z)nqx=`6QtmIbS)oN<A?'
'!q1MAEhyYGa{vuEFVMhz*5YTattY@d02}BAz*_aa0q~Qx+%BN#LJIDHnGz`owO)oOAPSc0dh&U!9BsFlL1BhS(~3%NiG4;_Bp1ja`F^~BVFGiP'
'vYrx^B={JglG;nOZ>pV2(y&^L_B%UGxlB1`+;j`ATTN~|+xl}Q8-2i`m~p;UfoTHgKJcW~ic8L@)px_Qs*5ObBi(&pVvO8(@uO~b_^(`@xHl>m'
's~ijEz{q-lBbhu$s!X@{Qi#KJ*zdjcUc-nk^6_^+KV>eo#4rc^v-xlNyM2nD@sXXshI$px%>k$#6Hq$Xm~vzd4;Z51uqnVWL0nXU4pBFChEC<y'
'g9gh-aMwy(ngTssb=_c&Sf5fKm9uflsDI8Q_q5|E5Ou3uq1K60T$@!@IK!7})}Ey<`?S?!9gy2b=V_B#GlroL*=B|{L-nfro52!g)p`13g6x%o'
'G|X@Y=yDX#=Txa_r#;J1tFmhsMK`RQ+30U*nuR2Oh-Umi#^%#uuYd4-z$%89gSad~>>nQV2bf+L&hYTyd4GgJ)wItBWSYf{F;<Y^)9uThU{$Dg'
'E4Twn9b7Qa)>GVo;mNSOe>gb%2!DFd2X8+e4fpp)F(9w=DiFX!RFLM#P&HVC0i%GSGp9?!!8FfS(50blANHf4IApmR&R{De{xhD<IlWjdny+44'
'Dn}SV0+5<G;357o_~CG+R&b4$5nSVY45a$;)A8x?Su1McohAk}^_bd9JK7*m2S0^s#d?+G@?AF(weSyMSpEWk(D-4!!1D9#rsaxL`0?46o-D87'
'Kw81G%t5=?OxD_tS`x0c74hGw)fFcg7{;OLtYKDU?~+MAYemQjY9=?IsVhmx7R_?AK}PW_0YHIEObXOp#At_SaB|hdP#}dO*AA}3NP)@|V_5id'
'1@JmsP{7%{_4VOFpH2I6vD!m3BW@!mv$vRHFn2HLCmcQ<X@?HsMOq>^wE0YhVsaT`^S7-Qb&Kqp#)Che;?}ARQaA#>_EF0;#;9JhRumDj{ky)f'
'Oj^VH1S4L;>3jiv6ZUv?5*WUp$}KHW{$m@*A(1<@LTbsQ?a{P+etzP?4zd)WLo{a2q^&Bxlmsq=A{WV+ta;RwRf(GOtPDIHsJYk~8J*0?(wgO9'
'i?J{$lq9CF@=eYxt$h#&0kfq%=u!3JJPMv!9jH{sswDqDH;d@(#;S}wunG~d8f6&P`KeV<e0*m~468-ca&k(CH8C2>)SfwZ3!&ep4r=;exYfl}'
'js2I*>PRW9%MdpSb?Ccvgo?h$^+B&{^&Tvz?{ea+gyP4G?<~2y!_~HQsAj4SoP*DJ4DYJ2Gx|c#ZJAw#tfq8B6bcKQ7)SDdpDlE)$;zLovE@n<'
'_zCT^k5g0AhUomLW}iH<w%&NF0tasl@Ba0Q!aXrwF!B{>qu+6=qS=@p=*{ln-muPu>c)Uj(CRcU__x7-q5Hta!T2hfW3Wr9go;>r2-rWqOmZ_K'
'^aZe+#FZe*Q(<bv5jlKy^!m@Q-X8te@%YuZum5~}HhzEf?wDhLA$kuz!GF0ufBawNQA~4_kl1X8DLXf&r(`V*3${kRZJc(+6Jl^*E#qcTqe3)!'
'^twhn4Aww%+pi9gL;2?9`^}z2pEi7?@gu7Ob?jv_J+S0aPxhFin6tKnCmdCMWBf%#w|fhUfWPk;OExU^$8TS~J$X&Rz+WAm;vyR{*y=AST58Vm'
'*H9T&k5PIb(qs|$;a{i%Gt+~_klgNoF)A5_>b}QuZuV$sd+JQ7#UpoRnh4=FR0qno?v6YLf-5efcpSFFR|f$vsER{32!4ay48|MtOh^{L%Ens7'
'CVSTJpUYW{(Vj1$02b?|6}@6hpvk{y7q#dW0sHt2g4ZBBA~qiNdOd#UW-$FJKJr{pM{9?%@{`)c<C{1&3sgR-FlseQSuY?JpT}m1e}5#uzkK-g'
'=kc49PY6kjP@t&-z$n<GMTK9Z5dMLqrju^yrp0j%9Yp~tlHVL|S>27!6I8=61mtP)=rJ|TWikxeN&`lvwCEVtScT~pE6}0Tj3?x|CSfY8MbYC_'
'aS3sLRV{>$H0za_IN`Sq4l<HHREO${IbBw9H}P3@u~TK4!ulp_7j(&9LPx6O+)^kCyCo~9kOV_#a?^-kYbA|f7Dk?_9lZ2uG6((uSP5Cb8z6ky'
'O>h;PK{MQV8NSU?o_}<>?LFE*P)k=?LVD(D`NK*0TEi(W(ZNZ#1a&bW$fl3V*{x}x>rR`h^T<flhORCAqvpF=gRGwtk^RhE#9+PDVWz(lhmyTA'
'B{S(6Ae{>gjI}P@IE;rz97f+p3HuBrYkpUxg}qe=FF9-m^E5-`sI8kJw#eSHj>#=@@8wv<j8bc8EuLPgC^&p6>#I5d$mU34z<t#meCWX_#5>6F'
'oSb`6a$?g&>s?1B4n`S@>;H36ToNRL=o=wdPZcWN3qtjhayW#Z3TCRSYBpH9H(f7Qg_B~#k+Sf^h!`^+)7U%(<f!lgMo~?tQnphi!@q7~3#wmj'
'yNFU=!9o{BX^{OaJq7ZF^f=3sLbVAnAosvf;i+M4MxIgd@Ijygx{_Zlu2Y)~t3z64cVO<EP)Q09eBeAzmIa8i<G7flDMtu5vv?Ta-LzwNP(|xt'
'Rh8t3)hA;#Y@D`dhmix@#<{D<d#p7(e9#JQ!3996Svs__z#+!2^bbcG<BF6}{Vl3S-pFrthSZrOUq1YI3H-alcU#bhYPZKV&4VaIDjfDf1tfJl'
'kz&6NUeDvjiUZVlwaN(a>?Pnai6IDG&y|meICcO}l+3eLBF-HXY$Ta#S8ky`hC6pt=I(m@EL%HFUOtBNq?sSme5cc{*VCxi%`By>v1tY;SOA#}'
'M#gOSQ|rT&7F0VGf08H-q20w(PivF_)0(T<+}HLX-6=Witw>~}|Gd?h#Wxt8BH51NXbfqX{2j_-gj{|^Upofy+B=69=+3|_CFj<?mQVqTayy~F'
'JS1n9mgR#%E>9M5y5w|WCPylER+=S6&4&weT@Rpb2jNcV@qk{9&~Ge!0sHZJ8|XR1_JA&cq3)#_@_f_%+BYI+1I*0wzF-ewH8v!=RgW7tfqk_U'
'-QljGZGlB+e3ZMgAzGifc{oA)z5pO_^K|tyro;d`NHhKr33@AT<a7cm;g|<G3Rdh_7*U}@XI4AWqGT@c<p9<{+?5hREi$?`x09Q!fYhlT=_%Oo'
'SXJv<qw>O`xw^Z7P?$cWRn7Y8UN;twvv&YP5E&hs8*C(dZrn~!(&aC|kyXaxFSwPe(MNBlS+VPkZOewLm3r`&S}p5MHF~$V)z~e&Lrt?Z`$%f+'
's(d~Fh}+=k+-;;-uU7K}kwTo`NwfuTQ)p3Qq6>RzYAKseq1`N}M3~&D&P~WEnE;n&*O<cTj##;ba;{^1Xw5CF5t_PSKit=6RV83*S`<4mSAAJ#'
'w~bY+gfKf4UuvLbo?^&tn^}K;7|FrL?1*Vn5WDV~paKpRUvv>K(-|=zT<!tvxe}du>uRw#T@{GSquc##kxN*ujA~qKsxMwfN_tn|Gt`%UdVY4^'
't!4$03VDiSFHHSzy^6Y<P*o)H;lnB+id5I*7%Lw|kM@2o1`LPnd}>=cst&gYkaf8WGk&rOan2(F1m^&HWO@lfYd`cUBAno~{sfsg5?MiV?_m$D'
'rrFe1AbQAbHI(pua3#hHsQ5=_u<<g$UiR6&t*R?g>*|ad#UiF;ia2iO{Uy61AwMQaCS>R<t3eR@8oW10RSEoKP8+&$>U8QLL|~2(+E$^1&xtZP'
'%iE+(s5dBwpw{Gli0z>M#yeuG)lI!w-XS!{r!NlS;KlmdvR-pu0=hl$MFn)*AJxm?hdm8^bXdD?==ka$pA_aR3Je(0X4joif}J`A-Oz4&tLs5p'
'>ji7>_~zmagJ=JO`ICoFlPlXYTPESpI&H&mqlsF%D8UU_@8AqkRqw`f4@R)|l0N{^R*G?NLvCpyqz%3LFlNVHzzPvRY!wx{=GsorA2|mdZ_LOv'
'nmTvYwROCsx$Flmws9@z90k?ja5v?!teb{v;3@^PZTr-i?ODzCAHbWY<((`jUr(4z{7)j*k&Ni4G^}SPT$W>uvUS_48{f*nf}X39neIS#YD(3l'
'DK0;T*(0su!9nmuqqaSg^cr}lj*|F_q5Q3GL{<4bnWamTQpO-|sbD={TPBC@ReV*aN}i5SOjKeT&Dc@=D=u3G6$Qp1YCLznX+L*2^q&jZs3G3_'
'+fN09^B*tP8#W`qq3@It@*4V37%S<A-kTnc-iA$xjt*`kt2*=|QnkWEz0~%52Phhl=+J{r4Zw9p8P6VS`|)1~pDK`9#CO3}e2s!CIiF1>B$cLL'
'0^2>PMB%yRMq-krt5UUB<qk*K!~!?zk`#!<)ybtr0n919%Q2DJQqqr8fUFWj2u$;MrW>ScVh)x7>E#ez^SD3=1UVb&I9;yxJhD==K=l7hJ^urx'
'p8qKVg1E6L+>U&%vXr!!K~ZAL?xtfL6(a?X-C+NmiM&Iz>+TdN4qDB57;CpXk35K<hw)cd|J!&i^TW1Wo>^|^!OYKt`7!h3heGqy%%_>3hx2T9'
'?>7I|yv^HqIrHNYO**^5U48r;;9(0bfbpOE-7KKfh#*~xY4bB=8&Y<v+`%qf<_yEo6Yj!LRyP)xs}oE3Pp)+lp~#`Sc<|N`S<-%t*s0nPg{Nw>'
'et=Di@+T=OhY~dtvy^Y9x<~tQWmL}4PaFaaSlg?9wCRYCm$KL2!%%Hwhgf~Pjotc*7AmSL;{XSG9#Gfb!=e4toIqZtvv`7Dh4!%vEo&`hJ8V9y'
'`z*Ma%G_xBU2~k^Yt<%0E@i;fvG&(%Z)QjbV<~5vo|#om%QM5?Ox0>y+sb^(V=;wU%^mBgExLS_Ow^R__VIcSeWTWeRlsG{8HRTb)|~K=T>+~H'
'ok-_1osQ#jI9_Hq?T%blOw*hyR5uqelm8b`O9KQH000080001DUG8oB1Ih#d0C)@l01^NI0AXogV{dJ6Z*yf|XmoUNE^v93R!wi?HW0lV@PC;4'
'BB`|%@Ad;|wmKwe&^>I^MUw3$2n0qN#cXI%C8;F4LH_#=DN&+jXN&q^)8ymK@Xebc7mLNEVYdRIQSu)w4Db?b!_8-4b!jIH)Oz(uvrA}Dtu|5{'
'cz<>E5eg|qDIaT{&S&%4IX<8QB@Hv^ZE)0JBq%hB%z@d^vJqI)%)c#alfsD*P%t4-Y0w*{5ZHXSm9;do<^`0<O2KP<2ai(STM*f5RZYfPja4fE'
'm68g0tqnmzR-gr9!|OY!XnxD<)}Re=u~^J!m6BU1DL})w6y&y%%77hzku~<qxMf_Vt>Hp@M8lLuyDc{|_uJ>Qes#;tX81!#q&Ud^IODL+bgP8m'
'x2ZGdZPT1?1UHt#`Rvo>*+qWx$JzN6WDs4xU%&YviJr5P8>x=orfY~d#x(l)=;)4{O?#Uba(i^jl;U_w)DUY&db57Bs+cyc;VX@b(vn2;Sw5tT'
'Oh3QSgpkEOjfap^@^v34T_lBITEm}Ma%Pi#%8GkgRq;hz+XJ0h)|2D;4Cqs01$oYEZt^_VD5@n`kt~5iFs-IvEM#43kGCHSOP!Q52cE9bA@goK'
'0lL8i;zX6R%Am@72Nv3m0&Gj7jgxc;u!`KFoL6|Z-!HwWNS996&z--QcMy$s_C8hA%i2ajdpgH2nlfZ9d$xfUhCg5u^}IT=EFz}Ndq#Ru$6gGv'
'Hp;Chj-#59T1m}Ol%zsFy7MT4@FtGZ(C`6H;s+*LD&6mwHczY-Ci`W*_Kpc2B*oy9adfM0SCV#4k`clyhf29-y5KxcT$kvlN=~AT!!!qn<F$z`'
'E-+?GiToWbuPK%s+ojZe;dLc}YdEKxow(`^I-tTvu!8Kk4VkZUs_Y^#+XuI0Pw{%`N;xf%i@5Vedq@jgVgOdIklj?AB)tWEDzGsSUwv-6l32pu'
'mbiZW7nNQ`3EQcbz-yn7F@MDdi~FHaLxuFEGAgH=exxP<u8;%k)Otym1Rfjo9U3RgT|ztHVJf>2gIx8U*jF{#ciY25QRl2}=gmSQ`+c6kp|{7$'
'ZZPBa-!fQF6N@4^&b2+t&Fk?h#;L*X^{(BfH24eZN8Y-AnJd(dtTkGuEwcOPb1@L$7t(Hq0xYqExbg`R8=`@(FPDhu!j|dk#FlB)lRyOYX1~ke'
'r?<blvLdD+_TDo1dHtiC?u=8~?7B_ezO=q6nz38H+Ak8{+56;Wd&z?zc6;qXB{m8lxO80orx*upuSeVP{M_~W#;U@HGZDvB800OU$~EwXLQgG>'
'jO;anFoXjVPX34D{53f0t)iz!t1vIo&`8JBPnPyYkcC;Lm&@<c<r<$FNp3z}d;oer(Ung|3x)2j#e^P}3b(SfZ!e<I3(viOI)U_<3@8!IP_uV#'
'Bfk(WJ5f!<z6%a6=w19MWS>VX*}-X`+VC|nb>i{lDX6VoPd#b&H50j4U~Z@lN5^1)Y?$`aw^wJLPXa#f9li`96X(ZauXC2KBb&eI=0FM27Ow51'
'lhp-wluVPK>>Etm#@;JV78z|yJAE9eLbrNg-rqaIHOzfZcgX%m(|IKi!@&=@c03$v7*luGJFaegh4?Gad6>_>0#Hi>1QY-O00;m80AgKY6}YYK'
'4gdfSH~;_;0001CX<uV+ZE$aMWnXD>E^v9p8~tzF#`Sjt{vWO!21zZ&SW4VwLYXxS+M#GybZyXI41+n5Cy6yhviz{EW%=Lt-hJ`zNJ?_nU`A_`'
'$9M1U-uK7F^SobKRi{PH5}#BoPgupOdCuCVq8VXH+K{p;wiVrzO;M58-;p=PzAS1MTwYvWyxlR9u`P`c<lQ^*A#DKJPZ`bMy#suEn&vfW0Bm0*'
'ESqm=oaS2+7dhaMn}CqFyR^Q%fUj-FV0>Su8LJ5_o90bHXinH;nWeF@k0#x-EX^6I3%%o-Bn8QfhV1A)yS#vz+m_We*%c1})>bj&(|ikwPcnLd'
'xju>Mo>dg~h@1Ce)r8PCNedDKmdlHZWwZed1pzw2*8&by0RcSEyS&&`#hygbrfu4aMG;AX3RMG807emCjg;b0&7s88ONiI`!6!eY4Xm7Lkn)hw'
'yh&pP^mkgbe<JC8@@>j8U=k-}bWgK1p-oZgnKX@J!J4QnfMN%Q;%8PD**#0X0cI9eO0%E%=l--h-T}k(_P&#x{+ay_RA_wSFxc0ae?T(gMlLVn'
'i~`wz;Us^PrR7u~V79uvAn<dS=E+K(`^FR8@qFUtw5mA4yjv0y^+8PH+=#XyO)Ij=3fce!r^hs58`@?~xCpL%vSr~F{>@mp46bHku0Ryk4Xqk?'
'wwPc8GTfg7))-(?gw8;yAWvbeOU8N-_Q4Q*!4LLAPpcTM3m{0{RJ~bV%`z4@EQ!{KC`l`LE-Ik%o<2qy%eT!gyuMP1*nOI?9FAVGO}!#nS~oX<'
'^wz~1f%*cPLx>O-uElbO+(%?zI|4ez_t2voV7Mn21bjpSu;>8<qIjsMu;a~)%wG|&6CY0s4gRDXf*?W1Ch}Ei4GE5PGdPiGS~K!1&06+NRl#;V'
'WB$Ia8?t6(3#5eYfuRC}V<J-<HrF?}syATE#Hk&a7zVdsp)>bTF|aZ@U(?jLH{5=G&kh@y#risj*6z6I$299fxT>{~s<jB7U(0!qV{@+LocUF-'
'yymm;LC!ind6s+Bx8ObQ`sX~YlBZPhPSQ}Rflb*?Blc;oW#7_-ml6^Bl6!8>Mif*6fQGVtxubPj+v@pTt7qM=e*^s$c>Y6vL?`5;!$C(QuRdNS'
'E1GYap^ygAhcwbs#j`X(I$R6_?#Q?%o+<vnrgcj*?uC9SS=^Tk0Afv)CY*$y%8K`N!SF^44b_wVL8yl`X?DY_IgmS+Zg=_sN2OzM{=XwIzYO*c'
'@fOTmUP+4O>E4oFF9#gb8y6*nam}a}yRLX~uE*N*&?l<f5+jgL?yZ#g{G+1fE{)%`1D4Z?R(3^khp|cl@o|9<S@FQCsM)1eg0H`~1*lI<#Tyva'
'3?TVqL;0T_i|=qh_yVHG|Eho%pVu_mGVk_}GG`5YY;1Cm8m^AOWkzGRXL(~ByeUfD9&iSZo3V{J6u_{mBZC@W7fn;_MHnYxdQzn^bGd>^8=COH'
'HP|2)wPp7V(Ls)HrPxEo?itMlc0NfF>wqpy<CRi?GiZ#vz^4aV<q(2vRw>qSWNOA4vZ^AgC}DAm>C>3){{Mznt{v|r{f{x@TY(SLr2pNDzs4B<'
'93_+iS^}v|0cjrc5Ef2je7G;tm=*UzD(lUoQYCs^kCL{Ma(T8Ll`}ssC&}oVWlwYH%d4}jsK}H=1~?>?9lgPw=OUXMahk_T<}f7|b}mV4o)dEN'
'KcuDL4sGI{GzYEHFPG$Fv0&##uFvu!GJ0oQv|!P?X!C?tCrE**F{ZSAB0!4&1`_@}1zV0Tl2y4|7l|RoZ}%)A>w{5`z#788UI)BHpki!HucLcW'
'Z!zV0ceQv%(>Y@OdMUGCd+a#L>||09AcpKo8JsSkIwOGn@`XesVzh|IXm&Wv1HmVlFpct_2WpWhyGWEy@b)oir5REfJ1h4jGpizZp+Q*rk@%Eh'
'dIGk;eyJR=Rx*XnR=bZmZcIy#5F%2#&aiT@E{Y5gy{%eiIKv1ebd#b}lp$Q4F`7qA)q$i`-BWl$XD=c^$ApQ@PiWwBX3c=u8RU^|4<{B7#%M*7'
'H0I9F$I{pM!4XV5GvQq$F!YjqHj<|>SnIrSk>UuFL{nv1c&TgEFMio487ZmxeS*RecfCzpW=!lG-G66YeiqJ<!dbx_fL6l3!K12Rm?#=^gQDw{'
'2_q1Qsg+DClj^gfFJG*BZnwuIUmh!v3NNalJ!~N)2LsrWLr(V)w3lrj?;w}pPC$%#LpYuf;E(}71ZI)65SyZU5JB|#-sJ3!r+E2ZK71IRrnn!O'
'wr$!EwtY9c_El1!2yK$@UFtN<ddFKSVKA(ybpI<&D$d(56xjjLkQ5aUt>$8b<SVia2KazT`ZyF*YBSs`8NCCwf#Bpz@&%cK=UA9LdPNq5Wf15Y'
'Sq8igAy`}~1lK1p$SEM+(?zg+>GJ_qDM^#E5Vl7g5pWtOo{`T1(e+YV1Ri(;GXZ*zE5F2*IUmTM$x^SocxefVSvBeo!tsavx8IGz+9CXJjOe0J'
'cw<MzPTdU?If+hG)I<dMC}Nlcchdf842mEM3DOF&=(9fo7apYB%hEzao{{AMs$1i^t73&hj^#JQlFFV9Hz-kwLDZSH6hK$Ko`MRGjR3*wGN{cx'
'6jf3WyM!m%YKYrX>bfTR?ehVpM!8pNtfEFER1~z}4KLXj!(+-`7AI8(X5JrfYFPoJH2tbS6$!oW`L0SeXvC3y1#oUacyoJpz^6dZ32QXkwyFyZ'
'4?xbu2zC;=%7SQ$I2~R=?^;}|VN;}8u%@*R<LjAWepD2U_Jls7`MK73%;vz{9TeuILxdCoBu2B!)+fwxLWKig9DVDWQW~loa$wCZghTrphG)hO'
'x*-SFDrJU(+#?`}Px9jnO-Kh;D7s$O(tGt|l{LbfjhlA8=j{F9nYP5ByeX0aElb$e6kiMwQ5p_Ka?C3f9UsuQO?10(B8<N6(<K#1|FEVq`AQ6>'
'#&mKdPHlI8IA*6N0>e`M0ufOKDk!(%xV(TfE+91o)p+1K@PJ~^Etz>jQ!2g)UVz7cxS0TjCb#g|7lwK8qQ8{2YJ!$z`VeecGo9da6Mr(9&4LPs'
'LFi{#_<>x#*-fXDVB(X>XZY)H_-h4lw|(@mTX~p)W8x2ds~Kvj1iL}cFNr+o*+f2-pJwyL&=Oh(0_}Zo7PkS~NkFUFDi)ITnGcch80yZw0nSOe'
'tI+1ClEG>M_>ppI4Nkh-+06=431`Qa=~NQU-)s>Q0i!cl(aI5|&c=>QnGpR6ZJT1!6HJ9TlmG*~(ra^*+pQS!1QKJ<=|q)uCWuqU)wl}`*Rdjk'
'Qx1HU7#tYZcVnrNleldrvojTj3+lvhEECiY2Nqu;d`Y4Wqj=v!)LD#!;)=)4*`3A)GS*e>>tk^$X2X(2M@+E#;a)}TX9Pl-st}bXSU|L_MxH(Q'
'$$)pf;bJuh^)SrB8DPR8K6^ehq~$Y7L>)<wY*KoO@p@Z-nj;COCzk3Q>XWJ%$f_8j<?JT8T|dV;%a&irrv~cvsOuD<L+;r0p8~4yh9BjZM`!#X'
'#}Cdp@A1iR{Mq>~I(rg8yf_fDcNZ=(4-=luA%g0+C@QFT2TnHhB$Rv`4hsZGFDNjWX+(D%*pBT4Cnfl;V)M}oo(d#8^CR;4NX<f_^@`2L%yR!k'
'eq;|s^=WFb(EM5j#x28<xJD*i1Nok@(nM#(^w<S_##?}7gT*JXLhv71!#u27)N@1+bOy6VHZ;rdgHP6NW8#w+){++2$LnR3@H#o(5YwBATfAte'
'CaK{FZ3t?^`?O(u5fX8#!y8hYtG+tV2Zu95Fpk9(X}AureV(^V9i0;qaPyUg0de#iWAD(I!mDsomD(G#0nnWbELL5pfd&-*;MKu2S9MPHqYg4|'
'yNxFA5xcDEIb!vMov6^rO=Y70+)cv`I!H*Kgpa~uk5(*<Pa{COCo3TJ>j-$K_#Avp@4I7xw+|&t;qkc5=Wv5*fj+vG9tdWeLr-!JsP-tVsC5yN'
'i2Hoeisven=8k5pK8<k5?*B2#826Vb4W>>kg1;OU2>Zw<r9w35PYHw7P(LOLx@CUc`4>P>3<JjqbDF8#iW*!M7b}L@Dq=lhhF)R)@>B6y-tIWY'
'Xl)v`SzHD$j#+}EF=5B?jvl93F0Waf=DN6Aag^Wue#&uX$5~M$vqwPtycPpT+i8Yt=ao2~qk219M|_$$<2;vMUxu3EX10`&_ujr0*ElN1N#lJm'
'qyiltukxh1T?XstN-*5j76j}>2poRt>N+5KuH4zDRMMxfGYFaLUs}LK0dZR~=&6*znt@^d01AF5A$0cbYd)5cHnHj+Vs?X`p7O*<GIlRZa$F?q'
'&11wzOtPIm@8(gQkxvb%+Lm~|FS1r#z+baWj&ZlEb}5Er6~@?_zTcs@r$?T&J@cLjY*N1lCJDPyd%xW{D(cN@@$%}nqp!Xj)UBaTU_`KZVPz<8'
'J-Ksr)rHC{5}jyB&%(E^@@5VlsqL#Ricxt<WTsVBwOgG;>%FIMIjz*ad?__};|A{o!G{4Q<ywg=;=D@M;fd}%$|>Fle{3zQeP=Fa)$oi(R11nL'
'Zh^&#=JYDP_G3MCE&$57;f3s0!C7nb6jOrp4M-l4O-8r%k@ekny19DS129&yTgB?kvK^oo(#e(Vf$2*w{ZhJ&Q<Vf+p{i*&L-WOevC3be^=3d>'
'xqN878BofLhw8x}QjV~DG`<^EoX~i-c52_92=VHKD5<!gOdj?=S`;!bTZBqp%t4a~f?J`gIQPD2F$gv9I~ajH)M+qY2Ta<1DQ^P)h5}-4&AdPn'
'YV{sy;t7_4h2^}R>So?bi;d4cp>bRIcq`lUyl>H+Qexfpj*AKmL0RMp=DEBpCjWgF_tWcx?udzh7UEoO=(a#!g}jQacN&<oIXl)Nmwlh%nK#`M'
'x0d?kVCoIkaRNwjI`mND34>RS-%^NP=t0-DC>0Ce1EFN!r_ab&ysPb&cM@Z^=y|yTLVu<yAN-b~mWKZVP)h>@6aWAK2mk;8VqI^-2**JT0018$'
'000yK003cWUt@1=aBp*EUu$7!XKZCIaCyyHZExeo5&mw#|AD<ASh7da_N8eMxh&w=GUCQ}Hf$wn>ly?_t}I3r$#R!EyRQFxpV=2_CDB*+TYX5x'
'UCz$T&if4Cy?b|6@xgEP#H(t_Cv`pd^0rv=*6T!3J$Q1)J^ok-NsF@1($YImg*>^w@Y1F!i!7BzT}AtQ`+GNCCA^~YQg7OFG4q<X&bSbv*YY$M'
'SQg7vaxbf^Y4Omt>L5Vt1)V5L9(rk&dle_*0IsCi-<$I%5qV=UA$UtKTAp#R2SQQ+)uJgmaPn$dv~{)MmGmZjTDROQ7ERqMuG8wtThw`1asfWy'
'y?eL6r>G{$w3A)Slf(c65<*aXfwbBCP1MzDQ44dnN#(38CidVpmMb8^vzE&>22p2oF4xO$VkEQPdRjlrV!>C7t}Q_&;%!^E+nX)_XUD}F@4t$M'
'q)3|B-^=;bGXM$Qz@MZ70w%UY4??BKr?O7-_)}f+V7R~M;V(~R>c#5HQQj>Y;TxzR;#F4X+z-5BYL9w_AfxdJgU3t3)sx0#Xam_Mx6zIMRs6iy'
'>=4zM{ug>PSPE}N{LKD3fQb!$8vTt8iCuPDR}$JCjGr3LhRm_O17erG1e&mHpUh?G?{gt%+Z<q7H@x!AIwXPDos7Rf47^l;bJ0Nm+`*zR1XHQB'
'Xz233g)?;dW5%0Jg+)J($JZlzBD?@_Q082wMHyo=BJc2nDCbJR>ZFKS%bPOII1BEEN8cVE-mj0gX#r8g*@$t4-k&le<vkNR!t$cs>Y3#GGW4D`'
'wHFrXeSod4ByF=994cu$h@`G%=s_4wCxtZ%ILzo#S=h9w(o!fdZTYg`pATTjOJ1c_#yzb-pf0<dyv}vWWm-bik#apk3IE99zwzC@wH7#aBV(d}'
'z{sm>QIKvE^CnU{*1*No!bgQj$mIRNk?d57TX0FCXr(UU+cPBYMDj=J2SN0}rOy;}vPi393VUGT-F>j7e8X=9tK(Jfvwu`9`mHW1U&z*1#}%e|'
';DI<JG2?d*1~$>qpCwo1EM&5fCC6_nWm`--x-nKTY!1U(KP9j#tVG?lnO+2eF*{$2?Bw=#G)_)$MrRk}+eki27DS)hLUMgmFuY>f+(oncsfNZR'
'Z=qK$G=Z?s^9ay}fH4oHny5)zsPjM(jjD9P-?M?XmI%?u9#KwbqEYnsqB$oO89X8T%)-^Xi|gcUbbfg<9-S%Q)3~;jNpwh;{_BPjGMHnI5XJ3U'
'+h;OE;;P#xySpnUNEHFU6YFz2NZ6(JGQHfk3i2*(vc%%%I~676tk**ApOKu>ldnP{;R!ZxkA2dQ7C_#nQ<?n!KwWB5r)_Q+^;WMCr`m$U;UFP$'
'S_804-Ttjkrlm3j*o?M>S5R%=#4tG{4h5iiuFd12tS7$VKIpstj)-*nM?$gN;%QM9)x1}D{Q}fSTV5hFAg0M$@e*vw6yOVD*95vgwNgU&3GR-m'
'q9>tUb|3r~EvqZ2E$u2`MbzaIZ({91tM!}gL1A)?!XzQ%OY0cENXfBPU8NYc*!5QIL_u{g7jh#W8vm`^dF%k*G1^J;qJ;pC)$3?6$I{oA6!Ez2'
'c&Mtxq@JslL8U|1CNdMoj6fs)*g$BAI;|h9#Dln6WYWdtq?5Q7ky#?!HBbH;M5)0grz5<5!*tX;s8FX6w?Jlt7F21W`qK&}&KxCjwX{W_UL`uY'
'tsxywqpm?(<o@$~xYRizT$15!ob)Z-AR>bky2e_%)z!<c5LT!u1<ivY{VW5Q$d&-?K^8v0tSDK;D6wJ}xjOU_Ve%w7y+o8oEtXI%79zPtq@2cA'
'qBNy$B<ne^C^Lz4%KdLXyw2|~(wOQyy?uFgdUBcEUj2M?I!ewiZeqHXwin}2ffLE}nUl-Qt6!6!KV4pYyg(jFZbth2ZOkOhhlM8>$?4U{>#N&Q'
'^6S;jPb<7+$1XYtPV;dN)IXe@{?vm=6FnP^M>iiYK3&|7FHS-D^y-(<&EH}+MU}*@VSVNOP#`GORr%z}8ZA`aOww$A4CQ)FH$jT`WLH(Z45?wt'
'yG*r38N?QngtM~lat}$V<B^kIbx#rLA73R;dUgz!meB;HeV-kD_%1rczathN25@0A-&g@a(ez6(#t-g}Og%=$c|2t^DVyQJ!RqwznGU{~I;^a9'
'2#HCM2Tb>82s01s_pybERK?9wD-_cJ#p_-ux>g`}XoV0Gc-F~uf|igC7BISQS!8uf2{|niT?Oz&$7PgcL;UAvI5rr}YAWR1dt;NBXCbMco&wq;'
'*3oYeD7$x1WS?^#MH1ziNfH?Q3t`f(rvP0}%>yn<kEq_=JKc`Yu6`byJDY5SqW9QCznxG%P~F!tTEdLrr(GGJlkN1iGCATjvLTOEL9(b38|tda'
'{NVlfhfc#Y1H0Zf$e--t;cositTuI7Qsh|&ot_NU(|WK=H4~tDz!W_O&&kM9Q?>FKK}a+_lOB6dLbtXMY$egnYpyhqHtD1&QS5OM?{>26V1Y6='
'JxGgzv4??sf`tew!7kPbvfX@h+ca)I+?p;Fy0`1g(n_|X<h=2Z917I`Znum|T4UW_?nq0?sB09yoQ&fw@`A$Hf~TTG5kIKXRu54I2*YY<!IbhS'
'_B*oB!F<Y7nstc2_NGs*w|d+{RRh=qvn5S#az6A{WIan31qt`MDqX?=$ZMIyu^lty$mgpPz-+?hoMONU_CeTHWwAh#$`dLHXwOs`uN1P@LW(?Q'
'u-%N8<p5&bNKZ5V=pTJ>#R)fjlWi$;3hgXp<2+Hs&mGgx?EZM=+bS;V?bW6!ocd}d`uhr=r>ujzG=!Hz_1lmxqL1|IsT#-G@e4J&tt_S~OvaRi'
'(HfO+NgixAH5-lL?xZI~#-JK&Lr$vr(NzykHx^u`YE)zjAsC7%C|_M8v850EQ!UTyuF6MxY-O<+FcCk$yw>5NugPZA7`JI9@M@uh^8^hmip0s2'
'Bm+i%Md{WK5bNMCEjF5`JcY0_^Db?Qoi>G{%!-3MqIe&hrN7m7DR<UUq3=RFXxO3<G%IEO!ry2iUlIN@t{`Us7*hVT@`Rr#@`4R5$u0Y53%F#7'
'_oy(p>$mxO3<1eH#mkT9$;Yd+(Iqv+>)l^2ZZEDrz4CHAzBSvy%S*3`sDZ7|{!&kdEYwCHvVWV2F)SLp%DJXxCuZAb|9WmCq;4T=hII)%yRLE='
'!*?GJ4>$Qh=U;ciWY!YqVA|LC|0GSKd2bq?zcoGT`NZ2&AmlC%*n1=X-4uwTACgGhr+96jX8s`;4Le@Ff5l8StfPPDgc(@+JD9cMZ!&^3!OPg-'
')0)Ne12RYG?zMahZIDF+CH!yeHI(vz^dQ)_%%Ctu;dik$2MU(k>K-`8@#pvt-}Y$ISheCU{#tm73wp4m@yxzfwE8W3$1D}>v1^vwZ0%fSOG<cV'
'K}=l+m0Rp>_UGbIBWBYIkMaUB*OYGCT6&$f{qWL9-PXD8QM0FtkKRcTc~k#5<_ni%r}p~~eZSJf5l$l{^*KpwUx)sjN}Duw+*1%b1>QIRdx2*J'
'O8J9R<z&BhQDLkBk3BVU^X$ly7*7jg9D!QL0_#JrUE4vdJFmBJp&i*f?O(T8kfT4_!enTJvem`VJfNx4D47}}m_<OdP&H!J+iSOebqX6eX&owc'
'!}c7~O#0F?fGS-@nl@4;i~&Fg=6j(#`rkeEYCfe9sYc~P9W2e4C1lF2PfIUIvm~BXgfH@!?RetQ%VtQq{$c$`q@ArmttEcPz<jbkp$AX*s=8h8'
'I%j=zaokl%J5K*%wO}ZHDGyuEh&-CU3uIp!66-GjyJOA1sc6t2&<y@_-A+I<a7aS$uxAk59p2k04nEst-GMwqz+c3EqHPdp?Pr4>X!R|sbUVl0'
'J`zS~&35BkrJXacgF#Qsd8420@?j_w0e?+F5$$)0zAn>RBkJ5mK|}fNykKv<g8ZBo(zo9-kG|~zH#My@pRl3pu_+LsUJ1MwcwSKHbKFmF+Zj;J'
'L|KWNg!ra^kVukHy__V-A4x(Z>m;%HV^eo(oK1yr+CD7rj%>;>{EyTR^`ARmRkis{Z4fq<*3f&FPYp-Yl?9Pyf<B94<AwDbq6+Q%d;bAYO9KQH'
'000080001DUA}qY-m?M#06zr)02crN0AXogV{dJ6Z*yf|Y;R*>Y+qq;a4v9pZB*NC+CUI}N8%qw^ueYgZqhzfsuZa-7nMR9l(ed<s<IY)3{#d}'
'XLb$Y?>lSbU`3b*JeM<b=H|g*FjdQ?#xra59#mOs%9JN-U}c>rvjo3Fz-MZN@dPrvDXl|bTY2Q9-|P1tDg!(6E3gJ(XL*f0f`zS24&K7W`;W=F'
'{4Nrhv(2ll4lk*aQZkAUc?VAwRygYSGF8^*I=UNy^On^TA=89L{DQyBR+v{B*#V<hWCBOxBg-AJ{0X(eV9@UsY&XK%czpJn7C^$EJt(&~*+IQg'
'Mv`#9cd%J^fR)OxI+(eLq2St9DCbp4t?gFC@qF?ULs5EY)xrO$NVUzBPD5?j!oORO3v1Wk?BX%#Lz;f?^GTMq5^^j+Q-Q8jCYE*%KlOVc4;5f+'
's3BRRN3)HiyPNCDIK7%p=BY4Dr#HW&5k%22ah|D+TVQxtk<3;G>S=>F%$85VVN*E|3k+>7Xjw7lA{C(==?VL)jD}52VIdTVt@#s;C+XGw-TeOX'
'CY{}1jc4iYcsBbozM4jV;SwUx6-EuJ?vLnRHzf`Oh&z>aLjJmcn5Ng0hfvxEb5iL)yKvEP2=5>=hd^gRm$)ZQ9zV1k>ukwJ@j4}GG;>vY%;ST#'
'WZaUioX|N=FmcU{g2QXAIMxvxtLJ^Ey?a8PH)26XY27uxfuCE-EE{mE%I9`xnm&hVYE)YUuiV<fj;UUARR~#Vax&JJ5^g6sM(nOS*`vORBRu6`'
';tv0Gp|bUo!{w&6&+v$m>DD+1_ZDm&=55R*Qzk<lcVM!Or@Ylp(F?I;DPu*B-`1X^aMcx?6rVG!zXtB|hwMft=!urMV`}b3&OvmClZ-uijhE5m'
'*YgW#Hvwct1S;Fo-q2<HGOY0lnM_wXrp<@0+onb%ZvO#LO9KQH000080001DUF@E59xem`0O1J$02u%P0AXogV{dJ6Z*yf|Y;R*>Y+rM9Z*pZW'
'aCxOxVQ<?u5d9t?|KZ@lkTf{LhGAb47+IF8t6Rs;Y-MX!6apokY}OK~lGKu*=zrgll3m4iiw)xkS)`7~d+(l}JQxg?T&zkwdyr~_3WPM6O1Xg='
'mFC)@$ne6h0t1D0$ug`^7$~HITFJc043`2vHrSvJ2ZO=sNugu~Nm4YXQJ5sat6D1BFDMJM(fah{^rTz;prz=xWKtM>G$l{BTe_8s3ARGD+oM00'
'+~7~Ux#1OdIcjFsy#%*3wTGRaWF^xYW*xfMGF=*}@ci@yXi(X25NiUMA+L=FmMKZ$8r9##i9&6dqUA;tS0Dw#^7ko~068-(Wg0D8JL*~%+RQl?'
'L<JYzB#E!FEJm)xpgY(OyBgX>7`u9X7jWb`T@+CtR-2qFzn$oDtQs5vK2om|xpC9p<_AuBU513n6qp%50Hz^Ndlk4s(-0pqYYh6{^&+~8BZ#kl'
'oklQu3A1?&(c8%~Ucv{NYVdguar8EZ+r{MOYH<&5qWck~GJk59ZT#$RIt{#F%fm1Gt=y5bEz1_!rzKB`kFL{UKge-0a!r?w8UOfm53W&9d$=3{'
'2Wahh;#lVN=*yFW3s#o<8Hkyzlp6h@-_*J>36atDl<pFqk6^<ET}dsb(YDi1rDXYdMv56(6*Z;lCa9ED?!J#%$p-?Rb3$q}jr-}w`&ql+UE=m<'
'o7!DRhp1^ZHOUt9gqg7=G1NvC_Qv;y_h-ZEY?#NxSLef<^WpL@uO&KMRW2e3{;#$svt_i1VKR&7t(g9}n%+fA@GnPj8Nhr7*YnxS>Et>ll?Cu('
'4tKXNtQwb5>?AhE$E<8}%tJf#jzhG=-!{csOqe%St$iZ+83cvoQNr@x@9hc?d)sf*zXsm*vj=|}O{41;2)ta(Z(0uid=)Js;Q9EH_}<nX1Yv<D'
'Ta!lo!;&2++kXe@-4E{r;DtL7OY$SDtUe^?&j||m+X4HYsg>^I`VQhXwpL<&PSNW{w$^zcvv!}M)@P;6SP6#d4VX0rt=7OPprCgrW!Wa|)yg>4'
'vS@@APh=J8ds_=`k*h`k7nFyS+qw_Y_82cXte+b5)uwd3>;tcu+K~Se5Y^PA`U9UKv%VsXvaT_Q-)`@==R@W`Snlnrker1H-Fj~MbYVY*?#r`g'
'25g5kt+u}ZpG9K7q+6*vT@W8fB(r<Gla1F2gn0LZBL>&1uO}3{JvfYOb@R?^4aR$K3;5(wX9P8b1nD{QbMNz!GrDtno5$c4jF%Mum1t*E6h_-Z'
'+fc7v@d6+sK6xz_-g(E#=Wk^RTPyQBhuj|QEOUipuh;i)7)5Wi-uXqN&;I~megV2kD{lV`34E|6f|2xC<P^nLj0^7w!8?*Vq?V-i_IPZ+`<?r_'
'mt|*xm-~+K+~Ibw2i5WqP)h>@6aWAK2mk;8VqGzbelf!g000#t000yK003cWUt@1=aBp*EUu|!6X>V>WaCxm+|8nED5&qvZeg~A9hSCYvQL-fa'
')F?B~`8>%qy=3m({21G#fhb7gnIc&bvaP6khQ3hWsjtv)0g|F9+P!o_{}2fRi`~WUw;v$KajukD$`c(d<&-^0U&o5gl}HwWpE40m*?ka69r(-_'
'i4Ni@<xI(FDiykiu`j}K%_eIW1{0B{?c<~4qxUhJsQ59JinF<lq!Jo^<&*SRT1-O8{CFxk!#p!HbsGGJ_rX%kB}?T$Rx<Ks+GZDFczoo?v6==E'
'BtRJ@Srn_K2+aT;Grf?I8>Bjj=B(qLdrzG=U5{OS{K%#<4dxM)jvfT5<v7mq(M-ilHXhGbdZpxej5(87X;>M>S{UO<|EyKX;#SKi{Rp3A5)z>o'
'us0w6BOYno@zGSym`vd)*`OiK;qei}^#%KTq}3`VH)tiI)K@{GUoYW?5Y9-}Y>AgFk`Gd`Ao_$wFgC<^R>71Uwp?N<$PSXHB=nD1s+FCltz;4k'
'UpAcETZc2}KmY!hQ)=8$Tcg~vrIu#7<u;-wH0})XvAXw-2NAAhZb-yNY$=`^9%3L~MN|8hy@Kz(W^+%unX`%5>(}gb;NjX1?Grt)KjUNQb@R4P'
'si}>oKT``mI6GaIA{jp-64HkG&k-l{rHpi%Yo=tn3N>f`N~H*_5xcqL%xio0?F>!X2>7=coQ<U`m?2aV18`~fmNOTTKXvh$#w+DZx5-8$=0-v('
'6Dw|MI9=J)tx+z&ZO4s~Ehm+xR12jm!{Z2Hypp?C(wfXAm5tez2?b{pSUL-1p^aZ|93$7cYx2ETD1f&8!7x~`uMc!J0!-Qq_En7`6&2`0D}_Z6'
'H72}fMDuV2=B7bkGf^XMx0-E{B<R<e2EMff;z_-c^>j<rdX?9UthpdN7xml{HA~w6pX-|^E097?8+n}G%k?M}%gI!*$02*%t=ul=A61~`co|cy'
'je)HVf}Np38-ifqWC}sZ--t6g4fNRFnn=mh0;u=FfHi>7<c;usN&+Q<ke(u-E}>GQ;yF~afS*WpV&YCNlZ8kFi|QoqgRvz=tgetg%zz<4i|&Q5'
'SEd%oWuPGvDne0AtQR6OTCz|^bG^tx>}^&~%#=mJS%?QJ0*RoaL15|M+aB^M2?IaSES}9+`WR?`A#+g%+vx@qE+v9-<B&sQ6^PIoGUv;0J@4}D'
'${P#_<R)=A&09~uxxBi%qE^H&d{K12GMBTx$sFCb0^4SWJ3Cv5!k7}&I5!^+qO<R!S$s#*?DO?Hkwf;-YkOXcT7L}1`h!e=8Jb#EYWWo{C|uO?'
'Gd%H8uR{#%I@MTqZhq>un0cL=d9J7&>LGiNI?CA<AS5d8sH}4Kqk>s#{WcB}nVfx;z?o^O;fIwbJMzbG;PB!ej^gY)je6^IlrV%5b%hYEf_sUP'
'xCH@_zhs}k;xiFGiZN#w(L9tSLt&WFxR?vQ7GfGdqEm<(ZNc_$Ld}Ed`~2oZe)AJ&?;;Qe2t9O1=BP_K6O>h#@{{;!#Xh3K@<EsH*0FlqF+aF{'
'<JpUMxMzMkp8YfT{1pHD{M@78P8WYQEAr07|K4hRFV@6;J{mZJ`cuv_57>2j$=2Bn)t|F#)O9C|vn<<2%m87YD&w4mjZlp<ZZ@{bNG<6CETr_D'
'XZUy`RC#PO&uzn9sCy%{>RU7=l;li{IpCtqY3SPZ*zp1jo7~ic0+rg+m93ekb91lk+Dc%CRjPr>ue=D%oJ(YLJYQkGH%7xP-&l%brz{?6hAq!y'
'B3oC!*=)#&?RL8%Gq%p2;R;S~lns93Yz-{s3|Q+UAK(v+Qexy_q0RK8e%TezdTs_zCZe<3+ayjKn<r?w-ZpP<C&&x$_7<x|A02;VCfTA}$7z|F'
'4(iy<SuzhlA=^A|@k#$T3v;(H<`jj>#0yGFDFmE5DqNl1=3tgrkakY%QO1azPi+aHMXP;N!NWZ8Qi^Z(rxA02&rZoPOIkl09~BB`=D?-_)iNh@'
'yJ2(i%LJXwXGZ64?&`Ubonm;vZ|qo7x`EQuFpo{*JU!ZbP!kAm`%{caB`W=R<B~-8u4$!{XgdajShx2QugJ71%J_*HcyrRZ%h(qrwD$sL-_ShD'
'?R<YJnY};4wC9~^#{FC}uZJ~5DiiGc(N30ITV<JFu&dw+E10b`n+OVPKa@hD7Mm?tx=LsxM4PaOAO#IDo!Yf*)jGzzD_oVWamiQv<7{M&UJSg-'
'(nkPFQ<<^Xn#<DSH_1u6P)!yJ;pZ?JrjkLYeKF*kfZUSFt64nDmPg>qh1^k9uC*@M8T=fBp41C5TA*QP#YtyRa;x)_>?DxZ7u8V8P5i`*M3T{J'
'X?E+`M9;8uP`%)RmgLZrCfPobx&bM^v*sRzF22utd}3drL(Rrur*Iyx!2Qj}0Hk4FfN#faU91Ym4UT!gtMI@=*2=fxCM)y2ZX?Zuq|vOI+2VKC'
'+Xruh-F=JQ{T}LnzvBVCXX&78x62kitjcv_b12@g&lNDHRq(NmrWGhEW?tq?Ye1Pu)Y@=)Q;df!jFEJ7%jkT<Qss1^g6JOLX%7_A0HLG7%(YrY'
'KAohLGSdtl<=%o8ZByj;P=#8R6lqMUz&36WX%&xm`(W-)R4|>(y{34z-)VaQE0-|VH@goVXmcvGgr5UT6F~46sIzL0O~5*djW^a=0m3Q@P=jez'
'V0L)5j31=gi-Eeb*!Ndho|FY(Q1N4wdE<oY4lA@{X&CDYUM(qUU~0p7h4y&#X}QKLP(-h?+$m-!QKK3U4c)gtyuW_;%XOWvhVJ#Z?-)9~fA{wK'
'yC2@W&+lLlylTf@U;Opm#m^VNy4#!GT2<2BFr2~yp4pUfUZS_ye{h$wsQb3_O~-S&WjUSZXURDmc>IJYau0vq4xUWD)|oPMk+0&*Z{J-alhkh*'
'b$hEp9XHE#%Wc!3>GE##8926}KBz;dXOuIf9y+yxSEmDh4r{y5t5;U>@|=?0f9iSrDx!%_ybhIoL4p36$KPNlk<E-5W@6efyNWJ}>Q_WrL>llz'
'X>N&8lH}eS3reRR{ZINCyL4N|j2t<$`3CuXS-e(eB@_Xf8s#URZTe?U84)3Tg3!7j6XsYdH@a3US*!Qi3wl$68pQ{1>}G4wqcA+}S~Jei3wpe-'
'v2t(UyB2XXHLH-D!L56+M{;w--S1sJaSO!+C54J*0>$%kVM31@DEi11?KBAk-EjZu8bVR-dv}1Gbckqi(_cvcJ`7S#XNq**P8%F|Yfz1<yPdsB'
'#j$UX;KHy`yl8$k>YW-@%p-}+22>JZ4gI@aFQ4BW@a89<8&}5(KnR3Qs&Oc18uObfN{F^EX8}299!hCRw^!%hX9YY`Y|RN#78#~^or})E<yl^J'
'd)eAjV)n4Mhli7e>*&n2aMth0`A(m&TYcKU)6VpqFHVpqmE_B(=Yu-1LFhd3!r41=RPT1W1YB0A0?*MV*ob-6?((`i((cue24j&ZfyXH<>2+IH'
'5~0_Cem~O`y7HXd37wf_>}4R*?_*oq>i3H&wk8vM-Th4T@=UY=`N!s7b<HONWzsE@!_X2+jh;9B+z{*nS11Tc@p^ClKI|$n!ET1Ry+Pq3%R%kY'
'Qgu-(r0S3GR|bKIxEicx-7{>rYSEf_`3-g2+f7G0JHtO49IOO9p<Rm-gDy7*J*Obb#El4NUYi!Ypk*q6xmm{MMJBvY1bfiWFeUHHUa^MJpam2v'
'4_lj~Tp*S+WSz4<821jh(A32eGl$NzrNZMGyj?w=3**OK{|>DOo&RbbFU@n$+qUP@Mo)p0oFrAWAm!JbbxzN{lSPxrCTOTs*n*U|#U_3Xb_k{c'
'-iD~}1U-S)LKbO_s4N|kq}dF{pqD5oV5DDm5S`y+tg|0(&OuVybN(PB@J<>AQ#oEGw1=_xQ&3~4O3|k>)WQ~l139#(E5z8?I614-2ILBTJ>_H-'
'pV9^>vv}QY8Od9|YuR#3uFATT(BX3u^E}Lkzi6}BO;Zq64kXBVEh~-~-Fe@9uIclT?Adq3`btw(poisXv3Gpv><XY`wv=@huzY^^Kcv1fQZ1ex'
'EIrbzBq_%;RwvD#O!hZ$C6U8)Lpiw3;LK@}dpJ|h?R)20&XsR(4=~IrV_501F+02|A4U%=hI^%Nvr6a2lp3AEnav%fXy(_RxemPd*+Oc!0eM8^'
'te;ZyP@LTaZh5D)BdGX4_fP%dJDgPbh`!rv+i&~`o6Q&U*6rpLyxFqf(97(rkdd>}vy=1O?#eMkWr}AHA--nb-glI?<MGjd0Z>Z=1QY-O00;m8'
'0AgL+@0&^M3jhG6B>(^v0001CX<uV+ZE$aMWnXY_Y-wY8E^v9JT3c`2xD|dk;QzqtWoJ-%u)w}Kod9W?pm|6)o1|!gBv3dKHKSM~YDvlV)bfAt'
'cX*Mg%Xrh&yRk>);kohLHBl737h0CLf^oJNl~BB7A}gAr+Ot}gMS5bnl1J7Y1gqN}7TAk7?4>-`QVXWU@2$|y=JMk5;v*nRnc{p_D3LZ&F@2D2'
'nXyVX?7%^yk*uh4p_sT8>cnIXa!nyCoKbBlEG1cq%Zrk$Jv|F?l}mMGHFoM|!`R0|q1jPp1m3i&(uOm~N6fTrRVskL$V8QjnDHvRyx8&d=8miE'
'Dg_o$+~KgSiLwSm;EFSS<YgH%abH8kqVclJ-8YB=8oXgCuh<T}wOV9scVczPs|tFGLB7g_G9P)H71EGpzrBE7qA0q&FmfeH-hyM1B&>k;R0AMr'
'#wb9O0^O<-{YR<Y+2&N!g}CJx)rsKEPJPOF)!<g#-e0*EZ)GMR%j-gvSr5pIB(=Ae==30tyvO#&-R66%L^^SKk(OL*=4>N-1&ffX;N?4`mQ_z>'
'eSLYs@B=Yl<vCPU;F_K8yq@TSpk<-xp6{f($xC@>OomRH3&sCXZV!~os!=?}eUj;LR*5@re9&|)0SHrgWsE2%DS^OZ(}K7x{=?~tHO$Rl8-uOs'
'LN{Qks0>Tr<PN${MNv2Ax|Hw?bBn9wU<b>2QHi6da5J*4vQ00rmw>p%c7jV1*GxBxZCT`eDY78WqY{#^(nWLX;nGs@%0Z=VlUUvL+rlf}fmr)a'
'jFt63wGxo~RydejS)@WjIX!^PbAP_A>(WTW)hW3xbjwS<0C$vRmdRf6`cR|;gchZ}WY01`Ld?X14^U*)sYli7B3)u7zh7EyPNhiT?s-v)v6hml'
'G8Eig+$ANVN!f#1nLM^yi1d&c1y#*8+eujh=rx2EBM6vf2LEUY2$uwypGW&z;FdG!0k+G{_U7mQ_CcBB5Ozyg*e5)GAtGd$t4!p)Et@3g6cJBb'
'f>=BDUF;M8!#Vm7TP$hBeY0|5?|SCLG5J~M8On;(V2>$seEG6wSI<dX&UL|jrBg+20QIqG^9}yB!v3mg#BsA1%_<^nQ4FJBuNmSoqaE@iL&b|D'
'C?2ysdhl6crwWljvWJ1fW3++q3$+SG@hXPV=+z(b-L$aZPjrtlT~ur@<cpK#7QUZBj8FEFehjNKo`e0Iot1TiouXc?2WScuhGK94Aqns>h&2Nl'
'^Y{4-2-2x$0%p;Ev46k^1v?|b^+3>R)mSV}$AjS+jF9z++xi}ABPAFzjsfsbX!!jM&4Og-kT^a|QLXGUd%k7A#KA&rNB~J}C|+3sLPIdMK@x`3'
'r!1U2z<B+{p4|~aLU~K<BZrHxn~+Ga|Ka4H;0e5KWHi5t9>`HcA)-NPH2^~WDGHY=$;44Ye=7RCTiK;f%e+A)B065z-JRGW7l<weOE+EGDlJu)'
'aedfH6f0ed1lu76;evVDwZ;1LE?UPS9fZlOO&d?cJ$AeH)M><(u2#qdT`oA)9i0uAU0dzseTRrERJVg?W^=rmOk0;DJZ@xZibGwn)p3;`kP12o'
')D_h&v{LL%OzV98)iuRYl*pfR8cpkBxhEj+P#<=Fxr2&p^GkOK{;I-nt$}*q0(D1q8e;Aot;sgM(q*;p%t}|DpbOpYpbID#tf{6$NkGLVCq^^+'
'y`W*4rCbjhwvU~t3r)8JOvtm^3Vo2+*^_b1@tD$v)bBC%3!O11I*%ZU(AXgy@*ANRG{(ytDk2VoXC94PW3=W^l3b)Im-Vo4l$9(cSPPLl$0diZ'
'ki}z$%9!de@@!+(*P|e*#6}Cc@s)$K9!LS0@AVejZ`Q$yz+(veC`bgw(Zu)r`sgXgpVeDG5&&iu+5Z&<hGQF}b;M;;ia^K&O%P)79EIJEo<b1%'
';R1q`3e%v{T1Mic0@YN<B~ha6gc}NxgbF5WcnI8JYK4JiKQ6Htl(*A0fA`8Z8OijHR-@}&umo3M;vBzb<6?nz>>p?(VFC2VK|WDi%;1epooRoD'
'p69(4iB4w1hH4;3o8nt%I2Za#*Gl-_hce$@=)MJd<(tk@>^V+3iTf7y4tGHHyltKpEYnqo8ePzv$|9+eOpCNd;~cZEUvtr*q(TcP8of2e!`H8-'
'c=`Gjb@v7hwAlWH9Pkq@D}nZ0WnfFz`kAW=(INGHd}=yX!-GQfrOo5K4>Ib72JK%Q`{YU+iW68L)vWQTb{yB~cF&gM@uUJLJoAEaJ@3Q_@O(jG'
'gO@wj$AIC?ED(*u&ieZ!&6Bmh*U;G}f&k``X)|<+#>J;WD%>90!$jPZal&^sdMl*61aLYs74S&^Ou$~|y(g`nBeKNWBIjWPyU6<(AZTtE_=Js7'
'C$L(X7(kB?k84}H9=NtRsAaM8D1n+G70c30$SyNHmuS;B+s3C%%AjU8z}!OwMlwDXgiUWI8`OuWl2?(Ls@bqGAhLmh-ErLytjZIrYMOdvo5<Jo'
'<w*0x@uj@Bz-4%5QD@<qpoMoDn)Ke|(jYMSUev!y(*c=Fa<fQstw3!0MR7KzbvNq6#hcQ`h&^hhyt2bsNH*fv(T>>Fq~s@*Ozbac>jMC~{H1Y;'
'G26){p8;G@%Q?=zLqRVHkkTmnYV=ezHOmTZF5S=BJm$d$##VlY%|DQMe6MG9GPS8E6$86Kpp9nr2I=R?n&Z*Av!S*aV<y9^Mc!sGEIJ4!4L?IZ'
'FpStbEglM`I4r$9JJZS7MDufcXZc-@_~D>!Ip~ie26lf`q}BK)qq?nNtwr@j$`}p)=x5msr>ve4b0OUN;>E_WjMqlvp@rx7L7esAivf{9mqOtP'
'g@s#oCIdHc@x!KL=3|Aa^%7Z?r=b!zB#6d-4?Sx`#m=f{WxgAc+P)Z$HD~k;nX}QRr^!CN{<ldsnY^oDr5WN4Rkjn7;|Ls>H#OkPPH1;nR2Dl$'
'O-^{#W;1D3Ng-KkO6#J!1z&HHm%qM!_v?pO$=iQ^c=IxO_u}K<?ZD2@>zrPP*gJJbM|Z-b=YBMU9-BhAGu+Y`e#xXJ7$2st<zn6SIMxLd?|5=z'
'6cWGx*wC$A2M<rt@naeT5B;K*G$l%6+qfxwIQqICJtjH2Sv@=8XIR%mr`kKq1a#C6j~<i3siueQKnURKaB2=f>w-%N_-JazFb&1?cMhR(pRH}?'
'1>C>TpIp%xEOV!<C%G!-*|dzIq~0iDDj#lNZq}+eLPl0`Md)uupWV-9W1B#2hK;H6{k=smH}23EOHEhM*VM{3I(Rp5#S#1^^`1mgD)WNale;^J'
'ig9WavmyiQ^lDL|TGNbPIVy3Wm?*1|?{NyS{Py`J2$)Z^{L)}Sa=o>0=%74T1(Wg!`K{5UIM!$g3hiDfYGRf`9;eJ@-dpy(&4}dDOKTX+)F*CQ'
'VF)~gCiD0#i2fs}e)%y7k!~7t?hWfGHup5PBO4uHX49-~|0#ePTC?r&G1w1Voo)CK!1UGlw*M4jShB|L;%abeQvYlx&!GgN)oe#k@g%%&&F_%6'
'{C<mn@tLT+9X$n5;akS-kP5=(5a%c|Y$t<_8Qd`F*akC?8JHV>Y|R_!)p#*xzl3w*AeC=}&+&pAZTnBdNZ1eMtZ^jvdlz<ua^I}{+vzh_I~l}A'
'QA86vo>E2Mrw+|D&)s)H(z4CWh|zrk<BE#4(tq0{m@PSP3EUzNzHrb7Y33PpezJX7c;5p@jY2$w(>$CT4&w;TN(OYj`%IBp=kai%69IB?9uLs3'
'Oz{j+ws)f*>5@B5&-c#cPfeVPBr@(Z$P{|h6rFXc9v3|Ugj}lUD!-={a-v}^bg9S9=PJM%*{^R@vbsWaM#$u@yS1}VkZ71IEg7Yvov1`!*sp5r'
'1nbjLh|+wMhu-eMTm*ba-XJfbdbB!XMJD)$y&9-#GyAkJjj1BGiEM;%$ca+*Un^oqej`kBKC-r|M2b2YcQF?5&n&{TeVcA_h;zkjB!osBAEVK`'
')WJ@@$F6&y{4Y>T0|XQR000O8003fLr)uo`7773Wp(X$T6#xJLVQF7uZ*6dIb7fy~bzy92ba^gtdCgkgkK4Ere-E(#15Ym}U2WHUL5kKF1#NS='
'B0yWDY2J*5z?CSQh%EU?YIj}N|9fZnCF;Ywn>*)FKddcsIGp)$W;iq2D2je$iisVsxl}wSKex2v>Ok@>%Rfs}HiD=vBZAch6HLr&)+#|O!V0d)'
'UNjp)cZ(>Bu3nU)*^w+OTh$7dWrXkcjZlQvb)%@_O)alpT)nWPa@#6ib&u^W@XDE#md~E5ZN+aLlMnD{XVhWO>y4ZEOMRG;x3sG0ZN-qkS$!yI'
'joMhMzf#Hm(G;wjk)JrL3QtIRmN!5nS;mE<eg~?3sAxT=k$}pM`n})3HRB)I&n=S*xn8}<D=H=Vmx1xUlr4+>_Q~q%1%Z!bcTB+aiby3$N}_uz'
'YS1b&(>Yaa(+D$dhhTZ|xC5l<CLCdRykK?CGQmo@A{CtG8YXXmyH+_aSV`MTWhKoO@J^8;2}S-0EZF@=hPq9NtX0oIZV5-}1()09_3tU43%&Ty'
'*H(k(^NQRyP36`$0%R?#hN_`F>GfhcBO8`3@w;O9oS0=C!E?HVfIg{|Mob&<iWEH8NX*by&aXr=yHYSX?;RR^k-;ol?iT_IQ-s%Y&vKQaqhxz9'
'r`R5pdtP}%o4I|p8LdW!W#_;1bh4@LK)8w?m;mE!Ygoi<#C8cZ37Nm?ou)lP@Y1&k3Q>H=Dqb^Kb4LXSSCQnFfvfA0<}>;i^)7F``x!^FzOl$x'
'd@teIYr2q4E5J1Mgk@9`2n&YPir4X9(ET1Zk9$t!d?t;U5RHgt-c8cTDTISyv}sf~S-2;RTaQVjPY}&a@B+zzyjB#B3Zx-sg^_6yCDt0)10Exw'
'ZG#<xyjYq!jE)>odYI`|@SCu@-7x`@#hOnNDpwMEEA{xoGRvf*5J~2vT@vyJxA{Y`{F3;%?KgBEU{gl_?jRsXt=Wkb4U_t?jI7|ocp>c3<C|8x'
'JAmY^kPR5itRITfbs5>CkZc29r83Jnx^WO&a5ONM7StoJ)83HY{wv{uj!^N!8O=RN*xWyf%$7>H-iTf7?8hwb4Cssz^~?aI_kMvkGNPXc-c+<g'
'-=`Bcaw$CA7TJ!=9aZ@@^dJmd#8GtSq3N`3l)Q!vA#f*e97*CL3A&DsU)pKBZg*UQ&?d6UzVGO<5HU26{@?{o#0WAn5EF7I?BnlRm!pvfu2ZHm'
'UW`n{9p{vdcoiSW@9eL8UZ^dcGTe#FYvK@F9DpD;Kevo;whE{4-gKnD1Ok_WJOYAIzG;;N2hU1SggplX_Ml*2N$f&xB6TBnAjW@KK{TJCY1pU='
'sV3Z4y|CKZN=|Vo(=@|Cs|Ob*V!bAFa1)K68AjWAGvQDGD-vGDfQ)n=yHHpUp*4pdh;KEa)!apzinBg`&XAnweN#dNKeK~I7SS_myAe55u=K<W'
'j06B!2;ZDI^!ik90yHOqyEu4me~+IQU<y62{IMtVYVoo>$pM-n&}YmLpx$q>oR~I5J3X{PJ3O{RH}RX;p%c!7X$KxP!oOLC^OpTm8?+go(^Wwf'
'03-z8$H`YxjP3U7klEoU)xpRNNgsqb4y7Z}@&%{wZ(y*GT5??lz^%Qb53#OOYMbAIdt4{}?(WVqTt?xXamD0pLcU2Scl*+7eJX$l>^w}}m?<5m'
'?)cJN@;XIEppNIoKao&Xbedy_&dFrcKX?VnaZ1%wXFC1hZwbyhUMR;X{@a-a9o_)J#8>yeCl25-AN<s*|Aj6%VUGKz;`w2*VJePfyWdv~Q*kN|'
'nYrU-l#mqad8|RXV3D@jsX`N}mX|P1ogZ|d*o~p$)_DWiq%<pfg_}`>IAVgTK!J+|tJQ<twhH}Pv#}ZGJy@{QrL(m2_?RMqwgqo2EKu!8ChjaQ'
'7}vU`nl8&*b`g&?yE;@1ZzEjR)`Dq4YH*)3$T&fzOTRS-4T|a&zBc@sX%M~_`_PQtaRbJ?SgW&el50qets8&ES@cn(brN)aT*$=c4y2|KE#r{;'
'ci?K@zrr9IpV`J?G`W#pzkHSW#*J!4-EU-dN78&wdy1K6iJR5ZrUIslSpS}pm&t5+psD!^#!Z_kJ@%tT6m%cVe$gtpG-d<pu<GfNpDBz+40_q+'
'G=qM&JY?XKg6XwB!5J<a-X)`zVT2wo_KtIAZRc2{r7qsVnsERh65GBS#Fhebz*_2HydrBmzJP$bZ&jv?tr;1NngZ)4;CTvuPqYftwXtIBol|y;'
'p=V&~-*}I5TTQeQO-Ig}G<L{Wx-mW*WGoH`+i`tncM-0VTekAO^z|B9-^>R6)0<&0vaPd%neH?;e8Jy8iIzE$#=TDRU-HkU#<+lAUN%%LD;dRi'
'2jiygaLtn)PT#$R@{(8BlTlkZP}{l3>q1AvdV^iinBFxU>QhScLte4uiyNLw9D>ll1Fb2{YCvN=Cf`ZS*<O)%`U|gh2*EJ63eD-U0EQn3Vn4wx'
'#|AcTq8CQlsjMYkKR;h#`L^bHQ1ojxvXTOf;iUtI6NB){di83#yh%a_vFiGKDm8W(YhcV3>-{&uoNU>({L60HYkPBC&&H;$1OK1CPcG2<8Y%31'
'Uvdlh_foQt&ZxW`NAGN=wB&H{;xdM>&A{3x^4|W6UOo~aq@z)!i-V~+g|G$GCQ40`o2jn?l#iy<B}9joCnOY7f@3H6w_2HRlhZ>uXheq5ux^+_'
'0hUc_d+MR8o09=nRJz>m@(G*GTQfK6b2JZz5wxFwfip3DjT$r`OeoCGzQsD8##t;{A1aGcvR@7j0xS{^h<=4-SmyXD1(FLDr1ZT_&*k*)Jrn3?'
'Qfr|DAGwE&W2=+Z8pF63T@ly>WUCGYB!!Zw81&j9JPs7|#f9er0FIkgzNL`4+vC}pkY3(PvCi{&G!xT|f~+ZIgDJ8JrChPy@I8RZG+?6NQGX-@'
'JM|y?Hir=Tsth(CVK{>fP9KDx>>c%xF$h-TJJVTLxq*x$jti7tj`~CRF*|`wyjJmGH|^Gz|I%D2$(RGaSuspr`<*8?(xeylgEBg5;Z7MCcXyl<'
'ky2RG$8cw4JrTS1he7#fHIODQqAsUF?cZf)x@I;1aIw%%6-Q;c+p|qC02<mt7kuuYF&4mNkbtbG86oQK0Sse)hkqSJ(VW9y%oubwgaaiG*KJkl'
'>F8#d&!__8WEx2F;r)kqV<On$2_kI!BCD6nVQ!?&e3VV~JD3)fg-OhGn%iemFa?<rfaWNTPz&=IN3AO7f5dm92!%u&f#b_rl&t61%jI%)bD1vF'
'_-@wwXJ*^L){MNrQK>(y^k2f*uPO8Y@2dQu3^hD`3EF3?iA_egnn?17ls21NO=Wl^TA$6B0xX`+bjBS6Fps>y1-`b2dRtG7=m#+uR`#!pix$X1'
'4g5b)O9KQH000080001DT^r#l8D$Ot0DLz902lxO0AXogV{dJ6Z*yf|a%E;|Ze?v{Zgehid9@qcj@vf&JwX0}t9^*n)u>Lh!M1K4AWf!dfC(B*'
'y9+E_L!c#ktd1<XqBIu_=HK@mUL=yDY)_iiLq;a4!*jop34-9qx~p=Qlf0pu_N-wy1-li0cu`lRZn7=oZ9`l9wW%BO_5;c4-M;1p4kgc@J%9Gw'
'mJxPW@V2O~$d2U&{W_z2@kLX2E#stNJ6cqvtJ=B)fO$fG+ZLSc>bxr%fwux6HZ9AaKdY<qo~-Xl3qUJ&3;UhP{TR8eo9j(k-x9!-HKb@c*%S?L'
'NzUN+uBZ@&3^oIT8o(?F0z_}q)H{-<o38B|mZoYiLaVB71zj9h=}~@PWkucp{)5+5|D&w0u7Jk=cg_1(zU|tg9Da7|eN$(QbG84Tw%f8;>&-vG'
'Pc_rt?~!k6;%#*wlXtW%>AGY+_OwV96H*TR+fltdeEy8U^Y+6#Iep3g>_A?z8s?m}_Rx=%vtQZOj#YZ(-5||RC9TxxCF6B@!}1Gy&l(tu#+a}Q'
'n86%O6dvbIQ#T6KMV-;|ebKP2?Z?t?U)D5FK|S`J#$*FzuW5Fj?gu=8i))q&rBWkOys4pt{e(a{rARgR<6JZvUmPh)pdv3?vEg{}zAlUGUVos$'
';|Fz9&{EBhzx%VJWzqH@78TzE?ZZxSKO|{EF{5gMPA{~MC0DGy5MyCrfNt0Zc#hHwBK++6vz%>6nk(+Yx@uEgfu84KBjr9HDMv)k-jH=&mvhZG'
'FpNr!C?Vogb3}nvH(~JhL;CKQkDq?|^*sIgr}X15@6RuSm;^zTfM+Q7VPt&O5Vdbomb<iTN^8y!^y1=U`tQp_FpcjnF5Z58oBn!!`JeO4sjp1d'
'C}E^vHrchJ4~=myY*48VyF+oXJk2}l>gEI-4Q+ePnqVa=8FDi(vUUk5#Av@O#aD1YmtEyJC-@k>EQ*~7lsH*4@T*L3g1&$f?SK>E!w_1*IZV$<'
'32<<$73j)xg=FGv+uY9wjtO&tdBb=@q&d>mW^CV*bMXhgH03yE08lFNFpLO->g!-mexfC3u`?sFgW2UKkh^XuSXVwL4}fb3(4*!2)oiwU3|7Xs'
'j~ai?cR*S&J`_l!(8EB_1re&kESV`Xip}uZ4`fI0!ddb%9?Oz^1s{F)O=Qp~(M{-n50VHs{T3<Tg2m1sdNoL1Z63dV7^?@4OAAz(gbw#5>bFJS'
'ZUOJVEqb$%9xoL$mW~^rZCP=(ZN1;gDW&F<6xq@N`U-Fvs84+UFzoX}An_u4yfX*o4lw$C;DXV7m_PWa0We-<9}PGr%OLz(L{py9E-&g}nyz36'
'zrjNELJe)R+3ndCD31)o$?dktwqYQLgF#>9fB`;dS&iAx!id;O(^U=!m-0h!HD@>Ia2SR+d^s!NU(|46UO2G9;i@Z3F>Pr(KD^C=ISXUAPai&='
'j~BrXM;5gpQv?N8<;)1bvT*Ix@Th1xlJG=YI=;_sAmFyZf`Ku5r9>)`Kf|DFH)r1hje-??5i|@!TgHNDdG>NPo6T1?xRJ#PFsb|+3{LE3nAF$<'
'N$-P7BVie&4en<=)q9UQCZn*o9*D`S$XqCGb?z}=bCoj9xzAj!b-gyvWn}ehy7dimB(p)^6gjI?Iic4+8quw)n;pbAOjaPV2#<#=oX4f}iM&tj'
'J$C|{nuGj1*Ad8Gu+o9h0Xjs&cO~8JbzlR7gVdju0u$2KBH5=|3!$3ux@$7_we%naLOv_&JsaX6Zz<$d3jz^Ur#OiiOF5`imGfAIf>|tySq>_&'
'B4-#sUPn3*R=I*SS{e6_7Zz#Iy_+^y_A1L|TCF)`WZF7b<0`D7ik9sl2NdKdYl>OD{?;;5T$f0Oq=@Se3q`(yq37h_n*|c&m@(iDDMiOLk_Mt_'
'&r0Bzc#j1ya3{&9c)Uv#jo}Pzgg424dy9W~puy58GO+@BO@MD~G##=uzx)9YAA|j#SpmZ!Yhi5`AN&CJTNs#5EN&Qu(s3b3mhV$gGsynrI`QgR'
'7TH5?5cMdJFsy}LQE78gb`eW^9?O$s6#^O@j@5GrxJ8S3z(GxyML*`VQaQ}@QkoX<8oXR@xCSQOE_9Xu%%k$AG4(jcMiC%z^(?aOY`_#O3{}<9'
'wlxUw+Gjz7<~!JWEC;Nh-zS<>dmoi=d^kx7OzXb@BfKnxkNthv?Q`0)#X~S438BW=`f;pf=FfpgXNzMPYgUaF2z62*jxJ0O%Ec;YG%5kq0)tV<'
'^<h|i*vmQ!asv@GS8W5O(q`x6p!XJErAncF?pyio+2FIavvvqR!EwvYOt)=+gF(+85wT5&LzlxU7{&%(kw0OS8xPv<K#F@(U5`F|cxGYmRcTWW'
'pQG3a8Fbp%2UB&S=RF=CEjM)4mL?EuqGT$|RYUhsWAH<&q1XIhltPcFO)RC5xzAH2AB};YCUAo|^TRU-w*w;AoYF!+Dq;(AFJxo)M@^0;iC&<3'
'K&_E|zQtsV?_lHvvgInkSo9Ev-4;-^Y&L-tmlKcj3_y6d-<P7pN1OW;l4VGWr$LPrb`0>QE;3f%U~_Q-NWWqNqs`sqr!beY8|B%Dz%Oby*;MUI'
'72VR(*nWXxMfLV3$`w)WrB%0Evj!@|s_)jhJ1L^jxY!*Uky&*PXd^gXMJljX_0}~F#zaIImI?Mq9nQPmKE&5WfX4}Tp(xH>be9soQ*UOihcD}<'
'046TXKGP=X`z>Pj@%QTY%ILKO|E)VhMQ0I7z`kNNoYRQPlAxxm;}j~7fb{VRF(G{d%V5n7f-2b!ty;+}U@2L5_@lx8GqxlEbWFB&eI4uY4u4CS'
'6kfxGDO8t;rhzz(AZ<e{DPpMM)BzH_QSc?ATE<R<5Ctc<TLyG>oNc<&#`IF^9u@ullC_9(zXy@scg&e7*@No<6USz7wzYx0uVtU1x)Jz`3`6u='
'z~Ail9Nj4g_KC7p7VFmMKO=Gl7t=~BWI~Rl>{zy?RguBYn%{WjHn&6vqD4(fP;+>w6f=hMj^hy)<yldv{tj>#2%#<ZV8R#ii<rEazlb8;bL+#~'
'-wB@5VHTt<j%+ChHoTtAW@ms>1NAp(f$)zqNR%MWHX(x_CV9=c28Rp+)a7!-1FHC@t1|e3Em-gzMT;kSy1Cs64Mdjc!-~Z67Fv~kZeOT3Hd65K'
'a*imk9wXPce6b$LVYl-69TFkghVIzmC6j)u?LecKF3!V=z^Xscu{|xgjj{LbLz=V+?RKo)BTGWoIbP@jj1s|pGeE|QS|urQv`Un`#2MZVg&+&D'
'<3ySLQLiWXE=|-r(m5oiL&7idz*7szIt4vl$aqi<@uF@mjBaUz^^<vqqb2KuH8;zmiT#YA`}s-CkiIH;U<M1m_=<%uWB2?a1NFdC_NCdR?@u;i'
'`_Z|olfpNaNcCf!`07Ur!Woo$<$Np1a|<8i#8^LyyoLJfFLe-;RQhC~W^{HN8mG8DLlw*ORNyRmv?y5>N*W?NNk-3U?2^>iPEy=ZaAwv?x~B}<'
'K&X?gR1w1k_vM+J>BCK_KB$++ISG959!#W$%q7K06h@EEHwH}#uB2XckHm{bv_|3$c{wTpoUfN-<iwmA8vLy^9WeZ}_yiDocltbF0!qJjLv!Zg'
'l=%Q<61N_%tfu&F(Qd<lgCJNC9frXmj=NB%znB2_c(iMX52}OM2~FIU5<V-lewSWaoQ~03TJ)D=lD1CUZNsQro=;xQ(2w-fCz$HVUc%O%6TC(5'
'oBe>?)yii(!-Es>`Vhs3-iw^ZJ^NCKvm?flPzz)klo*RsJai0J(G+q?U#_3T+|f;&5x|9yZiC<rqj<-PFEpa^5X+ejCYnqGb^6YRWMHroQO#iK'
'R9!WOl}MN1aN53o<Z%N};5I&Z01U~=GEkgNi`>7^uVj&z9V+nao`}ZAJt0-g+!RY*6f#Xwk@?jLcUo|MI@yveo%grYmPwNQm2B*bB+dSm>#rqx'
'J8;Jv!J`8kAI#5=XMQrLm&+#boW$?Ij>o1H3Vp?ZNk13)PG1=$!(FqqfYMN2l{C+u)K}QZ9N!(1DUx!_Ju7u95{8D$ODZQibc-$eoDFO%3qD99'
'L>EI-M)-9B_LW1On$Zs8a8(mw?R}L>@caz>nNZLVcqK+&uo^6~XP8IgMy_tQitw72Xq1@~-b_kQYEnhFSH9W{6bHz{bPmFf@Oqw6-`R=2l%|u~'
'%h2WZqLAoZf$Mdg^*E|rfzRg)>>|&S@BS6zWvrPz68?sKW!Iadu+lSBEL%+U-kh~<-O^I7<#wnU<>3y?jwj;5J8^GVZKwK7%4i8DeE>;Zb5N<1'
'=<x*B*5eCm?opRO@F~Y#TDbcB!BjBc(U)R}e$^LN7cM`Rvy}|tj*!O%jqaJ)I1PYLOM&pQZ)|#ks0hwuM30B*5#v2;{JD`OKl`qw{76({sw~8R'
'E67!}4|p2&Ml0)N^zbo7?wPjR%I4$C`0(*1&-ln-?89x^I3rX0k6k{+M;ywJ!Vq5a&D^UXV(U{XW`VpB1$qy!%I2byB#;M_5`VQUVHtN@UyHk~'
'o+6Bh+u^XtR3U-$I9f~}dVU7p-}|flYsU7v6T8MJq8sJPS6Mf9zxn5wDyOU*-Hq_k#qL_9%e4#xloJyD7u$v{d{$st;qemF)o;tjkB2>d@&+o2'
'UGQUZ?^_Y~Udg!k@(ad2i3jil@Zh59WI2EPfH0b(*G@d*QLYiIeV**oM*k_}xEv37GGFl?ouqm;`cIU#(d)*reoWTi30c4X^0Iznjx){&C8-nb'
'_rp?wdhm0bIR^yl=-pupRb$@vji*uLR?6X+GY&?R*o@3!lF9KwZ8Wc@{SQz}0|XQR000O8003fLbyZ0jSq%UHWibE%6aWAKVQF7uZ*6dIb7fz1'
'V`yb<VJ>iaty*nw+c*;bUZDSh=l#+EZq`YYBHLaL+bh~dfxX>gTPzM(6bda-HZPK>l9Ur4?#JKEkd$b>*iPC7y0I;CG#t)6^URR5+3fSS78y%b'
'Wi?N&VY$-Gt_3Sp$_w`91KY?<sOPCFYh|RBs$zFa-{yt7JAL}(>66bewp5uYm^6$R@H2?Biq%@d1c9>P|7!Uk0n+De*;o><MU@FHG*eZvWjswq'
'ZOzjs8(zqaTM!{DV|kSdbIMrvj!JA`jwmH)$}9H%eOZeYXkV7R%2**Q0ZPqgv!_pTtxA?8d1D(bl7vanUt5?|Da&yg2DPVAc3aD86-K<NwsZEY'
'v_kWu9a3*IURjxjLI31N{0lXjv-eUI+1%-uc>R^enNOdj1viGhF-F)=s?n)<tEybCp0(3YUOj!n;6Ji<Qply|di#nkl`7aZ`>Y$`p4Z}D6lUy!'
'yyr_kHl?Xd4X<CfkKXgb_-DBm!l<Uo#vb0uTUm-s^4*7w{lMyGS%48rVL7alvo{|<yk-?`HoA#sw_@vrl>9zPVNYl`O;QV;d~F1-O;#2~qSc;d'
'BIiwEudn88C9cortPt1di<8)Re=3;@1-t%xO}jR<9a{cX*s;H6D$i$gHd`xoo1k;GUqG8gHC8M9C2xvi_5~E7@lKEx_x2StR<r-H-&7?AYT}P`'
'T3R8hm0e$-Uo22toV-CjA2$0%G`VO>m<>_Iht@eeU66ei_!lj7x@)3O`TYZp1G@(kHUd9G+*dBJO=~Z{T4`YnjvyPg<Bw5yae8?<qVI^C?-&dJ'
'#H)IjNr)rKi;oI?ZaZ8Wg!wO`!gb020$_+R-hr7a40xt4{Zoo!&deI4w=v+VOD#oC5Qf4&6HK);xfTpwC6^%H3Ph+3#DfI@jo#f$V-YZ{s-G7C'
'NCmsOsli^t-rUTcxSN|z@Y|akhS+p-WAc0WoTIG_;9tpQ#Y|Hpo`~!<p<+!-g@;BJFxANdbXx+j8Hh%^Mtm;eqkzAlqSOGZh{*t!+M3hWGGSe~'
'B2n1{FfQ@~_l$<}#!^67UfjZ>lC|I&ey>%Qa%1r$M6A#oDeh=o<E%WJM?y>rfKnxyC`lH;LDr-+H&ZFNN(I`HV{Hnx!oMY`#aCiSNZA;flJ`}z'
'B!~`=oe!d-?mKQZ>WYe??`|Ux59=u<N-6MTS_oc|GTVxm?j0Mc{X;M<zAc^g-;y>qaT_usp7vXBL?XO5Bt}A5ry_xA21m61I2ushYYZiZgg2Q~'
'dtQW}wE#p<>0{a0p{Um9M&vvc7Gn*;Nekh3G1*AdaLh$huK<m6p`E^88}QJ9z*6DvCM(VBwM_SjGNM5LcB@|t&~dOyyo8yFlQ<&CSlU}utEtj8'
'xQvNCVW1iM;Iuv-x$xtPpa&v%Xs-NzTEZyWMrgdxQS7!53B*<op60-_553`S;QS%Uob9BXv-1GOT3|#cgkO=mrAXHhf2ng@x7W^&T0%sVPX5;+'
'SWjjl0Himd8O0g%=zypd@b(lywT4KOmVup^xsx^pKn^Ggs{-7XxRrpw5xR+rFAI?kiHfQ%5RKO?L?=rN3I2-`ElLEim5>6&ns1~EILk@>gzOxd'
'q+V-S4|$+mtqb@^$ac#$SGOYUEm@`;;2FhQl(DECOjxQSi;;>XbR#^!(~2A-S#mu_|FlX))`0MY+-@-7c!%yp4Wk0w$!&U4@U75hoW*&TLOK8J'
'{G^-P1^U>uvuN7YN$_=ax5NVoLeD7qEr8A=_TI@VQ+J6G;85ArYtEg1FWj46w&V-LqbF%X#Q8<UQ(v~^%~{3=lvjC_SrdIWYlPdniB=R1skqj>'
'hh>4Z`K^<c2W$G!VYmAVv?f~#|7(xnw!)KT1A}a5unFRM?wRNLX)sz#<vJi6@h-@LgtF2Qks$!NleNoS9AbOmV!>=lE;v?fpc9zN{?&?Za~lcE'
'?_Cn~jA17W^kA02+B+c4-N|ND=r|D$yk=j%^zD%3P#G(|y+)08;Ph&Thrjvoma1g&2P{SyJBouJT?wV#5MTKML0xXy$KOAFW<QcfKXU0y?j5{q'
'szO5T3uZ!!h`eFci9DsKxm}Nv>}X7*i^(BCd<9spOfcajGMgOe7c(4kpa8lglyEYLj_+o*jvKc5?X3p)5Lo)l3YjovggL`p4E_vuC4&DPAGyYb'
'BmT&(L9(h=X!BALpq$3+Gs>7R;-4dB7mI~|auUE7Ie+B-L@E<fq2--G5CL3(Wa<$*>{B5o^-_i8+R5<S!1WQadU5Gv_0q@c+0~+pRw@q;iCA=X'
'=1It}S%1>ry<OR8GC=C80g``=PjuEl@9`Bc%t2<7s;090(ul+67k<_c?4Q`>fbhyR&+T)E(e%d<|JcGFLA95|29lHHQiFrI=G19$?i@=5ETU+i'
'2_+`-LzKxbP6pFmQX30czq&h_VZ@HEZgJ(I`0}M&;?>1s*ET?DfO$+z|LJ^kdR|8i98=sml<$}&UoOxiUo1R5()q-yz2&{&AeIW6WGOAO<n9G$'
'=T~!f_7cQ{PWY~u_!9WsR;(U~lRmlY@_sHLg(Mu?YGbs39waS34%%;%_&yl9)3$J8_cbrNv5=q+jzl;2GWU?Gt1(j5vc$GP?R85gm)IM(Bk$EN'
'jXxm6k7@xtDn9!?yGcp?fdQ?M%646?$?4sYK;@7S?9FLz#GwY(;%cXXb><sxw7*3;@`6K&t|gl|om$0@Z55(`eH_!X(MP|-N%^_g&VGj9p}a(('
'oOw+^5DWeK#8ZH0P_gGH?D;>~tj~sKue!7<p~h><XQ=%%5YdPeI(MLt5Cx3HoJKTe&f;Id{{DwPML9WPe*-j$UPTzpC6tAKqV#uK!)j(9UBepc'
'LvW>H%g{h7XR-Xg=s5!oBb9;~!g^4hC|}}nNTsZ^JvIOb1O9&P7qyd{4{07t0B5#P#IhB!BFtdC+D-LiN;=xYuCIp-rBf6_@D%r)hDA+@gZlME'
'0-WQC3$|QYCx(4`|L<5esLC5%(VJsJjr{qOSYh%Ua%sO6wA_9q?~oWjP~d4tZ&*xMz=j5knbE$AYeWI>UTzcDbXCabMnl}BN{azq>LBP=q9!eA'
'bm}1+Y|Asp1pR|Jo36m74?QJ#TsQ-F6HzcnJJFBhC@#Uooma=(bI56;`Qttv2|L1XD34DTVg4K{-segF+2L+)*QmIls=LHKwx{m+TSU0`y5%8%'
'q*gB)JbEj&XR*9hwd){Rt4iwQh@46e9lBFBFikx8goXK!RQuM2^i8~x0q(u`4@`?VXS?St=wf1yC%aSgt}a43ao(F4c<#~lNMAsEz7Scqg7MS<'
'=b5t<1Og<2y8E1|yQ=#Z4DP<QW+$Co>w6<YC#{v$d;Q^h?7TJjrC-Bk;QTdD>EPoa7+m&Ne*ou7I_FOO!PFGSEurGh<pWPkyXh&i>kN-ej;KX@'
'`iqU536vubT_Vup0bQG}HwjpiKz4$%l6dQ5Wql`*jbUpHWq089C|srTf=$Efk$4I```u<V3HMO(ZmpDAyWWRPZ{0a!?O+ld)<cx>O}x+%?3#76'
'FFr)}L+7K<pZEH{XU+mwIvZuigF6Wy3tkP4(%mB%$uDP97YIf(#&m1u+pY}!7O2K2cYZwDs{ByFPGN7dB_^*K9uUyc4ba;XtO<F;AFi8ig?D}E'
'w4K<>+~3w&fj=9O#eK|hl_+wZwyW(L*mOqWde{#(YS-GQ*aMhdxjc&X_5OSjD(%akzAxHzmjA`IqQ^Z4&&k|PqUVlf;$13O_>aDU&C;g5Iz=sG'
'>{*4YSxUzy?nNdZ^>#Kz#-Q%NG<GdD9YTF=6d+Q{-M17ka=9LUYF&GLEz(<DTR2puz?G%IJ6E)AJPSN1%>G_X=mA^brI<1L<Y^P6HduPd;Hkjt'
'4V_8C@19(_6}d6rgFbjXjQIggiq3c=x!$|z_$N&rFwO3`#<Qw++rl=HurT8Yzzc;;l8Kc1p<}4`U4!7eq{%3PzE746!S&5~@@;oXay-5bgi*;8'
'Kt;3bKW4o+i_Uz0KJ(0R@UxEyX(z7$y!qe}0T3b_MU3t$-Trvgowu&<tkpM3Sb~89JO{^$O3g3FV10mk>ZJNMRFD-js>cvSY~vCi2m%8WdVsL='
'i#B6>))0j4(1-^}45u62ajtP$JP7V*_eaR?IN9Jn#4(4vBW;iF5&-EKMe$>ddBiC}3va;i#p7VBcNg^!cP}sd75nwW=yopX4?*M8fc79ZZb-$W'
'zz=|HO*XB$wOTG4mvbh@bX|?3`W3_6+|R|k-GJRXe^|lqEzP5qbh*=km&=?73u7#WCkkJnx+8rzoU$0pfx{Aa^a%gk@cX$I1{`Yw#=EC2RRLMO'
'<Pmgt9@icbkrVIu;eK}x%@#))j*fyIPz8I*91zq`&SXjw=6F^9z-@`)c~!W+wku^)2o8&cB+gb3pZpa{J8&C+OXo2XAW|g$Y#uuK|4>T<1QY-O'
'00;m80AgJt;+d5m2><}v8UO$m0001CX<uV+ZE$aMWnXh;V{~b6ZgVbhd6ijNZ`{Zceixws(9u4S6I1k9z$U@i46<>IV1cY{#KPu584x{Wai&)!'
'o9QNHO%(t4RQ17YsKXQ>k$u#CeBHXZxcIr&pjuV|*PVi#)m$mw3RyE<%2u<lqCLnC7sxwN@0nCg-huk6L`$Dqb}9pw3tq~72l4sY`Pub>1{7qi'
'RDou#s6e;8YBW<&fMTs=EiTbe;u7Szlld`b*9V~)&w11061Z=8ooiZGouA1%gNTW$0_4*|d1N~%qynT@=b%uInEewn5KT)wS}tmoQ|v{}OD|z2'
'rOI)j1&u;}l&DV*<>ek~Ky^`yhPMaOE|XP-yctRZ_mFishZ`;~F3!&iB`cPuMc1OG(iA0XB$|%bwQP;aG^uY^<-Fxt$+ZUUm-R#EXKv)c^`R8I'
'{<}gE+<uY<r~KZ}X2yMd<#*6C;vz+f8>u1H8Cr|`Eq~=2zQDeMx{cW16>knA`&{xmV!uj}0XdOAiO~7kUq-^2KkT!DhGRT@R&uX+wK+dy_%yg<'
'!{qKa^qd_j`4N=fuu^C@!n<5ZJHY<OKFS*Gc*UCrUFDb3PYwlZETdE-Nit!m;6EJ(Ok|ivjzPqfl<meCB0>wO8cFGu6AQib<yd2|Xz^cQg;2EJ'
'MKsGh8_!4sSJW6FD2WL=mLVg?mF&phCJ%IMJWWJJ!JH^D#vR6TJfudFus{|?Kmmi!H=+*Hy)I5XdN3?IRnN^DPIV+>MeJs~vis%^h8jZ}yn|ES'
'tDN3>mm-n(cJ&9Pq#`2|C7a3#d-&!kquio}3blTL>^m|LS4SieuE{R6HAW_)H-@NNB?AVf23A6C^on<!qSD5oUJdld&muLfH$An`O!ETEf~Un^'
'u{Tdsn=Z#xCMS_~EzOGGhi{_C1bzMbYPA}PyC~vLiF^+Mqf$Y{*X-S3r+1H$-b}?-T%a^#hlRmU%#vm`09HUgP~AcM6)=F?uhLgFPdJ#0+iwxB'
'iobyGHBuebeisESGu4gBEbE%1t=qhUp2L{2;dJ9?wiqpBHXnj6&+}0f?mE=7#esFa9^%pyi24ZC819gVhJH+gI$VuS=@LTf(NK<N5$(|l5w~zp'
'VH5m&9pE2dW8yw<?3?LsI!2+OzC&01oWB3`%jZvDKBU*zU(#Pb{q4g)f`|pdI65+pWf$2{ePXX!(AzM;FIVAXec)F={vq^I#ZYJ1eo%|}0Qb4r'
'qb%X-W^=XP&IrfVEw<8-EQfARu2qKx0`9Sbrt;Q&&P5>uTbWRpUcYp`j<Ih9lQ1F2+C8z*X(39KXcj5D0mu$W(gd^VnOKgRW7sv=G9Y;vSm<EG'
'ZYI4YB&#RA{UdfK%dUcC$@(<0+J2i=38?w5G%EQ$BN1|dW^RJfrV)ACgfxPpE6ZTIcmM#&BNa7t^de`B$l7ewg{BmpVSB=x(^9pRe;<e3b5dwl'
'aP&$3)+jG1cjjri!*skAO#}Hj5ec#xln~1X>yU0$Xx+TR;8nX|#ki*VxvwFU1iBKMqa>&cY*B#8hN0w!%m}F!PAiVz`;NtLUQle7l(iiam4vFn'
'#5FKttkn<fUk6!YdpAO1Kz39e^?_1<rvQP00i+Gj;4ON9Asj{cmHQC(Fd!Z%`XSVqpjV6tvJ6A7$}1LM{kUSk??bO>G+AQwIckx%l)XxNqQji8'
'e3Q~>+L{X`3@yKF)ngAw@z0Nfkkqb7NAj-MW8*Ivz8G1)<+o_Y2TNrWUlrf<TWe12F*;KQO(Rw>L~H7Pu-tOpQv?q$6x6aB^%^@(uC^R3_%|6j'
'uR8OT_e-a3HX6o<G)8JSTLM=TG#c4BBS)xwSTn54I)&{ACk$YmwJL!;?x{m5Uhg4%r0Ek}nY627*yv+pmNPau8UuDmQQAk6*}t8Yy+vEZs3)G1'
'!qIUcwBRAdLz!xmb>zSV!pl7I%%W3g-y+jw)ZU!w(mfuxhzh2lmUV)GFhNV-(qYSLUa3#r<mG+3kyz2KZ=P=}Gp<H#&nL7-_M{%85siB3i5fq!'
'_jqh|by+D3lL)kz2qau8(E%4slo1^Dbt}ipx33XIX80LTsUlD)P1j;U{F@YY*y|tbrW6J+{Ar87{&+Qu1QW|%!O_t#v@SeIc^f<r;~fe=4TNVQ'
'dyw(6#DQS$D2g!q{_2T9GtM^Jn=@br6tqc@;cqlPhFvWxvo{HJiI6Jo8pG*WLK5(d)&&b>Ul5-t;T>T#Oa=vowh=cOn!DbW5pV2=fJ_{0#%U6G'
'PQhlv7$@u@&?aKAaaMY=%(8wH5TCg*1G1gtV=^VMIZ8nuYzPR@+iDK8T#;MsUVXKC>7mA-x?J_1tX(WzqXOsF#Y076m-9hmt?wy)e*a=0d7(co'
'Z*=BZ2(;*eX#E=Fp;L_(92`({@igf@vLl89y^TD)95hdc#rL?I9@FY3vv%HG*9gcE5?W0up~vp0dl)KBn4%aon5}obWW05ED<k|$J;&T5mi&lN'
'0Pqc?i{H{N$$tGb+``a9t{`lejBb<Yp%$E#qM=6WZ<kK6I4!ubZrrlpVWo1eT!7^WPg+rTFl!eP+Zlv-@3pohM?Tvku&%H|lXu6n<c#g<@nCn!'
'5!c1*t0fsarEK<umeADS+gJSZl)Yy<U%OjayKoPKB6$;!uS)1@gYR_cM7mjTFWs`6H`^6K{VCoYKZDCczh}LTJf$~FQc&x7gxa!zk1VF=D4@*!'
'p64WVQcX(U^Xz6UMj3*=x(#_dPK@u5)MSWM<A_s7vUdsP*1MB{TRL#s+sDNyL3oN}E*CgFbCmdrS_|EwvIpKGqDNa{)7RK74FED#gf9O}QNt;^'
'kM|as7|@MG7B3fEk>gQdOX0VbH6`-&H<zBw6K#{W@MveY*z!gEY%<duRWUMy!A&zM2`r`N_9z(!4cPIcLrv=Eb#b3!O{?1{mE)+H4HjB~0kfdp'
'h!RDJS!Zq<^W%2Ll>XV!rVhZ`qYOb)biEQ_3UP@L2z_!e%?+zD9RlZ$8cs)$r}kc&NSU|qago_TYB(CFKCmRM7wo(7gB9SB>vFGbG?lPF#kt(V'
'VEaE%O9KQH000080001DT@bsJ!yW?w0B{BX022TJ0AXogV{dJ6Z*yf|b8mHSWG--dg;h;&<2Dez2grXIs3<_HvMei3Hc0EkV$nl^-RuYURul#;'
'jbg$QDUej0wCI2D3?+Nhjf%>KWRk=8=DnfDNs_#`m1@C1o3KZ(|HK-CQ^xC{t$}Jl56aYFdmdAu>UJ7U0AE>$Kf}%rrlvh#-hRlFB)Pn3ob6z{'
'ZH6#7+-^a4y>)>wj17u0{N=?o*Qu~S4~Vn2-FfJP8>(;~9@QgG*gfrP_3i}xDT45)#^!|#!EtwSIfH{X^&MkoP!BGqaR*InRgia=7eE)~{?4Ha'
'2-fKb%}PJL`v7XFwFPa!8eDj+ELl<Q2XpW_+*_#a(GUt7;8oYR=tmP6k@@ppIWjBn4z+e-9Qe&wH(&<%q<z@hLmbL+52Z#c&OE>d%5_l`+2l+9'
'B|gOJ%@ldmVIN(|ZI4Efv?y3+kQ{f31m(dFp&!DekFRke_ZTP>)p;RS8I;+?N?ghP$NOIsdQz>RvR&`cd#n>FzkviQV@v626U50&NF&CJsKz?5'
'O+!>{9%*<$K?HI0NGM+<##eNeaCgrEm2>J@$bSbgr1^$6qG!-)BPQ61@TU>~dxDre5g1s;fY+;N9<e1mDTq<YoUqJ`Qj)xBd{(>&l9q1eYY^SU'
'(*~OjB;DY3mCWS9*AMV{(5%3>_Sim?0i;V)>3db@@O7^}&{sk4Y=z9*!Vc>UxHb=HkP~@Ch#c_tw|t((SsOeF6055sza<H)Rgu&E%{sr~(fTI8'
'y_%76+Dls_#P3}1uv|;{5n4KMaZ>&<zwvBjx`F(P?c;aBlZWv~uW3pILRw2gOw)26=25i+7sC&oI3twlgpqOzsY`>5N5k>a;2N#+@|y40QZBD%'
'irk(mMDc$mRjaE5ANk(){GhI)s*wEsA4NN*G?R|FDT*{KuK7+$$#DUnda(#;S<HBht;hf2p;nL&Mw7r!ELfbJaWf6e)460bDI#CLhK(;6<rC1<'
'Vr;L}o~naLe%X_9u5YKFm}NZdF@^M0O3B1!Ni9jr@1#s^<>++4hWZM65xGu14e6i2#CCXj@efc-0|XQR000O8003fLJreJsJ_G;&<p=-(6aWAK'
'VQF7uZ*6dIb7fz4X>)0BZZ2?njaA=I<2Vq0N8*35`r;IY9H3=y)l(jTg}q8FaJ#@uRatH_X)O}lXFCNJ@xR~LPTIiz*ozQNCgYiJzWHV{8jVg>'
'D@!Peko(%G0<0x}Jc2yRj`!~`(OQ84SwKo^#l4>e%?)=@Ka<L+hPV}wQWz+lGTCS}+T96e#;Vq}1{fov(aJc&rBsePp`_j2+1=@^%VKk@f_-<V'
'go>143nLc5D#IJFiICQrw(zG@5^dwj?he5Vb55nz2!OPjI7L7#SB@Cqb?5mTKGeW1kS{lP<T1RrWC4{jKpZ?dqH9Da<TCU5eb74#mfRXHEjl$|'
'EG4>K)S?)W^UF!=)J^0}8FF$nwHdimL~~IRkfl~ax-`Z!&?pC_P$dv}f})L;tVIb5i}iLxR#0m&DZyY>Ti5#rtg3@2qLe;|b-Z}UHg(6Fjy~kS'
'SVc?dz|Hise>H9R5(cl_ps?QOO))7s70#Z>^7Pq(JyWt0%QTkGj@kX?g)(R?g~!J0FCf8$&i3>*SWnH6h8(+^fm{ot<cyw7+1d5io9plA?0))v'
'dVR%yzhGalKc8RHd4>%ZIvIC0RRq-%Ys!kY<e9LHueqrCqK2gJeV~>L?CiHzI?=$nG0G(LQWe<!wXmMM0ES3_(h^S2?mnM*9niO#qEdbm_Nwny'
'+4++VB!YDMge~SZuom#xC(s*(<ZnsNd@q<bfb9?x%A{v4C5@0wizldUet7h0FZ`jEKlRq5<B!qWSK&KsQxup)sW?nEo2O*<Lfx|lI-y?WG>t!#'
'!7#nfJ6U+m`I2zryV_=xPxApEkHW2@kTRt~Cc$Y&HFqmI9}{jPhQWy~u8eCuQYPUkLyT)ME}`GPoYL`ld!PfKAxQ82FX*e$Ep~*wbO3yN=zNkL'
'Rj+pRnv%_-kc__0#{)?sFGA`Tryqscy6)~u$!<EFc^(wdANjfN|D%;9ilG43aGg_*?WgHmhJwUd=|SgF?iFrYcla?S+&Rf`JZor>mXIi?w$%K6'
'j6YkBGW`)eSP`Z*|N8(ZeuFbq)`x{~`K@VjsW0RrJhBji1r$_PAKqv9U)LsBp&FzX{;8^Azfok?%fqYng&I}+wzU=v_PWyoopd+7Py=OtW#`+$'
'^75*mS+U|+0U%%ac}n`=DKh*3Qv87Yg4Y&n=4Hv2o7Q315N+gRfNdj_O_s!`&1e2Uj#d<Q-p`%Jp!oghSVbMQMzXlHZBDhL<IJh3iYzK({BE=&'
'OPlX4^S*EPoWHi3jiI_SAlciCfa8*o@W?{dHvjeD05zzX<z7dTf<fwtbU@{TJ1@tu<&rMlY?8+4Ga7zT*VZrW71uD^Kb%jH5BnfA^Iu`X>ppj&'
'bav3M7=a^G59iBISC=9s8~nbobc?8`dV6eH;0md0{NN{pN(P?_lz>Ay@L~nt<1bK40|XQR000O8003fL+k%OQsRIB20|)>B6951JZDnL>VPAA|'
'VQzC{a%o~^E^v8;RcmkJI28Sk#DBQ!FO#f7W-7JP2q`O}lAR*818QMLD^yvA81QalN45iHXZzpx+6hfyA4Le_>zm_q?#qObspJE*Qx-PNSgr?$'
'oiHWWa8UXWu49;31Cm*>=TPuM=_6=vD=iIJ1$M__`UFY>e-Sn&g!FqGtqP!YQ=u296hu)fZGlOtEaE6>`n`UyvHot9d|N2<_Gpf#K`NQu<zjV$'
'nc>pC-|uC716DI>)>^E1z^Y8Bq2G@nXL4JyEgyoh8vcT>O7aMPj)3L4I#6De_Be!<QaL2>E6WWJVekv&!q{ad*7kYW?*V>1gE}zu6qts{_j>uD'
'GM+m+&b@F8MC|Sb^_Kf66K}cYavzX~bUYc++xy9ke!rVM&C*BuFde7&Bm!cEJd!X*=AsNjP`WWiEmFq!V$GBP2OOv0Cby|WT$j}aptE%F)~YBg'
'%c(t<@1fpKA7+brO6R}LJhqVb^98s!b=j(e%oWS1R?5ByKf1k3>3H(!us_<O{CHlM7f1b@M^{%_QM=QeWhN=uOK=qdk_+CUFk8_1-D5f$KRFUt'
'Bn-n2rmPBDH-|~C8hcp(8NMyJv?c<IRp6^3?v!YYffDM14cl5xP7h94U9c<83Tnik91n60nNceWZY(QGlgxF+JGAx~-;KyI5bs3z+i2{o3}44!'
'Vmok+!4wjH)HW{}e~#R^KPAcQS*6HKxGtvnEj8AmstmRK#U{RwBCyz9Tr=1G$*1_!d4!w@gRxn*h(Cxk65p!9A$AL&+BVb30yS%kJ{znfi(Y>1'
'Hn_|>W?Xes1*Ft!#a1F0_SkXjSW)=~&MY8s;Nd)UDFBX={L^cTx<PhLrd`-hN%!_c#}e0K(YkzY-^SzqQz#C}3>tA~G-^Al>so8f1{z_Vy*|>p'
'yH$7EwG1>9DC)72u9#A-l@7?o@UKTacPJhb+PWi6CTo}az;+-POvu2`Qpa%k#x3VEz?mL1z1@eNn=82M=;qahAUNMn61ediPNwDcGyIgmhrz%r'
'8hAw?!hdLbI--gq#j7YFmGSexJlY*zPlEz)uS_u7wj+!d<H;1Ji#g6SxF3D}vKW0yVR?4Q^GUAn)(BK(aZt8Ri>3TV@@jb9B*R@oLG2pG2QA!W'
'#SRJFZ5L-1MltA&1@C=14cZ~lNn&yqrzE+-!312cRVL&%A(h<>K0Ctxg5BUHELq{MU<@ds)a8a!(nPJ!RKNE(P)h>@6aWAK2mk;8VqJYSYHi~R'
'007}1000sI003=eWNBeva%FRGb#h~6b1raswOMUX<2n}pj>LbkvR~4<ZD|=UGYiv^n3*yl!O$%*b0vf#H*s4JNt}%xUex8k?>V;f+7wu=ZY9*l'
'v5(K|<L4O1alVKw4#@Fwl&}SfSR~RXB4Q#V8ir&kq>_guVQIt_VUJ13efY8Di)AlmQe>&m$Zs)|xI`q11w9^{Rei^CwzuY~h=}LSGnJ*x^9YX;'
'kt#ytSSYHvh~@Uy_EtVBpJJbj;@KCmVvp*crm`3l<<H+z#KnVLW{QX9V>U}t;WJrIS1d~AJY?knkC@dkp=ufOS+3p$9#Jp-Zce453D4i1TwP7Z'
'k3LIKdz|E*oPIwcqpG(3ujcnpnz~NJ0!}@2nU}jvI$aX5In6>fI;Npy9Tc>^1!JT>(PJ^4g)DG8r1zQ3ga{9|w+MVv1|E%x-n_qH%B_gFry&oh'
'VvS8*E9*eZ(E@*nm|1~*PV9S#P5^#JsVu&Co%8Xx(<ATb?Cj!~cm3n+^!)T{{LPz=FE6gAU&ohVi!(<foK8O!4=e>;A-V(-!F7iCR;3xk?<X+4'
'W4F^bP%c^Bq3+(`J<L3*C0HZ^Mz|yamqtNIHnoHS$<lk4Iu+ApLLXw!xLi?r=lUWQ-G)=G1FR-CZjA`~B6;%c0Q$Bvv!7<MYaB0`Y6|bv;$u^~'
'@23m{Qc=kY8`<xQ4PGjxH+xbLv4lgkc&v1=;M=OK(x-|WAGlgV@v+#QJ3E;5J1bcCyaQ>gGE6lV`y$}+V&qWibMACNEg}`D6jj9FXD(6_@|a=p'
'S|`{{!8v`9pDKyFyUx#OdU<(Ji*EH}y#W@uW+?y(xb<du+ad4BaIm{))-jz2#J#v2r>RK0<R=X?rhivOClu?tB6v{op1(*%mUsz7L;~oT%28Po'
'vK!dHi%Q&*|5Q^SkZNk=U@#~sn37D=1=E!fa>PDJxicPTkHm;4_qzmqbw4yPw;q&K2uvwN64%+0PjY82cg$L+Q%;NMBagBOX1aRC4si?y_u&l`'
'6oeq~Ob&v-Q=Yh$#Sn;M{RkGqiiZEKd(hVMMJAzyMmiYHP5mrQS*+?CS9ZjdlCU=5em#=4%xOx+RZUxP8P1nA`Bq=B#2s{vqCKss-dN+D!(-|z'
'e$PAvbA*+F)9G|{TwUvq&i|B6^}xLnyd7gxHHP&#wz*ZOkjG<F)f>Iri0S7M8In(*$o`<Es#+uaN_2HjzfFz5JJ4?pwgQ?EdYeLSJynW*W5m}S'
'n1e^k{+)3kD#jjf35I})5o)wx2MCE>@(%XpG4AU{GmXyRr{RbF=79>pa1=ybN;uYtzhwR$nKAGlpb80Tis%xC!q+V5ki-m5GJ=o`Jc#IJ@xo#c'
'pmi4OpDZZxhHvHy2hEBy&p9N5zUwGi2rXv80b4Q0zFp%5)A7HqPp4z=>-Xu!^`wmqK%%VHiNJF+9-0hHvlwZaju;$gEA9B_gB*1&KQ(U2VtP+`'
'h&CUBo%2k)EO$rGI*dGs^v+qs81sxt*R|qCh2{CtqUi;UV}`!pH3+k2zp)$X(0KZD?45r>1BbgGF+=s)w)gSR5Bo->%^NFne{X}x^V1*R<<-&j'
'$~ze!eVblfoEx!_er}n1ot6#s(Tv9Q@_Ndx_>^y6aeFkw%&o595QqjzD8s^Ina$_W6TqR72gfG4W(OD$r@7iS#XMSR6?b+(k7shuzt}*GR>p+t'
'i!~Y6tYEat4XjSLWgK|(l)*l^+si4PioiDFe9r?WJd;Q+HU|g#Fwn(+$Kyal0XWam6P=dQpR(V~#~jUM26g^qOFm#g#lUJFCz&GiltxVE9y2}I'
'&M#oL&~Rb-g*1x+IsVV07c)#^(;)R9mfVL!Z9doIPFeOy+4a~OL9eAkz7o~T-K@_9&AAE95zi@ju!JrL_b)m!&7aFnZnQjzaqk{Whv7E(qbFGi'
'F7yl!MIP(i9}JvsB^6-KQapeQfEnqg$}y8i4?KWqEg85<X<1^9U)V9)I>MlKZY#p13Ph$x6*VWPlQB*XK&9&VxAD&(*Jo#4-9X}+_i%h^fH%#Y'
'6<WI1_@Hm1g7JUF#a~4h24z5OvglNU136+bI#}i&w*CPZqj`Y3G?1>{(iqh-XKCrko7;Na?VDHBx-{LG<;W6NTAfr%{Q_c*>xuEbB_dY-&EWRN'
'(eIsGOW3M!Y`AjWptsWa0aa#U@#rt_^!(`i*qfaGH9os^P<^aM&3<_VkmL{s0jMqbF}c3V7g!W7xEiAHunEF;Ur3kT!H4(z1I;xGcWL#>rx6(-'
'|F>4=kLKmOeE4&|)ys};V%w<>_X;h0CMzxT`Eg}rK5L4@0%<O2dU(~6gGw*qk1Slr%AzLi*|4%&Js)CG;)sG6E?MJR2SvBTq^GhNz||K~1PzjF'
'zYjpXR5uXOThkL)?iq?VMbE95KnU<nE}`@?pbTQ6lV2ogCT?dzu5|h_0OuOu<zmhFKgiV*&IU3M=|YkylUPp7XMrs!iDqLmry{g2Om)KtBvZl9'
'MM{&UW)Cct{TlpMc1GyK9n5t{Pc4!BYAEU%IbO>U=n3uUWAUoVO;IREi7Aa2%nhNn&s_vZ`mn9b2F-H+@LMbhtrVJ543j^|P`?5)wxLgN&PAqe'
'dnu@^%A$PY@IX(yae9zco+h*wVE6Y*a?<IxZ>K+PjvBt?t>S%c;I?)lP-H^E?6dq@=%uu^h^|>Z&=ERgp+4$$5?79ZuUaKwmlHEkp8J;?(1@K#'
'Lb1?!7I1l|qal?@fZ+cYv;5fSkcqmCm-q|FUcp2As65JrL@p#wFE3444I|d)jwBQr7e%Fj&tlO~vdAc;j{>kDOPU6jo*97}2o7Nq<aP|0fxG$?'
'?|gU|65;z<3hJo`#$qd0BEA9Y1K^X%lUj;6Co28IcJl)Z1UroC&1Aji+wsGIM*U!F6x*iuOb03_9C&$_1+Z~>a{M|uy_}4v`al5xA)D4HWA6#U'
'P0~*}&>^h3T+ixBa(sL~8Go;?FE`7(2Ruyvx!*iZN_Fr+`E$RmE%+Nc4Sgq=-+1@07~Wme^ywI$Tz&lqS>And0+62SjhoHB8jqXkOUEL7mO*LB'
'mNmg&{q`tT`gQ{AtRD#ov52|v*19taEC#03Z#LHNXn@FuMOE|@5r)`muC=Tcd~9wCP?~G>xv2b2_m@^H=zVmu<>xT0UD>72G8f{s3z>nGyH?Fw'
'QoXjjc6{|(IJcSyo%SWjI$_o4GDA1We$Bcv+!}doWw|Sq4*ir(y0xb2jke?G4Yv|4))lL;RPzU*24HgW1V>ZR0KZ^<dmFsbN$v}086g!PBGf0~'
'LW}eDD-8Cyq(DiKmA&9LBd}<gt9G{Z%~iu+=$_X%c)6Q4Tny81l!yK8qG`g-%1ZhgWP{{&0l3<m^5$nlBH(93YUyg1A!CU<=nvO#Sxr1!(jqS|'
'h5z)7scOx_bv$Y=f1q5sMTgiT!roO@@bt>e9zQ17Rlg$tjw9l|L5Iei29bMh$sAk|hD~F20lpD=UsHI~&ftl@WbDozR0iQdv4dT*+cs`*lkr(G'
'tr-yJ{r4aD>i=d~zm#ZFy7F8rt}TlGy{nZ&^67TWUgmfGe>1Ha*3$o3Wch<Qhj?qJTQQ3gbH0Fj)(nyczp;hEm)MhFo3A@i?9$f%gJLY)-uf?4'
'O9KQH000080001DU29Q+E~*9q03RCw01^NI0BvPtX<=V#Z(?6_bZ>HHE^v9(SzT}2HWYm~;Qzq#C01uUZjk|lHw{Z$cT3mQf#U!hf<TaMDp4bg'
'8b!4+JpcDyen_Gu%5~bH!1NH=<mDxK&f&e6PQd6Id0vq5j1kX6;X0v=BR`HQ_jyR;wA<-+bSq7Zk33wHZYL0<`6dbD+kE6{yfH>F@=r!mE}e8c'
'=hHV6<bs>wE#V~o)WheKH)p5byXh<Maz33+JYj+R2s5&#oM2~|auy~%XBg4Xg!P<m=k@f%d^UaWUA#a2F!5fUUQWEL+54(jyN&Q`cx37WKE~j+'
'zK+w2Gw;vI-}@J<jjDdTo%z}MWO_CCE+?>#H<#dJ?LYKh4o7_yMAYZKJuYiwh3lWN<AB%MWIj8a<lbKN(K?KKCw&ygdmZXw!qH3N$WcrzBR@^i'
'Ia!7NJG!{!l#y|_gWy?_0C^Ez`jMB4=3Yvoz|o!Jk(`Hd!2DrCSQ@4<@si~9%Q47Lk>R2o%LvFUuJ#$zD%QOvjf3!3O^gP=BAz7?xdwdu;GW$W'
'YY77cIz@KA4LdovAEpG&vRFuW!Wd<3LP23F-KurKxmXE_)aGmv!vT}FX>ISU8aexd`)#YKp)1c~uXHL9?w4QJgx}HCSb%J_CO*Hbw>>vLe@toY'
'E$C`9Myqhi6;BWaZqUEz1C0qNR=kW6egvo4I~kpf(Sp(l4Bq-tnp;T9^SCjbku-rIM6@#l-h~fC&^_6}c(IXTO+E&A855HPcq_*2wqi#+Tz20P'
'e+5bglWmnf{MSGNI{;t@;-rtoeEpiy>2gUD4t>~9k|<otuz4(C!G`IsqM!n_ob}q~jAdQp*EEJR#s~9FLhubNvpG>tw0PiXbB_qEjnU*mtPRYn'
'WEiKJGURhk)(MA04SJ1lNlyn2r?w!@HnkS<u$&>cWUVPR^HDnM+dF})mL83VtKThF0uHx;x5Y@DMw)TA7H%!pPO}N<D-;Zga#bJzDZuAh3IbdK'
'8ZHtFZ<*hUWd5uO_rplysU3^`K+1w3-@~dGTNWlBA_*^fKV+J!B5GUJR-lb7zKZW-`Z)#yA#b3MbRp78-Uy6mL^kgBOseBp`;?Z5+3pRW=PS{P'
'hs3LdLQ2&0kQESwvY0$1WXa*13_7^oA;gFRbWR@l_4vhTG`iWf46&=js)t|DkE2lwFvS45c@NH{r-t@gSk@@!<C;cg3;=!)05fPHSpfLQe13tp'
'wmt1yp?n04wH<2x8u4F#DB~EE7GPrmva)3`F+-ww#Ie<Z?g#V&_|Q?#;!2=3Od~>)-e`D&o}->IQk75z6HU3=u#}rX0h}ZP33gkqb%z}@NhcZi'
'U|f5ShE6YR<mme_ZnP<)n%R5<H7FQEhHhqNAT5h{OQ%jL;Ln=bW!|o(F~D5a{E)L`(T8hjtHf=Vo6fL_L3Q(%C@uDHX;u(eIy4Hdep@UQ60Uk#'
'`Xv~y{9Eg-30j?_CM3_?;+FH}nz1>50C6|X5>a!*8MR%b+^zWgjO~6#Mff|CNFO0MBS&iTBivYhReM8A-6$S>aW%)atq1rRVlQE_A05jaF#yKW'
'<E`RtS2r)Rw7YF?5DQL4!DXiwD{k?)on)-p1)FxBv0@l(TB*j8Ww5ZZy)(X7PBP?bsbYD4&0{OiiM3LG%)!waz;=G}nbn-m|5JKKRk>G%zfZnZ'
'g+(pr&RIsPbQ;JN3xa~(Qmk9+8d=~s%D74}hi+k|N;^?JHSc`F3KkHiVGLS_yS8AqFD2w40v9JeYpFe1ig_fK@1_F#M!qN$LRFF$m5yzJA!Kmj'
'ifFb9A|F1g8Jz=o#23@cxu_qlWJy2AdIxpJtd9{)u-Yq}Se-Dc!Nd|6c1_f4q+LsLZki|y;=b;TKLv<V{%wj^AeOqXN&V10SJb~pZ_CIJ*BT4B'
'5D=ynb{F2>c}qX>7ZIu6_`e%W2WU;M{?SU4`ve`yH%wse6`IYjU)ES92-MMqTQB1a;#<^Kg`3lR5|=6zcz+d|I(XSmQ+?NRO873{yC))~*f7|Y'
'kc&S}AjKwoRy6%6{6w&00W=8XFugk*%{FP6b#_reTY~9#C*qK*sZswt6?+QZZ=z1Q5I=)j*z=bPIJjKO6?`Gfm6S>@$~(+cGg3`76@H!yTD|o_'
'zV$~2EjtZX29%r%^nVkg376{UyONMSknOn=KB@@SH){10lb$ZMm!8U}{9TUU5%juPllWnsGa3z&$cMXY!sx>W>wkx+Yr0$ghlpVOKTt~n0u%rg'
'000080001DU9eRo+Pw|{0CX|{01W^D00000000000HlEc0001CX<uV+ZE$aMWiD`eP)h*<6aW+e000O8003fLu@CGSlLi0)s1E=D6#xJL00000'
'00000q=D!T003cWUt@1=aBp*EUt@1=ZEtRKE^v8JO928D02BZK00;m80AgM1tGt8yBme+_fB*m#00000000000001_fvyz*0AXogV{dJ6Z*yf|'
'Wn^h|Z*nehc~DCM0u%rg000080001DU7Llb)6Z-G0Q8^%02BZK00000000000HlH1IsgD+X<uV+ZE$aMWnX1(XK8L_E^v8JO928D02BZK00;m8'
'0AgM4ZTkbt1ONbd3;+NU00000000000001_f!n750AXogV{dJ6Z*yf|XmoUNE^v8JO928D02BZK00;m80AgKY6}YYK4gdfSH~;_;0000000000'
'0001_fz_-40AXogV{dJ6Z*yf|X>u-bc~DCM0u%rg000080001DU2nn&$3YAL03RU$02BZK00000000000HlHNy8r-TX<uV+ZE$aMWnXJyXJ>3>'
'E^v8JO928D02BZK00;m80AgLfdEwr(0ssI%1pojS00000000000001_fn&%30AXogV{dJ6Z*yf|Y;R*>Y+qq;a4v9pP)h*<6aW+e000O8003fL'
'?4EHRE(8Do;Ryf$82|tP0000000000q=8Gz003cWUt@1=aBp*EUu<t<VQgP>bZ>HHE^v8JO928D02BZK00;m80AgJ+iGDG|3;+NXA^-ps00000'
'000000001_fw0d20AXogV{dJ6Z*yf|ZEtjGZ*DGdc~DCM0u%rg000080001DUEA-QN$U#$0Hq}W02BZK00000000000HlGX+yDS!X<uV+ZE$aM'
'WnXY_Y-wY8E^v8JO928D02BZK00;m80AgLIYV7+K3IG71CIA2x00000000000001_fx_qj0AXogV{dJ6Z*yf|adlyAX>@rmaCuNm0Rj{N6aWAK'
'2mk;8VqF{IDj8)C004Y9000;O0000000000005+c3-bT~VQF7uZ*6dIb7fz0WoBt^Wo>0{bS`jtP)h*<6aW+e000O8003fLbyZ0jSq%UHWibE%'
'6aWAK0000000000q=BXZ0RUlXUt@1=aBp*EUvpz<Wo=<DaCuNm0Rj{N6aWAK2mk;8VqGHQnUx+1007z=000&M0000000000005+cF%JO%VQF7u'
'Z*6dIb7fz1Wn*+{Z*FrgaCuNm0Rj{N6aWAK2mk;8VqFlsl*1ka003|X000vJ0000000000005+cgBSq-VQF7uZ*6dIb7fz1Z*^{DE^v8JO928D'
'02BZK00;m80AgJ|67Qit1ONc#2mk;S00000000000001_fzlfR0AXogV{dJ6Z*yf|c4>2IZ*DGdc~DCM0u%rg000080001DUE6|*hp7Vq00RgB'
'022TJ00000000000HlFJAOQevWn^h#UvzR|ZgXRDX<}tAaCuNm0Rj{N6aWAK2mk;8VqJYSYHi~R007}1000sI0000000000005+c93%k%ZDnL>'
'VPA4(b8mHWV`XzLaCuNm0Rj{N6aWAK2mk;8VqI%dfi9{B0018w000sI0000000000005+cFE0TAZDnL>VP9%*VqbG~Z*pZWaCuNm1qJ{B000yK'
'6aXa!000R#0RR91'
)
BUNDLE_SHA256 = '5887f18ea0255a0e69fb6c7dec2cd5c9fb1fd6bb5a7b8fabbf3a28ecc19c5c77'
raw = base64.b85decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256, 'Damaged code bundle'
RUNTIME = Path('/kaggle/working') / ('sh01-onecell-' + BUNDLE_SHA256[:12])
CODE = RUNTIME / 'code'
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name in archive.namelist():
        target = CODE / name
        if not target.resolve().is_relative_to(CODE.resolve()) or not name.endswith('.py'):
            raise RuntimeError('Unexpected file in source bundle')
        data = archive.read(name)
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError(f'Bundled code was locally edited: {target}; preserve it before rerunning')
        target.write_bytes(data)
source = Path(SOURCE_PATH).expanduser()
if not source.is_file():
    raise FileNotFoundError(f'Mount the video and set SOURCE_PATH to the exact filename: {source}')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this cell')
subprocess.run(['nvidia-smi'], check=True)

"""Repair a private editor venv while retaining Kaggle's CUDA package paths.

Uses only the standard library until pip is working. This source is inlined by
build_one_cell.py so the notebook does not need a checkout or a second download.
"""
import importlib.util
import json
import os
from pathlib import Path
import site
import subprocess
import sys
import venv


def prepare_editor_environment(runtime, *, host_paths=None):
    """Create or repair the owned venv; never install into the host interpreter.

    Debian/Kaggle can expose dist-packages in the kernel but omit those paths
    from a venv. A local .pth file adds the actual host paths after local packages.
    The host's pip can install into a pip-less venv using its --python option.
    """
    runtime = Path(runtime).expanduser().resolve()
    runtime.mkdir(parents=True, exist_ok=True)
    root = runtime / 'venv'
    python = root / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    log_path = runtime / 'setup.log'
    child_env = os.environ.copy()
    for key in ('PYTHONHOME', 'PYTHONPATH', 'PIP_TARGET', 'PIP_PREFIX', 'PIP_USER'):
        child_env.pop(key, None)

    def execute(args, *, check=True):
        """Keep subprocess startup diagnostics, including sitecustomize errors."""
        result = subprocess.run([str(x) for x in args], env=child_env,
                                capture_output=True, text=True)
        with log_path.open('a', encoding='utf-8') as stream:
            stream.write(result.stdout + result.stderr)
        if check and result.returncode:
            raise RuntimeError(
                f'Editor environment setup failed. Log: {log_path}\n'
                + (result.stdout + result.stderr)[-3000:])
        return result

    if not python.is_file() or not (root / 'pyvenv.cfg').is_file():
        # Do not rely on distro ensurepip, and do not clear an existing runtime.
        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(root)

    probe = execute([python, '-I', '-c',
        'import sys,sysconfig,site,json; print(json.dumps({'
        '"prefix":sys.prefix,"version":list(sys.version_info[:2]),'
        '"sites":site.getsitepackages(),'
        '"purelib":sysconfig.get_path("purelib"),'
        '"platlib":sysconfig.get_path("platlib")}))'])
    info = json.loads(probe.stdout.strip().splitlines()[-1])
    if Path(info['prefix']).resolve() != root.resolve():
        raise RuntimeError(f'Refusing to install outside the editor venv: {info["prefix"]}')
    if info['version'] != list(sys.version_info[:2]):
        raise RuntimeError('The editor venv uses a different Python version. Use a new runtime directory.')

    local_sites = []
    for value in [info['purelib'], info['platlib'], *info['sites']]:
        path = Path(value).resolve()
        if path.is_relative_to(root) and path not in local_sites:
            path.mkdir(parents=True, exist_ok=True)
            local_sites.append(path)
    if not local_sites:
        raise RuntimeError('No package directory inside the editor venv was found.')

    if host_paths is None:
        host_paths = [*site.getsitepackages(), *sys.path]
    shared = []
    for value in host_paths:
        if not value:
            continue
        path = Path(value).expanduser().resolve()
        if (path.name not in ('site-packages', 'dist-packages') or not path.is_dir()
                or path.is_relative_to(root) or path in shared):
            continue
        if '\n' in str(path) or '\r' in str(path):
            raise RuntimeError('A package directory contains an invalid newline.')
        shared.append(path)
    # Also bridge distro purelib/site differences inside the venv itself.
    for local in local_sites:
        paths = [p for p in local_sites if p != local] + shared
        (local / 'sh01-kaggle-packages.pth').write_text(
            ''.join(str(p) + '\n' for p in paths), encoding='utf-8')

    health_code = (
        'import json,sys,pip,wrapt; '
        'print(json.dumps({"prefix":sys.prefix,"pip":pip.__file__,"wrapt":wrapt.__file__}))'
    )

    def healthy(result):
        if result.returncode or 'Error in sitecustomize' in result.stderr:
            return False
        try:
            data = json.loads(result.stdout.strip().splitlines()[-1])
            return (Path(data['prefix']).resolve() == root and
                    all(Path(data[key]).resolve().is_relative_to(root) for key in ('pip', 'wrapt')))
        except (ValueError, IndexError, KeyError):
            return False

    health = execute([python, '-I', '-c', health_code], check=False)
    if not healthy(health):
        (runtime / 'dependencies-ready').unlink(missing_ok=True)
        spec = importlib.util.find_spec('pip')
        if spec is None or not spec.origin:
            raise RuntimeError('The notebook Python has no pip. Install pip for that interpreter first.')
        # pip's directory __main__ prepends its entire site-packages directory,
        # allowing legacy dataclasses/typing backports to shadow the stdlib.
        # Its cross-interpreter runner redirects only the pip package instead.
        runner = Path(spec.origin).parent / '__pip-runner__.py'
        if not runner.is_file():
            raise RuntimeError('Host pip needs its cross-interpreter runner (pip >=22.3).')
        # --ignore-installed makes both bootstrap packages local without removing
        # any globally installed version. --no-deps prevents unrelated changes.
        print('Repairing editor pip and startup dependency…', flush=True)
        execute([sys.executable, '-I', runner, '--python', python,
                 'install', '--disable-pip-version-check', '--no-input',
                 '--ignore-installed', '--no-deps', 'pip>=23,<27', 'wrapt>=1.16,<3'])
        health = execute([python, '-I', '-c', health_code], check=False)
        if not healthy(health):
            raise RuntimeError(f'Editor startup check still fails; see {log_path}\n'
                               + (health.stdout + health.stderr)[-3000:])
    return python, child_env

PYTHON, setup_environment = prepare_editor_environment(RUNTIME)
# Use a private environment; preserve Kaggle's CUDA-enabled Torch installation.
import importlib.metadata
constraints = RUNTIME / 'constraints.txt'
constraints.write_text('torch==' + importlib.metadata.version('torch') + '\n')
setup_log = RUNTIME / 'setup.log'

def run_setup(command, *, env=None):
    """Record setup failures without putting secrets or huge pip logs in output."""
    with setup_log.open('a') as stream:
        result = subprocess.run([str(x) for x in command], stdout=stream, stderr=subprocess.STDOUT,
                                env=setup_environment if env is None else env)
    if result.returncode:
        print(setup_log.read_text(errors='replace')[-4500:])
        raise RuntimeError(f'Setup failed; full log: {setup_log}')

stamp = RUNTIME / 'dependencies-ready'
setup_version = 'bootstrap-v3-av18;torch=' + importlib.metadata.version('torch')
if not stamp.exists() or stamp.read_text() != setup_version:
    print('Installing isolated editor dependencies…', flush=True)
    run_setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-q','-c',constraints,
        'fastapi>=0.115,<1','uvicorn>=0.30,<1','pydantic>=2.9,<3','requests>=2.32,<3',
        'httpx>=0.27,<1','python-dotenv>=1,<2','Pillow>=10','numpy>=1.26',
        'transformers>=4.51,<5','accelerate>=1,<2','bitsandbytes>=0.45,<1',
        'json-repair>=0.30,<1','faster-whisper>=1.1,<2','av==18.1.0','sentencepiece'])
check = subprocess.check_output([str(PYTHON),'-c',
    'import torch, bitsandbytes, transformers, json, pathlib, site; '
    'assert torch.cuda.is_available(), "CUDA unavailable in fresh subprocess"; '
    'import sys; roots=[pathlib.Path(p) for p in sys.path if p]; '
    'libs=[str(p) for root in roots for p in root.glob("nvidia/*/lib")]; '
    'print(json.dumps({"torch":torch.__version__,"gpu":torch.cuda.get_device_name(0),"libraries":libs}))'],
    text=True,env=setup_environment)
gpu = json.loads(check.strip().splitlines()[-1])
# Test the actual decoder in the isolated interpreter before loading any models.
run_setup([PYTHON, '-c', """
import io, wave
from faster_whisper.audio import decode_audio
buffer = io.BytesIO()
with wave.open(buffer, 'wb') as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(16000)
    wav.writeframes(b'\\x00\\x00' * 16000)
buffer.seek(0)
assert len(decode_audio(buffer, sampling_rate=16000)) == 16000
"""])
stamp.write_text(setup_version)
print('GPU ready:', gpu['gpu'], '| Torch', gpu['torch'], flush=True)
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run_setup(['apt-get','update','-qq'])
    run_setup(['apt-get','install','-y','-qq','ffmpeg','fonts-dejavu-core'])

MODELS = Path('/kaggle/working/sh01-editor-models'); MODELS.mkdir(exist_ok=True)
def download(url, path):
    """Atomic public model/binary download; retry transient network errors."""
    if path.is_file() and path.stat().st_size:
        return
    partial=path.with_suffix(path.suffix+'.part')
    for attempt in range(3):
        try:
            request=urllib.request.Request(url,headers={'User-Agent':'SH01LocalEditor/2.0'})
            with urllib.request.urlopen(request,timeout=120) as response, partial.open('wb') as out:
                shutil.copyfileobj(response,out,1024*1024)
            partial.replace(path);return
        except Exception:
            partial.unlink(missing_ok=True)
            if attempt==2: raise
            time.sleep(2**attempt)

# The standalone Piper distribution avoids Python-version wheel mismatches.
PIPER = MODELS / 'piper' / 'piper'
if NARRATION != 'off' or 'voiceover' in REQUIRED_FEATURES:
    if not PIPER.is_file():
        print('Installing local narration engine…',flush=True)
        packed=MODELS/'piper_amd64.tar.gz'
        download('https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz',packed)
        with tarfile.open(packed) as archive:
            archive.extractall(MODELS,filter='data')
        PIPER.chmod(0o755)
    locale, speaker, quality = VOICE_NAME.split('-')
    language = locale.split('_')[0]
    voice_url=f'https://huggingface.co/rhasspy/piper-voices/resolve/main/{language}/{locale}/{speaker}/{quality}/'
    voice=MODELS/(VOICE_NAME+'.onnx')
    for name in (VOICE_NAME+'.onnx',VOICE_NAME+'.onnx.json'):
        download(voice_url+name,MODELS/name)
    download(voice_url+'MODEL_CARD',MODELS/(VOICE_NAME+'-MODEL_CARD.txt'))
    run_setup([PIPER,'--help'])
else:
    voice=MODELS/(VOICE_NAME+'.onnx')

WORK = Path('/kaggle/working/ai-compose-onecell-jobs') / time.strftime('%Y%m%d-%H%M%S')
WORK.mkdir(parents=True,exist_ok=True)
environment = {
    'PATH':str(PIPER.parent)+os.pathsep+os.environ.get('PATH',''),
    'HF_HOME':str(MODELS/'huggingface'),
    'AI_COMPOSE_HF_MODEL':TEXT_MODEL,'AI_COMPOSE_HF_LOAD_4BIT':'true',
    'AI_COMPOSE_HF_MAX_NEW_TOKENS':'3000','AI_COMPOSE_VISION_HF_MODEL':VISION_MODEL,
    'MEDIA_CLIP_WHISPER_MODEL':'small','MEDIA_CLIP_WHISPER_DEVICE':ASR_DEVICE,
    'MEDIA_CLIP_WHISPER_COMPUTE_TYPE':'float16' if ASR_DEVICE=='cuda' else 'int8',
    'MEDIA_CLIP_WHISPER_CPU_THREADS':'4','MEDIA_CLIP_WHISPER_CACHE_DIR':str(MODELS/'whisper'),
    'AI_COMPOSE_TTS_MODEL':str(voice),'AI_COMPOSE_MUSIC_PATH':MUSIC_PATH,
    'LD_LIBRARY_PATH':os.pathsep.join(gpu['libraries']+[os.environ.get('LD_LIBRARY_PATH','')]),
    'AI_COMPOSE_USE_NVENC':'false','AI_COMPOSE_VIDEO_CODEC':'libx264',
    'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'4',
}
payload = {
    'brief':BRIEF,'preset':'viral_short','target_seconds':TARGET_SECONDS,
    'output_width':OUTPUT_SIZE[0],'output_height':OUTPUT_SIZE[1], 'output_count':1,
    'captions_enabled':True,'fetch_assets':True,'llm_required':True,
    'refinement_iterations':REFINEMENT_ROUNDS,'local_asset_library':LOCAL_ASSET_LIBRARY or None,
    'asset_sources':{'local_library':True,'wikimedia':True,'pexels':True,'pixabay':True},
    'editorial_policy':{'source_audio':'preserve','supplementary_visuals':'require',
        'camera_motion':'dynamic','motion_graphics':'hero'},
    'editor':{'enabled':True,'narration':NARRATION,'narration_budget_seconds':12,
        'caption_style':'highlight','theme':'editorial','progress_bar':True,
        'music':'library' if MUSIC_PATH else 'procedural','procedural_sfx':True,
        'transition':'dissolve','min_graphics':2,'min_visual_layers':1,
        'required_features':REQUIRED_FEATURES,'visual_review_required':True,
        'strict_requirements':True,'publish_outputs':False},
    'sfx':{'mode':'adaptive','volume':0.20}, 'music_volume':0.12,
}
config={'source_path':str(source),'env_file':PRIVATE_ENV_FILE,'environment':environment,
    'work_dir':str(WORK),'payload':payload,'timeout_seconds':TIMEOUT_SECONDS,'keep_api':KEEP_LOCAL_API}
config_path=WORK/'private-config.json';config_path.write_text(json.dumps(config,indent=2));config_path.chmod(0o600)
# Run outside the notebook kernel: no stale imports and no foreground Uvicorn cell.
from IPython.display import Audio, FileLink, Video, display
import numpy as np

def beep():
    """Notify at readiness, completion, and failure; browser autoplay rules apply."""
    t=np.linspace(0,.5,4000,endpoint=False)
    display(Audio(np.sin(2*np.pi*440*t),rate=8000,autoplay=True))

beep()
print('Running the integrated editor. Evidence, drafts, and logs are saved under:',WORK,flush=True)
runner_log=WORK/'driver.log'
with runner_log.open('w') as stream:
    process=subprocess.Popen([str(PYTHON),'-u',str(CODE/'ai_compose_kaggle.py'),str(config_path)],
        cwd=CODE,stdout=stream,stderr=subprocess.STDOUT,env=setup_environment)
    offset=0
    try:
        while process.poll() is None:
            time.sleep(2)
            with runner_log.open() as current:
                current.seek(offset);message=current.read();offset=current.tell()
            if message: print(message,end='',flush=True)
    except KeyboardInterrupt:
        process.send_signal(2)
        try: process.wait(timeout=25)
        except subprocess.TimeoutExpired: process.kill();process.wait()
        beep()
        raise
with runner_log.open() as current:
    current.seek(offset);print(current.read(),end='')
beep()
if process.returncode:
    display(FileLink(str(runner_log)))
    display(FileLink(str(WORK/'server.log')))
    for diagnostic in WORK.glob('*-output.zip'): display(FileLink(str(diagnostic)))
    raise RuntimeError('No passing final render was produced. Review the saved diagnostics; requirements were not silently disabled.')
result=json.loads((WORK/'latest-output.json').read_text())
for output in result['outputs']:
    display(Video(output['local_path'],embed=True,width=360))
    display(FileLink(output['local_path']))
display(FileLink(result['archive']))
display(FileLink(result['receipt']))
if 'api_connection_file' in result:
    print('Loopback API remains available. Connection details:',result['api_connection_file'])
print('Save a Kaggle notebook version with outputs to retain these files after the session ends.')
